# Eval Baseline CQTdiff+ Fine-tuned Ready

One-shot standalone baseline CQTdiff+ fine-tuned evaluation notebook.

Notebook ini menanam salinan `code_final_run_v2.py` di dalam cell `EMBEDDED_CODE_FINAL_RUN_V2`, lalu menjalankan jalur yang setara dengan:

```bash
python code_final_run_v2.py --phase eval --models baseline_cqtdiff_finetuned
```

Preflight di awal notebook menyiapkan eval-only usage:
- validasi dataset/preprocessed/masked supaya pipeline tidak fallback ke preprocessing ulang;
- install missing Python requirements;
- pastikan repo external CQTdiff/audio-inpainting tersedia;
- siapkan checkpoint `baseline_cqtdiff_finetuned` dari folder fine-tune;
- download checkpoint official MusicNet audio-inpainting CQTdiff+ jika belum ada.


In [ ]:
from __future__ import annotations

import csv
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tarfile
import urllib.request


PIPELINE_STAGE_NAME = "code_v4_musicnet_cqtdiffplus_44k"
COLAB_DRIVE_ROOT = Path(os.environ.get("COLAB_DRIVE_ROOT", "/content/drive/MyDrive"))
DEFAULT_DRIVE_DATA_ROOT = COLAB_DRIVE_ROOT / "THESIS CODE"
TARGET_SR = int(os.environ.get("PIPELINE_TARGET_SR", "44100"))
SEGMENT_SAMPLES = int(os.environ.get("PIPELINE_SEGMENT_SAMPLES", "184184"))
SEGMENT_DURATION = SEGMENT_SAMPLES / TARGET_SR
EXPERIMENT_CONFIG_ID = (
    f"musicnet_cqtdiffplus_sr{TARGET_SR}_n{SEGMENT_SAMPLES}_"
    f"dur{SEGMENT_DURATION:.6f}s"
)
GAP_DURATIONS_MS = [100, 300, 500, 750, 1200, 1700]
DATASET_RANDOM_SEED = 42
TARGET_MODEL = "baseline_cqtdiff_finetuned"
SOURCE_FINETUNED_CHECKPOINT_DIRNAME = "baseline_cqtdiff_finetune"
EXPECTED_FINETUNED_CHECKPOINT_FILENAME = f"{TARGET_MODEL}_best.pt"

CKPT_URL = (
    "https://huggingface.co/Eloimoliner/audio-inpainting-diffusion/resolve/main/"
    "musicnet_44k_4s-560000.pt"
)
CKPT_FILENAME = "musicnet_44k_4s-560000.pt"
MUSICNET_URL = "https://zenodo.org/record/5120004/files/musicnet.tar.gz"
MUSICNET_METADATA_URL = "https://zenodo.org/record/5120004/files/musicnet_metadata.csv"

IMPORT_CHECKS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "librosa": "librosa",
    "soundfile": "soundfile",
    "scipy": "scipy",
    "resampy": "resampy",
    "tqdm": "tqdm",
    "torch": "torch",
    "torchaudio": "torchaudio",
    "omegaconf": "omegaconf",
    "hydra": "hydra-core",
    "einops": "einops",
    "torchvggish": "torchvggish",
    "visqol": "visqol-python",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "PIL": "Pillow",
}


In [ ]:
def info(message: str) -> None:
    print(f"[baseline-eval] {message}", flush=True)


In [ ]:
def fail(message: str) -> None:
    raise SystemExit(f"\nERROR: {message}\n")


In [ ]:
def run_command(command: list[str], cwd: Path | None = None) -> None:
    info("Running: " + " ".join(command))
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)


In [ ]:
def project_root() -> Path:
    if "PROJECT_ROOT" in os.environ:
        return Path(os.environ["PROJECT_ROOT"]).resolve()
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    return Path.cwd().resolve()


In [ ]:
def is_colab_runtime() -> bool:
    return (
        "COLAB_GPU" in os.environ
        or "google.colab" in sys.modules
        or importlib.util.find_spec("google.colab") is not None
    )


In [ ]:
def maybe_mount_google_drive() -> None:
    if not is_colab_runtime():
        return
    if COLAB_DRIVE_ROOT.exists():
        info(f"Google Drive already mounted: {COLAB_DRIVE_ROOT}")
        return
    try:
        from google.colab import drive
    except Exception as exc:
        fail(
            "Runtime terlihat seperti Colab, tapi google.colab.drive tidak bisa di-import.\n"
            f"Detail: {exc}"
        )
    info("Mounting Google Drive at /content/drive")
    drive.mount("/content/drive")


In [ ]:
def default_data_root(root: Path) -> Path:
    if "MUSIC_INPAINTING_ROOT" in os.environ:
        return Path(os.environ["MUSIC_INPAINTING_ROOT"]).resolve()
    if is_colab_runtime():
        return DEFAULT_DRIVE_DATA_ROOT.resolve()
    return (root / "music_inpainting").resolve()


In [ ]:
def default_external_root(root: Path) -> Path:
    if "BASELINE_EVAL_EXTERNAL_ROOT" in os.environ:
        return Path(os.environ["BASELINE_EVAL_EXTERNAL_ROOT"]).resolve()
    if is_colab_runtime():
        return Path("/content/baseline_cqtdiff_external").resolve()
    return (root / "external").resolve()


In [ ]:
def path_config() -> dict[str, Path]:
    root = project_root()
    base_root = default_data_root(root)
    stage_root = base_root / "training_stages" / PIPELINE_STAGE_NAME
    source_preprocessed = Path(
        os.environ.get("BASELINE_EVAL_PREPROCESSED_DIR", stage_root / "preprocessed")
    ).resolve()
    source_masked = Path(os.environ.get("BASELINE_EVAL_MASKED_DIR", stage_root / "masked")).resolve()
    source_finetuned_ckpt_dir = Path(
        os.environ.get(
            "BASELINE_CQTDIFF_FINETUNED_CHECKPOINT_DIR",
            stage_root / "checkpoints" / SOURCE_FINETUNED_CHECKPOINT_DIRNAME,
        )
    ).resolve()
    external_root = default_external_root(root)
    audio_inpainting_dir = Path(
        os.environ.get("AUDIO_INPAINTING_DIR", external_root / "audio-inpainting-diffusion")
    ).resolve()
    cqt_diff_dir = Path(os.environ.get("CQT_DIFF_DIR", external_root / "CQTdiff")).resolve()
    ckpt_path = Path(
        os.environ.get(
            "AUDIO_INPAINTING_CQTDIFF_WEIGHTS",
            (
                base_root / "official_checkpoints" / "audio_inpainting_cqtdiff" / CKPT_FILENAME
                if is_colab_runtime()
                else audio_inpainting_dir / "experiments" / CKPT_FILENAME
            ),
        )
    ).resolve()
    return {
        "root": root,
        "base_root": base_root,
        "stage_root": stage_root,
        "dataset": base_root / "dataset",
        "source_preprocessed": source_preprocessed,
        "source_masked": source_masked,
        "preprocessed": stage_root / "preprocessed",
        "masked": stage_root / "masked",
        "source_finetuned_ckpt_dir": source_finetuned_ckpt_dir,
        "finetuned_ckpt_dir": stage_root / "checkpoints" / TARGET_MODEL,
        "finetuned_ckpt_path": stage_root / "checkpoints" / TARGET_MODEL / EXPECTED_FINETUNED_CHECKPOINT_FILENAME,
        "external": external_root,
        "audio_inpainting_dir": audio_inpainting_dir,
        "cqt_diff_dir": cqt_diff_dir,
        "ckpt_path": ckpt_path,
    }


In [ ]:
def ensure_python_requirements(root: Path) -> None:
    missing = [
        package_name
        for import_name, package_name in IMPORT_CHECKS.items()
        if importlib.util.find_spec(import_name) is None
    ]
    if not missing:
        info("Python requirements already importable; skip pip install.")
        return

    info("Missing imports: " + ", ".join(sorted(missing)))
    requirements = root / "requirements.txt"
    if not requirements.exists():
        fail(f"requirements.txt tidak ditemukan di {requirements}")

    if "torch" in missing:
        torch_index = os.environ.get("TORCH_CUDA_INDEX", "").strip()
        torch_cmd = [sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio"]
        if torch_index:
            torch_cmd.extend(["--index-url", torch_index])
        run_command(torch_cmd, cwd=root)

    run_command([sys.executable, "-m", "pip", "install", "-r", str(requirements)], cwd=root)


In [ ]:
def ensure_git_repo(path: Path, url: str) -> None:
    if (path / ".git").exists():
        info(f"Repository already exists; skip clone: {path}")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    run_command(["git", "clone", url, str(path)], cwd=path.parent)


In [ ]:
def ensure_external_repos(paths: dict[str, Path]) -> None:
    ensure_git_repo(paths["cqt_diff_dir"], "https://github.com/eloimoliner/CQTdiff.git")
    ensure_git_repo(
        paths["audio_inpainting_dir"],
        "https://github.com/eloimoliner/audio-inpainting-diffusion.git",
    )


In [ ]:
def ensure_checkpoint(ckpt_path: Path) -> None:
    if ckpt_path.exists() and ckpt_path.stat().st_size > 0:
        info(f"Checkpoint already exists; skip download: {ckpt_path}")
        return

    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = ckpt_path.with_suffix(ckpt_path.suffix + ".tmp")
    info(f"Downloading checkpoint to {ckpt_path}")
    try:
        urllib.request.urlretrieve(CKPT_URL, tmp_path)
        tmp_path.replace(ckpt_path)
    except Exception as exc:
        if tmp_path.exists():
            tmp_path.unlink()
        fail(
            "Gagal download checkpoint MusicNet CQTdiff+.\n"
            f"URL: {CKPT_URL}\n"
            f"Target: {ckpt_path}\n"
            f"Detail: {exc}"
        )


def find_finetuned_checkpoint(source_dir: Path) -> Path:
    candidates = [
        source_dir / EXPECTED_FINETUNED_CHECKPOINT_FILENAME,
        source_dir / "baseline_cqtdiff_finetune_best.pt",
    ]
    for candidate in candidates:
        if candidate.exists() and candidate.stat().st_size > 0:
            return candidate

    best_candidates = sorted(path for path in source_dir.glob("*_best.pt") if path.stat().st_size > 0)
    if len(best_candidates) == 1:
        return best_candidates[0]

    shown = "\n".join(f"  - {path.name}" for path in best_candidates[:10]) or "  - <none>"
    fail(
        f"Checkpoint fine-tuned tidak ditemukan di {source_dir}.\n"
        f"Butuh file {EXPECTED_FINETUNED_CHECKPOINT_FILENAME} atau satu file *_best.pt.\n"
        f"Kandidat yang ada:\n{shown}"
    )


def ensure_finetuned_checkpoint_layout(paths: dict[str, Path]) -> None:
    source_dir = paths["source_finetuned_ckpt_dir"]
    expected_path = paths["finetuned_ckpt_path"]
    if expected_path.exists() and expected_path.stat().st_size > 0:
        info(f"Fine-tuned checkpoint already OK: {expected_path}")
        return
    if not source_dir.is_dir():
        fail(
            f"Folder checkpoint fine-tuned tidak ditemukan: {source_dir}\n"
            "Set BASELINE_CQTDIFF_FINETUNED_CHECKPOINT_DIR jika lokasinya berbeda."
        )

    source_ckpt = find_finetuned_checkpoint(source_dir)
    expected_path.parent.mkdir(parents=True, exist_ok=True)
    if _same_path(expected_path, source_ckpt):
        return
    if expected_path.exists() or expected_path.is_symlink():
        expected_path.unlink()

    try:
        expected_path.symlink_to(source_ckpt)
        info(f"Created fine-tuned checkpoint link: {expected_path} -> {source_ckpt}")
    except OSError:
        shutil.copy2(source_ckpt, expected_path)
        info(f"Copied fine-tuned checkpoint for pipeline name: {expected_path}")


In [ ]:
def ensure_musicnet_dataset(dataset_dir: Path) -> None:
    """Skip download/extract when the Drive dataset folder is already present."""
    audio_dir = dataset_dir / "audio"
    if dataset_dir.exists() and any(dataset_dir.iterdir()):
        info(f"MusicNet dataset folder already exists; skip download/extract: {dataset_dir}")
        ensure_musicnet_metadata(dataset_dir)
        return

    if audio_dir.exists() and len(list(audio_dir.iterdir())) > 10:
        info(f"MusicNet dataset already exists; skip download: {audio_dir}")
        ensure_musicnet_metadata(dataset_dir)
        return

    dataset_dir.mkdir(parents=True, exist_ok=True)
    audio_dir.mkdir(parents=True, exist_ok=True)
    tar_path = dataset_dir / "musicnet.tar.gz"

    if tar_path.exists() and tar_path.stat().st_size > 10_000_000_000:
        info(f"Using existing MusicNet archive: {tar_path}")
    else:
        info("Downloading MusicNet audio files (~11GB archive)")

        def progress_hook(count: int, block_size: int, total_size: int) -> None:
            if total_size <= 0:
                return
            percent = min(count * block_size * 100 / total_size, 100)
            print(f"\r  Progress: {percent:.1f}%", end="", flush=True)

        urllib.request.urlretrieve(MUSICNET_URL, tar_path, progress_hook)
        print("\n  Download selesai.", flush=True)

    info(f"Extracting MusicNet WAV files to {audio_dir}")
    try:
        from tqdm import tqdm
    except Exception:
        tqdm = lambda items, desc=None: items

    with tarfile.open(tar_path, "r:gz") as tar:
        wav_members = [member for member in tar.getmembers() if member.name.endswith(".wav")]
        for member in tqdm(wav_members, desc="Extracting"):
            tar.extract(member, audio_dir)

    info(f"Dataset berhasil diekstrak ke {audio_dir}")
    ensure_musicnet_metadata(dataset_dir)


In [ ]:
def ensure_musicnet_metadata(dataset_dir: Path) -> None:
    metadata_path = dataset_dir / "musicnet_metadata.csv"
    if metadata_path.exists() and metadata_path.stat().st_size > 0:
        return
    try:
        info(f"Downloading MusicNet metadata to {metadata_path}")
        urllib.request.urlretrieve(MUSICNET_METADATA_URL, metadata_path)
    except Exception as exc:
        info(f"MusicNet metadata tidak bisa didownload; lanjut tanpa metadata. Detail: {exc}")


In [ ]:
def _same_path(left: Path, right: Path) -> bool:
    try:
        return left.resolve() == right.resolve()
    except FileNotFoundError:
        return left.absolute() == right.absolute()


In [ ]:
def ensure_stage_link(expected: Path, source: Path, label: str) -> None:
    """Expose root-level preprocessed/masked folders at the stage path used by the main pipeline."""
    if not source.is_dir():
        fail(
            f"Folder {label} asli tidak ditemukan: {source}\n"
            "Sediakan folder ini di root Drive, atau set env "
            f"BASELINE_EVAL_{label.upper()}_DIR ke path yang benar."
        )
    if _same_path(expected, source):
        return

    expected.parent.mkdir(parents=True, exist_ok=True)
    if expected.exists() or expected.is_symlink():
        if expected.is_symlink() and _same_path(expected, source):
            info(f"Stage {label} link already OK: {expected} -> {source}")
            return
        if expected.is_dir() and not any(expected.iterdir()):
            expected.rmdir()
        else:
            info(f"Stage {label} path already exists; using it as-is: {expected}")
            return

    try:
        expected.symlink_to(source, target_is_directory=True)
        info(f"Created stage {label} link: {expected} -> {source}")
    except OSError as exc:
        fail(
            f"Gagal membuat symlink untuk {label}.\n"
            f"Source: {source}\nExpected by pipeline: {expected}\nDetail: {exc}"
        )


In [ ]:
def prepare_stage_layout(paths: dict[str, Path]) -> None:
    ensure_stage_link(paths["preprocessed"], paths["source_preprocessed"], "preprocessed")
    ensure_stage_link(paths["masked"], paths["source_masked"], "masked")
    ensure_finetuned_checkpoint_layout(paths)


In [ ]:
def _first_missing(paths: list[Path], limit: int = 10) -> str:
    shown = "\n".join(f"  - {path}" for path in paths[:limit])
    if len(paths) > limit:
        shown += f"\n  ... dan {len(paths) - limit} path lain"
    return shown


In [ ]:
def validate_preprocessing_config(config_path: Path, metadata_path: Path) -> None:
    if not config_path.exists():
        fail(
            "preprocessing_config.json tidak ditemukan. Eval-only dihentikan supaya "
            f"pipeline tidak membuat preprocessing baru.\nPath: {config_path}"
        )
    try:
        cached = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception as exc:
        fail(f"preprocessing_config.json tidak bisa dibaca: {config_path}\nDetail: {exc}")

    expected = {
        "config_id": EXPERIMENT_CONFIG_ID,
        "target_sr": TARGET_SR,
        "segment_samples": SEGMENT_SAMPLES,
        "gap_durations_ms": GAP_DURATIONS_MS,
        "seed": DATASET_RANDOM_SEED,
    }
    mismatch = [
        f"{key}: cached={cached.get(key)!r}, expected={value!r}"
        for key, value in expected.items()
        if cached.get(key) != value
    ]
    if mismatch:
        fail(
            "Preprocessed artifacts tidak cocok dengan konfigurasi eval baseline ini.\n"
            + "\n".join(f"  - {item}" for item in mismatch)
            + f"\nConfig: {config_path}\nMetadata: {metadata_path}"
        )


In [ ]:
def read_metadata_rows(metadata_path: Path) -> list[dict[str, str]]:
    try:
        with metadata_path.open("r", newline="", encoding="utf-8") as handle:
            rows = list(csv.DictReader(handle))
    except UnicodeDecodeError:
        with metadata_path.open("r", newline="") as handle:
            rows = list(csv.DictReader(handle))
    if not rows:
        fail(f"metadata.csv kosong atau tidak valid: {metadata_path}")
    if "clean_path" not in rows[0]:
        fail(f"metadata.csv tidak memiliki kolom clean_path: {metadata_path}")
    return rows


In [ ]:
def repair_metadata_clean_paths(metadata_path: Path, preprocessed_dir: Path) -> None:
    rows = read_metadata_rows(metadata_path)
    needs_repair = False
    repaired = []

    for row in rows:
        clean_path = Path(row["clean_path"])
        candidate = preprocessed_dir / clean_path.name
        if not clean_path.exists() and candidate.exists():
            row = dict(row)
            row["clean_path"] = str(candidate)
            needs_repair = True
        repaired.append(row)

    if not needs_repair:
        return

    backup_path = metadata_path.with_suffix(metadata_path.suffix + ".bak")
    if not backup_path.exists():
        backup_path.write_text(metadata_path.read_text(encoding="utf-8"), encoding="utf-8")

    with metadata_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(repaired[0].keys()))
        writer.writeheader()
        writer.writerows(repaired)
    info(f"metadata.csv clean_path repaired for Colab paths. Backup: {backup_path}")


In [ ]:
def validate_eval_artifacts(paths: dict[str, Path]) -> None:
    required_dirs = [paths["dataset"], paths["preprocessed"], paths["masked"]]
    missing_dirs = [path for path in required_dirs if not path.is_dir()]
    if missing_dirs:
        fail(
            "Folder dataset/preprocessed/masked yang dibutuhkan belum ada:\n"
            + _first_missing(missing_dirs)
            + "\nJalankan preprocessing/restore stage dulu, lalu ulangi eval baseline."
        )

    metadata_path = paths["preprocessed"] / "metadata.csv"
    config_path = paths["preprocessed"] / "preprocessing_config.json"
    if not metadata_path.exists():
        fail(f"metadata.csv preprocessed tidak ditemukan: {metadata_path}")
    validate_preprocessing_config(config_path, metadata_path)
    repair_metadata_clean_paths(metadata_path, paths["preprocessed"])

    gap_dirs = [paths["masked"] / f"gap_{gap_ms}ms" for gap_ms in GAP_DURATIONS_MS]
    missing_gap_dirs = [path for path in gap_dirs if not path.is_dir()]
    if missing_gap_dirs:
        fail("Folder masked per gap belum lengkap:\n" + _first_missing(missing_gap_dirs))

    rows = read_metadata_rows(metadata_path)
    missing_clean = []
    missing_masked = []
    for row in rows:
        clean_path = Path(row["clean_path"])
        if not clean_path.exists():
            missing_clean.append(clean_path)
        filename = clean_path.name
        for gap_ms in GAP_DURATIONS_MS:
            masked_path = paths["masked"] / f"gap_{gap_ms}ms" / filename
            if not masked_path.exists():
                missing_masked.append(masked_path)

    if missing_clean:
        fail("File clean preprocessed dari metadata tidak ditemukan:\n" + _first_missing(missing_clean))
    if missing_masked:
        fail("File masked yang dibutuhkan metadata tidak ditemukan:\n" + _first_missing(missing_masked))

    info(
        f"Eval artifacts OK: {len(rows)} clean segments, "
        f"{len(GAP_DURATIONS_MS)} gap folders."
    )


In [ ]:
def configure_environment(paths: dict[str, Path]) -> None:
    os.environ["PROJECT_ROOT"] = str(paths["root"])
    os.environ["MUSIC_INPAINTING_ROOT"] = str(paths["base_root"])
    os.environ["CQT_DIFF_DIR"] = str(paths["cqt_diff_dir"])
    os.environ["AUDIO_INPAINTING_DIR"] = str(paths["audio_inpainting_dir"])
    os.environ["AUDIO_INPAINTING_CQTDIFF_WEIGHTS"] = str(paths["ckpt_path"])
    os.environ["OFFICIAL_CQTDIFF_ADAPTER"] = "official_audio_inpainting_cqtdiff_adapter"
    os.environ["MAID_ADAPTER"] = "official_maid_adapter"
    os.environ["RUN_PHASE"] = "eval"
    os.environ["RUN_MODELS"] = TARGET_MODEL
    os.environ.setdefault("PIPELINE_CPU_THREADS", "1")
    os.environ.setdefault("PIPELINE_TORCH_THREADS", "1")
    os.environ.setdefault("PIPELINE_NUM_WORKERS", "2")
    os.environ.setdefault("CQTDIFF_DIFFUSION_STEPS", "35")
    os.environ.setdefault("CQTDIFF_SIGMA_MIN", "1e-4")
    os.environ.setdefault("CQTDIFF_SIGMA_MAX", "1.0")
    os.environ.setdefault("CQTDIFF_SIGMA_DATA", "0.063")
    os.environ.setdefault("CQTDIFF_SCHURN", "10")

    for value in [paths["root"], paths["cqt_diff_dir"], paths["audio_inpainting_dir"]]:
        text = str(value)
        if text not in sys.path:
            sys.path.insert(0, text)


In [ ]:
def validate_cuda() -> None:
    try:
        import torch
    except Exception as exc:
        fail(f"PyTorch tidak bisa di-import setelah install dependency: {exc}")
    if torch.cuda.is_available():
        info(f"CUDA OK: {torch.cuda.get_device_name(0)}")
    else:
        info("CUDA tidak tersedia; evaluasi akan fallback ke CPU. Ini akan jauh lebih lambat.")


In [ ]:
def print_path_summary(paths: dict[str, Path]) -> None:
    info("Path configuration:")
    info(f"  project root      : {paths['root']}")
    info(f"  Drive/data root   : {paths['base_root']}")
    info(f"  stage root        : {paths['stage_root']}")
    info(f"  raw dataset       : {paths['dataset']}")
    info(f"  source preprocessed: {paths['source_preprocessed']}")
    info(f"  source masked      : {paths['source_masked']}")
    info(f"  pipeline preprocessed: {paths['preprocessed']}")
    info(f"  pipeline masked      : {paths['masked']}")
    info(f"  fine-tuned ckpt  : {paths['finetuned_ckpt_path']}")
    info(f"  source ft ckpt   : {paths['source_finetuned_ckpt_dir']}")
    info(f"  official ckpt     : {paths['ckpt_path']}")
    info(f"  external repos    : {paths['external']}")


In [ ]:
# Embedded copy of code_final_run_v2.py so this notebook is self-contained.
EMBEDDED_CODE_FINAL_RUN_V2 = '# Generated from: code_it.ipynb\n# Converted at: 2026-05-16T16:11:45.272Z\n# Next step (optional): refactor into modules & generate tests with RunCell\n# Quick start: pip install runcell\n\n# # 🎵 Music Audio Inpainting Pipeline v2\n# \n# Notebook ini mengimplementasikan pipeline hybrid SSL + Diffusion untuk music audio inpainting.\n# \n# ## Struktur Notebook\n# | Cell | Isi |\n# |------|-----|\n# | 1 | Cek GPU & Install dependencies |\n# | 2 | Mount Google Drive & setup folder |\n# | 3 | Download & Preprocessing MusicNet |\n# | 4 | Definisi FiLM Layer (improved init) |\n# | 5 | Definisi Fungsi Evaluasi (LSD gap-restricted dB, FAD, VISQOL_ODG) |\n# | 6 | Helper functions (memory management, checkpoint, group-aware split) |\n# | 6.5 | Dataset, DataLoader & Shared Utilities (mask, crossfade) |\n# | 6.6 | Training Loop (Reconstruction-based + CFG dropout) |\n# | 6.7 | Shared hybrid training helpers, shared decoder builders, checkpoint utilities |\n# | 7 | **BASELINE: CQT-Diff+ standalone TRAINED (tanpa encoder)** |\n# | 8A | Training CLAP + CQT-Diff+ |\n# | 8 | Evaluasi CLAP + CQT-Diff+ |\n# | 9A | Training CLAP + MAID |\n# | 9 | Evaluasi CLAP + MAID |\n# | 10A | Training AudioMAE + CQT-Diff+ |\n# | 10 | Evaluasi AudioMAE + CQT-Diff+ |\n# | 11A | Training AudioMAE + MAID |\n# | 11 | Evaluasi AudioMAE + MAID |\n# | 12 | Gabungkan & visualisasikan semua hasil |\n# \n# ## Model yang Dievaluasi\n# - **Baseline**: CQT-Diff+ (tanpa encoder SSL, tanpa FiLM)\n# - **Config 1**: CLAP + CQT-Diff+\n# - **Config 2**: CLAP + MAID\n# - **Config 3**: AudioMAE + CQT-Diff+\n# - **Config 4**: AudioMAE + MAID\n# \n# ## Alur Hybrid Terbaru\n# - Jalankan cell training hybrid terlebih dahulu untuk menghasilkan checkpoint best per kombinasi.\n# - Jika checkpoint sudah ada, cell training akan skip secara default kecuali `FORCE_RETRAIN = True`.\n# - Cell evaluasi hybrid selalu mencoba memuat checkpoint terlatih.\n# - Jika checkpoint hybrid belum ada, evaluasi akan berhenti dengan pesan yang jelas.\n# \n# ## Gap Duration yang Dievaluasi\n# 100ms, 300ms, 500ms, 750ms, 1200ms, 1700ms\n# \n# ---\n# ⚠️ **Pastikan Runtime → Change Runtime Type → GPU (T4) sebelum menjalankan!**\n\n\n# ---\n# ## CELL 1 — Cek GPU & Install Dependencies\n# **Jalankan cell ini pertama kali setiap sesi Colab baru.**\n\n\n# ============================================================\n# CELL 1: CEK GPU & INSTALL DEPENDENCIES\n# ============================================================\n# Cara pakai: Jalankan sekali di awal setiap sesi.\n# Estimasi waktu install: ~3-5 menit.\n# ============================================================\n\nimport subprocess\nimport sys\nimport shutil\nimport platform\nimport os\n\nCPU_THREAD_LIMIT = os.environ.get("PIPELINE_CPU_THREADS", "1")\nfor _thread_env in [\n    "OMP_NUM_THREADS",\n    "OPENBLAS_NUM_THREADS",\n    "MKL_NUM_THREADS",\n    "VECLIB_MAXIMUM_THREADS",\n    "NUMEXPR_NUM_THREADS",\n]:\n    os.environ.setdefault(_thread_env, CPU_THREAD_LIMIT)\n\n\ndef log_gpu_environment():\n    """Log environment GPU lengkap untuk dokumentasi run training."""\n    print("\\n" + "="*70)\n    print("GPU / ENVIRONMENT LOG")\n    print("="*70)\n    print("$ nvidia-smi -q  # notebook equivalent: !nvidia-smi -q")\n    if shutil.which("nvidia-smi"):\n        subprocess.run(["nvidia-smi", "-q"], check=False)\n    else:\n        print("nvidia-smi tidak ditemukan di environment ini.")\n\n    print(f"Python: {sys.version.split()[0]}")\n    print(f"Platform: {platform.platform()}")\n    try:\n        import torch as _torch\n        print(f"PyTorch: {_torch.__version__}")\n        print(f"CUDA available: {_torch.cuda.is_available()}")\n        print(f"CUDA version: {_torch.version.cuda if _torch.version.cuda else \'not available\'}")\n        if _torch.cuda.is_available():\n            props = _torch.cuda.get_device_properties(0)\n            print(f"GPU name: {props.name}")\n            print(f"VRAM: {props.total_memory / (1024**3):.2f} GiB")\n    except Exception as e:\n        print(f"Torch/CUDA detail belum bisa dibaca: {e}")\n    print("="*70 + "\\n")\n\n\nlog_gpu_environment()\n\n# --- Cek GPU ---\ntry:\n    import torch\n    if torch.cuda.is_available():\n        gpu_name = torch.cuda.get_device_name(0)\n        vram = torch.cuda.get_device_properties(0).total_memory / 1e9\n        print(f"✅ GPU aktif: {gpu_name}")\n        print(f"✅ VRAM tersedia: {vram:.1f} GB")\n    else:\n        print("❌ GPU tidak aktif! Pergi ke Runtime → Change Runtime Type → GPU")\n        sys.exit()\nexcept ImportError:\n    print("PyTorch belum terinstall, melanjutkan instalasi...")\n\nprint("\\n📦 Menginstall dependencies...")\n\npackages = [\n    # "librosa",       # Load audio, CQT, Mel-spectrogram\n    # "soundfile",     # Baca/tulis file audio\n    # "audioread",     # Backend untuk librosa\n    # "transformers",  # Load CLAP dan AudioMAE dari HuggingFace\n    # "accelerate",    # Loading model besar lebih efisien\n    # "einops",        # Operasi tensor yang lebih mudah dibaca\n    # "timm",          # Library model vision, dipakai AudioMAE\n    # "tqdm",          # Progress bar\n    # "pandas",        # Simpan hasil evaluasi ke tabel\n    # "matplotlib",    # Visualisasi hasil\n    # "scipy",         # Operasi sinyal\n    # "numpy",         # Operasi array numerik\n    # "resampy",       # Resampling audio berkualitas tinggi\n    # "torchvggish",   # VGGish embeddings untuk FAD legacy\n    # "git+https://github.com/ashvala/AQUA-tk.git",  # Legacy perceptual package, tidak dipakai untuk metrik final\n    # "visqol-lib-py", # ViSQOL perceptual quality metric\n]\n\nfor pkg in packages:\n    print(f"  Installing {pkg}...")\n    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], capture_output=True)\n\n# Clone repo CQT-Diff+ dari GitHub ke folder portabel.\n# Repo resmi: https://github.com/eloimoliner/CQTdiff\n# Vast.ai/local: gunakan PROJECT_ROOT (default: current working directory), bukan path Colab.\nPROJECT_ROOT = os.environ.get("PROJECT_ROOT", os.getcwd())\nEXTERNAL_DIR = os.path.join(PROJECT_ROOT, "external")\nCQT_DIFF_DIR = os.environ.get("CQT_DIFF_DIR", os.path.join(EXTERNAL_DIR, "CQTdiff"))\nAUDIO_INPAINTING_DIR = os.environ.get(\n    "AUDIO_INPAINTING_DIR", os.path.join(EXTERNAL_DIR, "audio-inpainting-diffusion")\n)\nAUDIO_MAE_DIR = os.environ.get("AUDIO_MAE_DIR", os.path.join(EXTERNAL_DIR, "AudioMAE"))\nMIDI2PERFORMANCE_DIR = os.environ.get(\n    "MIDI2PERFORMANCE_DIR",\n    os.path.join(EXTERNAL_DIR, "DDPM-Midi2Performance-Model"),\n)\nos.makedirs(EXTERNAL_DIR, exist_ok=True)\n\nif os.path.exists(os.path.join(CQT_DIFF_DIR, ".git")):\n    print(f"  CQT-Diff+ repository sudah ada: {CQT_DIFF_DIR}")\nelse:\n    print(f"  Cloning CQT-Diff+ repository ke {CQT_DIFF_DIR}...")\n    subprocess.run(\n        ["git", "clone", "https://github.com/eloimoliner/CQTdiff.git", CQT_DIFF_DIR],\n        check=False\n    )\nif os.path.exists(os.path.join(AUDIO_INPAINTING_DIR, ".git")):\n    print(f"  Audio-inpainting CQTdiff+ repository sudah ada: {AUDIO_INPAINTING_DIR}")\nelse:\n    print(f"  Cloning Audio-inpainting CQTdiff+ repository ke {AUDIO_INPAINTING_DIR}...")\n    subprocess.run(\n        ["git", "clone", "https://github.com/eloimoliner/audio-inpainting-diffusion.git", AUDIO_INPAINTING_DIR],\n        check=False\n    )\n\n# Tambahkan repo ke Python path agar bisa di-import\nif CQT_DIFF_DIR not in sys.path:\n    sys.path.insert(0, CQT_DIFF_DIR)\nif AUDIO_INPAINTING_DIR not in sys.path:\n    sys.path.insert(0, AUDIO_INPAINTING_DIR)\nif os.path.isdir(AUDIO_MAE_DIR) and AUDIO_MAE_DIR not in sys.path:\n    sys.path.insert(0, AUDIO_MAE_DIR)\nMIDI2PERFORMANCE_MAIN_DIR = os.path.join(MIDI2PERFORMANCE_DIR, "main")\nif os.path.isdir(MIDI2PERFORMANCE_MAIN_DIR) and MIDI2PERFORMANCE_MAIN_DIR not in sys.path:\n    sys.path.insert(0, MIDI2PERFORMANCE_MAIN_DIR)\n\nprint("\\nDependencies siap.")\nprint("\\nMode final: semua komponen model harus memakai implementasi asli.")\nprint("   Proxy/replika dinonaktifkan; pipeline akan berhenti jika model asli belum dikonfigurasi.")\n\n# ---\n# ## CELL 2 — Mount Google Drive & Setup Folder\n\n\n# ============================================================\n# CELL 2: MOUNT GOOGLE DRIVE & SETUP FOLDER\n# ============================================================\n# Cara pakai:\n# - IS_LOCAL = True  -> Jalankan di lokal (tidak perlu Google Drive)\n# - IS_LOCAL = False -> Jalankan di Google Colab dengan Google Drive\n# ============================================================\n\nimport os\n\n# --- PARAMETER -----------------------------------------------\nPIPELINE_STAGE_NAME = "code_v4_musicnet_cqtdiffplus_44k"  # Native audio-inpainting CQTdiff+: 44.1 kHz, 184184 samples.\nIS_LOCAL = True   # Vast.ai/local default. Ganti ke False hanya jika menggunakan Google Colab.\nPROJECT_ROOT = os.environ.get("PROJECT_ROOT", os.getcwd())\nBASE_LOCAL_ROOT = os.environ.get("MUSIC_INPAINTING_ROOT", os.path.join(PROJECT_ROOT, "music_inpainting"))\nLOCAL_ROOT = os.path.join(BASE_LOCAL_ROOT, "training_stages", PIPELINE_STAGE_NAME)\n# -------------------------------------------------------------\n\nif IS_LOCAL:\n    BASE_DATA_ROOT = BASE_LOCAL_ROOT\n    DATA_ROOT = LOCAL_ROOT\n    print(f"Mode lokal aktif. Base dataset folder: {BASE_DATA_ROOT}")\nelse:\n    from google.colab import drive\n    drive.mount(\'/content/drive\')\n    BASE_DATA_ROOT = "/content/drive/MyDrive/music_inpainting"\n    DATA_ROOT = os.path.join(BASE_DATA_ROOT, "training_stages", PIPELINE_STAGE_NAME)\n    print(f"Google Drive terpasang. Base dataset folder: {BASE_DATA_ROOT}")\n\nprint(f"Training stage: {PIPELINE_STAGE_NAME}")\nprint(f"Stage output root: {DATA_ROOT}")\n\nPATHS = {\n    # Dataset mentah dibagi antar stage agar tidak perlu download ulang.\n    "dataset":      os.path.join(BASE_DATA_ROOT, "dataset"),\n    # Artefak berikut diisolasi per stage agar pipeline lama tidak tertimpa.\n    "preprocessed": os.path.join(DATA_ROOT, "preprocessed"),\n    "masked":       os.path.join(DATA_ROOT, "masked"),\n    "outputs":      os.path.join(DATA_ROOT, "outputs"),\n    "results":      os.path.join(DATA_ROOT, "results"),\n    "checkpoints":  os.path.join(DATA_ROOT, "checkpoints"),\n    "logs":         os.path.join(DATA_ROOT, "logs"),\n    "plots":        os.path.join(DATA_ROOT, "plots"),\n}\n\n# Final thesis guardrail: do not use local proxy/reimplementation models.\n# CQTdiff+ and MAID need thin adapter modules because their official code does\n# not expose the exact FiLM-training interface used by this notebook.\nOFFICIAL_MODELS_ONLY = True\nOFFICIAL_CQTDIFF_ADAPTER = os.environ.get(\n    "OFFICIAL_CQTDIFF_ADAPTER", "official_audio_inpainting_cqtdiff_adapter"\n)\nMAID_ADAPTER = os.environ.get("MAID_ADAPTER", "official_maid_adapter")\n\n# Diagnostic experiment: train the official CQTdiff+ backbone together with\n# the adapter reconstruction head. This is heavier, but helps test whether the\n# silent/noisy gap comes from the frozen backbone + small head setup.\nCQTDIFF_TRAIN_BACKBONE_EXPERIMENT = False\nos.environ["CQTDIFF_TRAIN_BACKBONE"] = "1" if CQTDIFF_TRAIN_BACKBONE_EXPERIMENT else "0"\nprint(f"MusicNet CQTdiff+ train backbone experiment: CQTDIFF_TRAIN_BACKBONE={os.environ[\'CQTDIFF_TRAIN_BACKBONE\']}")\n\n\ndef validate_official_model_configuration():\n    if not OFFICIAL_MODELS_ONLY:\n        raise RuntimeError("Final pipeline harus berjalan dengan OFFICIAL_MODELS_ONLY=True.")\n\n    import importlib.util\n\n    required_adapters = {\n        "MusicNet CQTdiff+ original": OFFICIAL_CQTDIFF_ADAPTER,\n        "MAID original DDPM-Midi2Performance": MAID_ADAPTER,\n    }\n    missing = [\n        f"{label}: module \'{module_name}\'"\n        for label, module_name in required_adapters.items()\n        if importlib.util.find_spec(module_name) is None\n    ]\n    if missing:\n        raise RuntimeError(\n            "Final pipeline disetel official-only, tetapi adapter model asli belum tersedia:\\n"\n            + "\\n".join(f"  - {item}" for item in missing)\n            + "\\n\\nBuat module adapter tersebut di PYTHONPATH atau set env var "\n              "OFFICIAL_CQTDIFF_ADAPTER/MAID_ADAPTER/MIDI2PERFORMANCE_DIR ke nilai yang benar."\n        )\n\n\nvalidate_official_model_configuration()\n\nfor name, path in PATHS.items():\n    os.makedirs(path, exist_ok=True)\n    print(f"Folder \'{name}\': {path}")\n\n# Buat subfolder output untuk setiap model (termasuk baseline)\nALL_MODELS = ["baseline_cqtdiff", "baseline_cqtdiff_finetuned", "clap_cqtdiff", "clap_maid", "audiomae_cqtdiff", "audiomae_maid"]\nfor model in ALL_MODELS:\n    os.makedirs(os.path.join(PATHS["outputs"], model), exist_ok=True)\n    os.makedirs(os.path.join(PATHS["checkpoints"], model), exist_ok=True)\n    os.makedirs(os.path.join(PATHS["plots"], model), exist_ok=True)\n\n\n# Persistent Vast.ai-safe logging and environment capture.\nimport sys\nimport subprocess\nimport shutil\nimport platform\nimport json\nimport logging\nimport atexit\nfrom datetime import datetime\n\nRUN_ID = os.environ.get("RUN_ID", datetime.now().strftime("%Y%m%d_%H%M%S"))\nTRAINING_LOG_PATH = os.path.join(PATHS["logs"], "training_progress.log")\nERROR_LOG_PATH = os.path.join(PATHS["logs"], "errors.log")\nVALIDATION_LOG_PATH = os.path.join(PATHS["logs"], "validation_metrics.log")\nGPU_LOG_PATH = os.path.join(PATHS["logs"], "gpu_environment.json")\nNVIDIA_SMI_Q_PATH = os.path.join(PATHS["logs"], "nvidia_smi_q.txt")\n\n\nclass TeeStream:\n    """Mirror notebook stdout/stderr to persistent log files."""\n    _pipeline_tee = True\n\n    def __init__(self, stream, log_handle):\n        self.stream = stream\n        self.log_handle = log_handle\n\n    def write(self, data):\n        self.stream.write(data)\n        self.log_handle.write(data)\n        self.log_handle.flush()\n\n    def flush(self):\n        self.stream.flush()\n        self.log_handle.flush()\n\n    def isatty(self):\n        return getattr(self.stream, "isatty", lambda: False)()\n\n    @property\n    def encoding(self):\n        return getattr(self.stream, "encoding", "utf-8")\n\n\ndef configure_persistent_logging():\n    """Persist prints, errors, and logger records outside notebook output."""\n    global _TRAINING_LOG_HANDLE, _ERROR_LOG_HANDLE\n    os.makedirs(PATHS["logs"], exist_ok=True)\n\n    if not getattr(sys.stdout, "_pipeline_tee", False):\n        _TRAINING_LOG_HANDLE = open(TRAINING_LOG_PATH, "a", encoding="utf-8", buffering=1)\n        sys.stdout = TeeStream(sys.stdout, _TRAINING_LOG_HANDLE)\n        atexit.register(_TRAINING_LOG_HANDLE.close)\n\n    if not getattr(sys.stderr, "_pipeline_tee", False):\n        _ERROR_LOG_HANDLE = open(ERROR_LOG_PATH, "a", encoding="utf-8", buffering=1)\n        sys.stderr = TeeStream(sys.stderr, _ERROR_LOG_HANDLE)\n        atexit.register(_ERROR_LOG_HANDLE.close)\n\n    logger = logging.getLogger("music_inpainting")\n    logger.setLevel(logging.INFO)\n    logger.propagate = False\n    existing_paths = {getattr(h, "baseFilename", None) for h in logger.handlers}\n    for log_path in [TRAINING_LOG_PATH, VALIDATION_LOG_PATH, ERROR_LOG_PATH]:\n        if log_path not in existing_paths:\n            handler = logging.FileHandler(log_path, encoding="utf-8")\n            handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))\n            handler.setLevel(logging.INFO if log_path != ERROR_LOG_PATH else logging.ERROR)\n            logger.addHandler(handler)\n    return logger\n\n\nLOGGER = configure_persistent_logging()\nLOGGER.info("Run started | stage=%s | run_id=%s", PIPELINE_STAGE_NAME, RUN_ID)\n\n\ndef capture_environment_snapshot():\n    """Save nvidia-smi -q plus Python/CUDA/Torch/GPU details to logs/."""\n    env = {\n        "run_id": RUN_ID,\n        "stage": PIPELINE_STAGE_NAME,\n        "timestamp": datetime.now().isoformat(timespec="seconds"),\n        "python_version": sys.version,\n        "platform": platform.platform(),\n        "cuda_available": None,\n        "cuda_version": None,\n        "torch_version": None,\n        "gpu_name": None,\n        "vram_gib": None,\n    }\n\n    try:\n        import torch as _torch\n        env["torch_version"] = _torch.__version__\n        env["cuda_available"] = bool(_torch.cuda.is_available())\n        env["cuda_version"] = _torch.version.cuda\n        if _torch.cuda.is_available():\n            props = _torch.cuda.get_device_properties(0)\n            env["gpu_name"] = props.name\n            env["vram_gib"] = props.total_memory / (1024 ** 3)\n    except Exception as exc:\n        env["torch_error"] = repr(exc)\n\n    if shutil.which("nvidia-smi"):\n        result = subprocess.run(\n            ["nvidia-smi", "-q"],\n            capture_output=True,\n            text=True,\n            encoding="utf-8",\n            errors="replace",\n            check=False,\n        )\n        with open(NVIDIA_SMI_Q_PATH, "w", encoding="utf-8") as f:\n            f.write(result.stdout)\n            if result.stderr:\n                f.write("\\n--- STDERR ---\\n")\n                f.write(result.stderr)\n        env["nvidia_smi_q_path"] = NVIDIA_SMI_Q_PATH\n        env["nvidia_smi_returncode"] = result.returncode\n    else:\n        with open(NVIDIA_SMI_Q_PATH, "w", encoding="utf-8") as f:\n            f.write("nvidia-smi not found in this environment.\\n")\n        env["nvidia_smi_q_path"] = NVIDIA_SMI_Q_PATH\n        env["nvidia_smi_returncode"] = None\n\n    with open(GPU_LOG_PATH, "w", encoding="utf-8") as f:\n        json.dump(env, f, indent=2)\n\n    LOGGER.info("Environment snapshot saved: %s", GPU_LOG_PATH)\n    print(f"Persistent logs: {PATHS[\'logs\']}")\n    print(f"Environment snapshot: {GPU_LOG_PATH}")\n    print(f"nvidia-smi -q log: {NVIDIA_SMI_Q_PATH}")\n    return env\n\n\nENVIRONMENT_SNAPSHOT = capture_environment_snapshot()\n\nprint("\\nSemua folder stage siap!")\nprint(f"Root folder stage: {DATA_ROOT}")\n\n\n# ---\n# ## CELL 3 — Download & Preprocessing MusicNet\n# \n# ⚠️ **Jalankan SEKALI saja.** Hasil disimpan ke Drive dan tidak perlu diulang.\n\n\n# ============================================================\n# CELL 3: DOWNLOAD & PREPROCESSING MUSICNET\n# ============================================================\n# Cara pakai:\n# - Jalankan SEKALI saja. Hasil disimpan ke Google Drive.\n# - Set SKIP_IF_EXISTS = True jika preprocessing sudah pernah\n#   dilakukan sebelumnya untuk menghemat waktu.\n# - Estimasi waktu: ~15-30 menit\n# ============================================================\n\nimport os\nimport numpy as np\nimport pandas as pd\nimport librosa\nimport soundfile as sf\nfrom tqdm import tqdm\nimport urllib.request\nimport random\nimport time\n\n# ============================================================\n# KONFIGURASI\n# ============================================================\n\n# Set True jika preprocessing sudah pernah dijalankan\nSKIP_IF_EXISTS = True\n\n# Seed tetap untuk semua sampling dataset agar eksperimen reproducible\nDATASET_RANDOM_SEED = 42\n\n# Native MusicNet CQTdiff+ configuration from audio-inpainting-diffusion.\n# The default backbone is trained at 44.1 kHz and 184184 samples (~4.18 s).\nCQT_NATIVE_SR = int(os.environ.get("PIPELINE_TARGET_SR", "44100"))\nCQT_NATIVE_SAMPLES = int(os.environ.get("PIPELINE_SEGMENT_SAMPLES", "184184"))\nTARGET_SR = CQT_NATIVE_SR\nSEGMENT_SAMPLES = CQT_NATIVE_SAMPLES\nSEGMENT_DURATION = SEGMENT_SAMPLES / TARGET_SR\nEXPERIMENT_CONFIG_ID = (\n    f"musicnet_cqtdiffplus_sr{TARGET_SR}_n{SEGMENT_SAMPLES}_"\n    f"dur{SEGMENT_DURATION:.6f}s"\n)\n\n# Gap duration yang dievaluasi (dalam milidetik) — DIPERBARUI\nGAP_DURATIONS_MS = [100, 300, 500, 750, 1200, 1700]\n\n# Persentase dataset yang digunakan\nDATASET_FRACTION = 0.5\n\n# Jumlah sampel evaluasi (bisa override via env var untuk test run cepat)\nN_EVAL_SAMPLES = int(os.environ.get("N_EVAL_SAMPLES", "100"))\n# Eval final harus fresh by default: hapus WAV rekonstruksi lama sebelum inpaint,\n# lalu generate ulang semua output untuk checkpoint/model yang sedang diload.\nEVAL_REUSE_RECONSTRUCTIONS = os.environ.get("EVAL_REUSE_RECONSTRUCTIONS", "0").lower() in {"1", "true", "yes", "on"}\nEVAL_CLEAR_RECONSTRUCTIONS = os.environ.get("EVAL_CLEAR_RECONSTRUCTIONS", "1").lower() in {"1", "true", "yes", "on"}\nEVAL_GAP_POSITION = os.environ.get("EVAL_GAP_POSITION", "center").strip().lower()\nif EVAL_GAP_POSITION not in {"center", "random"}:\n    raise ValueError("EVAL_GAP_POSITION harus \'center\' atau \'random\'.")\nEVAL_RANDOM_GAP_MIN_CONTEXT_MS = int(os.environ.get("EVAL_RANDOM_GAP_MIN_CONTEXT_MS", "250"))\nEVAL_GAP_WINDOW_PERCEPTUAL = os.environ.get("EVAL_GAP_WINDOW_PERCEPTUAL", "0").lower() in {"1", "true", "yes", "on"}\nEVAL_GAP_WINDOW_PAD_MS = int(os.environ.get("EVAL_GAP_WINDOW_PAD_MS", "250"))\n\n# Jumlah segmen maksimal per lagu\nMAX_SEGMENTS_PER_FILE = 5\n\n# Metadata MusicNet dipakai untuk stratified sampling composer + instrument\nMUSICNET_METADATA_URL = "https://zenodo.org/record/5120004/files/musicnet_metadata.csv"\n\n\ndef gap_context_seconds(gap_ms, segment_samples=SEGMENT_SAMPLES, sr=TARGET_SR):\n    """Return left/right context for center-gap evaluation in seconds."""\n    gap_samples = int(round(sr * gap_ms / 1000))\n    context_samples = max(0, segment_samples - gap_samples)\n    return (context_samples / 2) / sr\n\n\ndef log_native_experiment_setup():\n    print("\\n" + "=" * 70)\n    print("NATIVE CQT-DIFF EXPERIMENT SETUP")\n    print("=" * 70)\n    print(f"Config id        : {EXPERIMENT_CONFIG_ID}")\n    print(f"Target SR        : {TARGET_SR} Hz")\n    print(f"Segment samples  : {SEGMENT_SAMPLES}")\n    print(f"Segment duration : {SEGMENT_DURATION:.3f} s")\n    print(f"Eval gap position: {EVAL_GAP_POSITION}")\n    print("Effective center-gap context per side:")\n    for gap_ms in GAP_DURATIONS_MS:\n        print(f"  - {gap_ms:4d} ms gap -> {gap_context_seconds(gap_ms):.3f} s left/right")\n    print("=" * 70 + "\\n")\n\n\nlog_native_experiment_setup()\n\n\n# ============================================================\n# FUNGSI DOWNLOAD\n# ============================================================\n\ndef download_musicnet_metadata():\n    """\n    Download metadata MusicNet untuk mengambil label composer dan instrument/ensemble.\n    Jika download gagal, pipeline tetap jalan dengan stratifikasi fallback "unknown".\n    """\n    dataset_dir = PATHS["dataset"]\n    metadata_path = os.path.join(dataset_dir, "musicnet_metadata.csv")\n\n    if os.path.exists(metadata_path):\n        return pd.read_csv(metadata_path)\n\n    try:\n        print("📥 Downloading MusicNet metadata...")\n        urllib.request.urlretrieve(MUSICNET_METADATA_URL, metadata_path)\n        return pd.read_csv(metadata_path)\n    except Exception as e:\n        print(f"⚠️ Metadata MusicNet tidak bisa didownload ({e}).")\n        print("   Stratified sampling akan fallback ke label composer/instrument=\'unknown\'.")\n        return pd.DataFrame()\n\n\ndef download_musicnet():\n    """\n    Download dataset MusicNet dari Zenodo.\n    MusicNet berisi 330 rekaman musik klasik.\n    Kita hanya pakai subset sesuai DATASET_FRACTION untuk efisiensi komputasi.\n    """\n    dataset_dir = PATHS["dataset"]\n    audio_dir = os.path.join(dataset_dir, "audio")\n\n    if os.path.exists(audio_dir) and len(os.listdir(audio_dir)) > 10:\n        print("✅ Dataset sudah ada di Drive, skip download.")\n        return audio_dir\n\n    os.makedirs(audio_dir, exist_ok=True)\n    MUSICNET_URL = "https://zenodo.org/record/5120004/files/musicnet.tar.gz"\n    tar_path = os.path.join(dataset_dir, "musicnet.tar.gz")\n    \n\n    if os.path.exists(tar_path) and os.path.getsize(tar_path) > 1e10:\n        print(f"Menggunakan archive MusicNet yang sudah ada: {tar_path}")\n    else:\n        print("?? Downloading MusicNet audio files...")\n        print("   Estimasi ukuran: ~11GB full archive")\n\n        def progress_hook(count, block_size, total_size):\n            percent = min(count * block_size * 100 / total_size, 100)\n            print(f"\\r  Progress: {percent:.1f}%", end="")\n\n        urllib.request.urlretrieve(MUSICNET_URL, tar_path, progress_hook)\n        print("\\n  Download selesai.")\n\n    print("  Mengekstrak MusicNet audio...")\n    import tarfile\n    with tarfile.open(tar_path, "r:gz") as tar:\n        wav_members = [m for m in tar.getmembers() if m.name.endswith(\'.wav\')]\n        for member in tqdm(wav_members, desc="Extracting"):\n            tar.extract(member, audio_dir)\n\n    print(f"? Dataset berhasil diekstrak ke {audio_dir}")\n    return audio_dir\n\n\n# ============================================================\n# FUNGSI STRATIFIED SAMPLING\n# ============================================================\n\ndef _track_id_from_path(audio_path):\n    """Ambil MusicNet track id dari nama file audio, misal 1727.wav -> 1727."""\n    return os.path.splitext(os.path.basename(audio_path))[0]\n\n\ndef _normalise_musicnet_metadata(metadata_df):\n    """Rapikan nama kolom metadata agar robust terhadap variasi source."""\n    if metadata_df is None or metadata_df.empty:\n        return pd.DataFrame()\n\n    meta = metadata_df.copy()\n    meta.columns = [str(c).strip().lower() for c in meta.columns]\n\n    if "id" not in meta.columns:\n        return pd.DataFrame()\n\n    meta["track_id"] = meta["id"].astype(str)\n    if "composer" not in meta.columns:\n        meta["composer"] = "unknown"\n\n    # MusicNet metadata umum memakai ensemble; beberapa mirror menyediakan instrument.\n    instrument_col = None\n    for col in ["instrument", "instruments", "ensemble"]:\n        if col in meta.columns:\n            instrument_col = col\n            break\n    meta["instrument"] = meta[instrument_col] if instrument_col else "unknown"\n\n    meta["composer"] = meta["composer"].fillna("unknown").astype(str)\n    meta["instrument"] = meta["instrument"].fillna("unknown").astype(str)\n    return meta[["track_id", "composer", "instrument"]].drop_duplicates("track_id")\n\n\ndef stratified_sample_dataframe(df, n_samples, stratify_cols, seed=DATASET_RANDOM_SEED):\n    """\n    Ambil sample deterministic dengan proporsi strata sebisa mungkin terjaga.\n    Dipakai untuk stratifikasi composer + instrument pada pemilihan file dan eval.\n    """\n    if len(df) == 0 or n_samples <= 0:\n        return df.iloc[0:0].copy()\n\n    n_samples = min(int(n_samples), len(df))\n    rng = np.random.default_rng(seed)\n\n    work = df.copy().reset_index(drop=True)\n    work["_sample_row_id"] = np.arange(len(work))\n    for col in stratify_cols:\n        if col not in work.columns:\n            work[col] = "unknown"\n        work[col] = work[col].fillna("unknown").astype(str)\n\n    grouped = list(work.groupby(stratify_cols, dropna=False, sort=True))\n    quotas = []\n    for group_key, group in grouped:\n        expected = len(group) * n_samples / len(work)\n        base = int(np.floor(expected))\n        quotas.append({\n            "key": group_key,\n            "group": group,\n            "quota": min(base, len(group)),\n            "fractional": expected - base,\n            "tie": rng.random(),\n        })\n\n    remaining = n_samples - sum(q["quota"] for q in quotas)\n    for q in sorted(quotas, key=lambda item: (-item["fractional"], item["tie"])):\n        if remaining <= 0:\n            break\n        capacity = len(q["group"]) - q["quota"]\n        if capacity > 0:\n            q["quota"] += 1\n            remaining -= 1\n\n    sampled_parts = []\n    for offset, q in enumerate(quotas):\n        if q["quota"] > 0:\n            sampled_parts.append(q["group"].sample(q["quota"], random_state=seed + offset))\n\n    if sampled_parts:\n        sampled = pd.concat(sampled_parts, ignore_index=True)\n    else:\n        sampled = work.iloc[0:0].copy()\n\n    if len(sampled) < n_samples:\n        missing = n_samples - len(sampled)\n        sampled_ids = set(sampled["_sample_row_id"]) if "_sample_row_id" in sampled.columns else set()\n        unsampled = work[~work["_sample_row_id"].isin(sampled_ids)]\n        if len(unsampled) > 0:\n            sampled = pd.concat([\n                sampled,\n                unsampled.sample(min(missing, len(unsampled)), random_state=seed + 999)\n            ], ignore_index=True)\n\n    return sampled.sample(frac=1.0, random_state=seed).drop(columns=["_sample_row_id"], errors="ignore").reset_index(drop=True)\n\n\ndef select_stratified_audio_files(all_audio_files, metadata_df, fraction, seed=DATASET_RANDOM_SEED):\n    """Pilih subset file audio dengan seed 42 dan stratifikasi composer + instrument."""\n    audio_df = pd.DataFrame({"audio_path": sorted(all_audio_files)})\n    audio_df["source_file"] = audio_df["audio_path"].apply(os.path.basename)\n    audio_df["track_id"] = audio_df["audio_path"].apply(_track_id_from_path)\n\n    meta = _normalise_musicnet_metadata(metadata_df)\n    if not meta.empty:\n        audio_df = audio_df.merge(meta, on="track_id", how="left")\n\n    for col in ["composer", "instrument"]:\n        if col not in audio_df.columns:\n            audio_df[col] = "unknown"\n        audio_df[col] = audio_df[col].fillna("unknown").astype(str)\n\n    n_files = max(1, int(round(len(audio_df) * fraction)))\n    selected = stratified_sample_dataframe(\n        audio_df,\n        n_samples=n_files,\n        stratify_cols=["composer", "instrument"],\n        seed=seed,\n    )\n    return selected\n\n\n# ============================================================\n# FUNGSI PREPROCESSING\n# ============================================================\n\ndef preprocess_audio(audio_path):\n    """\n    Preprocessing standar untuk satu file audio:\n    1. Load audio\n    2. Konversi ke mono\n    3. Resample ke TARGET_SR (44.1 kHz, native MusicNet CQTdiff+)\n    4. Normalisasi RMS ke target level (-23 dBFS approx)\n\n    Menggunakan RMS normalization alih-alih peak normalization\n    agar dynamic range antar segmen tetap terjaga — penting untuk\n    ViSQOL yang sensitif terhadap loudness statistics.\n    """\n    audio, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True)\n    rms = np.sqrt(np.mean(audio ** 2))\n    target_rms = 0.07  # approx -23 dBFS\n    if rms > 1e-6:\n        audio = audio * (target_rms / rms)\n    audio = np.clip(audio, -1.0, 1.0)\n    return audio\n\n\ndef split_into_segments(audio, file_seed=0):\n    """\n    Potong audio panjang jadi segmen native MusicNet CQTdiff+ (~4.18 detik).\n    Ambil maksimal MAX_SEGMENTS_PER_FILE secara random\n    agar dataset lebih beragam.\n\n    file_seed: per-file seed agar hasil reproducible di setiap run.\n    """\n    rng = random.Random(file_seed)\n    if len(audio) < SEGMENT_SAMPLES:\n        return []\n    possible_starts = list(range(0, len(audio) - SEGMENT_SAMPLES + 1, SEGMENT_SAMPLES))\n    n_segments = min(MAX_SEGMENTS_PER_FILE, len(possible_starts))\n    selected_starts = rng.sample(possible_starts, n_segments)\n    return [audio[s : s + SEGMENT_SAMPLES] for s in selected_starts]\n\n\ndef compute_gap_bounds(audio_length, gap_ms, sr=TARGET_SR, gap_start=None):\n    """Compute sample-exact gap bounds within one native segment."""\n    gap_samples = int(round(sr * gap_ms / 1000))\n    if gap_samples <= 0 or gap_samples >= audio_length:\n        raise ValueError(\n            f"Gap {gap_ms}ms tidak valid untuk audio_length={audio_length}, sr={sr}."\n        )\n    if gap_start is None:\n        center = audio_length // 2\n        gap_start = center - gap_samples // 2\n    gap_start = int(gap_start)\n    gap_end = gap_start + gap_samples\n    if gap_start < 0 or gap_end > audio_length:\n        raise ValueError(\n            f"Gap bounds keluar audio: start={gap_start}, end={gap_end}, length={audio_length}."\n        )\n    return gap_start, gap_end\n\n\ndef build_gap_mask_array(audio_length, gap_ms, sr=TARGET_SR, gap_start=None):\n    gap_start, gap_end = compute_gap_bounds(audio_length, gap_ms, sr=sr, gap_start=gap_start)\n    mask = np.zeros(audio_length, dtype=bool)\n    mask[gap_start:gap_end] = True\n    return mask, gap_start, gap_end\n\n\ndef apply_gap_mask(audio_segment, gap_ms, sr=TARGET_SR):\n    """\n    Buat versi audio dengan gap di tengah segmen.\n\n    Gap ditempatkan di tengah agar model punya konteks\n    yang seimbang di kiri dan kanan.\n\n    Menggunakan int(round(...)) agar gap_samples selalu konsisten\n    antara preprocessing dan inference — menghindari off-by-one.\n\n    Returns:\n        masked_audio : audio dengan gap diisi nol\n        mask         : boolean array (True = posisi gap)\n        gap_start    : indeks awal gap\n        gap_end      : indeks akhir gap\n    """\n    mask, gap_start, gap_end = build_gap_mask_array(len(audio_segment), gap_ms, sr=sr)\n\n    masked_audio = audio_segment.copy()\n    masked_audio[gap_start:gap_end] = 0.0\n\n    return masked_audio, mask, gap_start, gap_end\n\n\n# ============================================================\n# JALANKAN PREPROCESSING\n# ============================================================\n\n\n\ndef write_preprocessing_timing(status, total_seconds, total_segments=0):\n    row = {\n        "stage": globals().get("PIPELINE_STAGE_NAME", "code_v3"),\n        "status": status,\n        "dataset_fraction": DATASET_FRACTION,\n        "total_segments": int(total_segments or 0),\n        "preprocessing_seconds": float(total_seconds or 0.0),\n        "preprocessing_time": f"{float(total_seconds or 0.0):.1f}s",\n        "timestamp": pd.Timestamp.now().isoformat(),\n    }\n    timing_path = os.path.join(PATHS["results"], "preprocessing_timing.csv")\n    pd.DataFrame([row]).to_csv(timing_path, index=False)\n    print(f"Preprocessing timing saved: {timing_path}")\n\n\ndef _current_preprocessing_config():\n    return {\n        "config_id": EXPERIMENT_CONFIG_ID,\n        "target_sr": int(TARGET_SR),\n        "segment_samples": int(SEGMENT_SAMPLES),\n        "segment_duration": float(SEGMENT_DURATION),\n        "gap_durations_ms": list(GAP_DURATIONS_MS),\n        "dataset_fraction": float(DATASET_FRACTION),\n        "max_segments_per_file": int(MAX_SEGMENTS_PER_FILE),\n        "seed": int(DATASET_RANDOM_SEED),\n    }\n\n\ndef _preprocessing_config_matches(config_path, metadata_path):\n    """Return True only if cached preprocessing belongs to this native CQT setup."""\n    if not os.path.exists(config_path) or not os.path.exists(metadata_path):\n        return False\n    try:\n        with open(config_path, "r", encoding="utf-8") as f:\n            cached = json.load(f)\n        expected = _current_preprocessing_config()\n        keys = ["config_id", "target_sr", "segment_samples", "gap_durations_ms", "seed"]\n        if any(cached.get(key) != expected.get(key) for key in keys):\n            return False\n        meta = pd.read_csv(metadata_path)\n        if meta.empty:\n            return False\n        if "sample_rate" in meta.columns and not (meta["sample_rate"].astype(int) == TARGET_SR).all():\n            return False\n        if "n_samples" in meta.columns and not (meta["n_samples"].astype(int) == SEGMENT_SAMPLES).all():\n            return False\n        return True\n    except Exception as exc:\n        print(f"⚠️ Cached preprocessing config tidak valid ({exc}); preprocessing akan dibuat ulang.")\n        return False\n\n\npreprocessing_start = time.perf_counter()\n\npreprocessed_flag = os.path.join(PATHS["preprocessed"], ".done")\npreprocessed_metadata = os.path.join(PATHS["preprocessed"], "metadata.csv")\npreprocessed_config = os.path.join(PATHS["preprocessed"], "preprocessing_config.json")\nexpected_mask_dirs = [\n    os.path.join(PATHS["masked"], f"gap_{gap_ms}ms")\n    for gap_ms in GAP_DURATIONS_MS\n]\npreprocessed_artifacts_ready = (\n    _preprocessing_config_matches(preprocessed_config, preprocessed_metadata)\n    and all(os.path.isdir(path) for path in expected_mask_dirs)\n)\n\nif SKIP_IF_EXISTS and preprocessed_artifacts_ready:\n    print("✅ Preprocessing sudah selesai sebelumnya, skip.")\n    print("   Set SKIP_IF_EXISTS = False untuk memaksa preprocessing ulang.")\n    if not os.path.exists(preprocessed_flag):\n        with open(preprocessed_flag, "w") as f:\n            f.write("done")\n    write_preprocessing_timing("skipped", time.perf_counter() - preprocessing_start, 0)\nelse:\n    if os.path.exists(preprocessed_metadata) and not _preprocessing_config_matches(preprocessed_config, preprocessed_metadata):\n        print("♻️ Preprocessing lama tidak cocok dengan native CQT config; membersihkan artifact stage.")\n        for stale_dir in [PATHS["preprocessed"], PATHS["masked"]]:\n            if os.path.isdir(stale_dir):\n                shutil.rmtree(stale_dir)\n            os.makedirs(stale_dir, exist_ok=True)\n\n    audio_dir = download_musicnet()\n    metadata_df = download_musicnet_metadata()\n\n    all_audio_files = []\n    for root, dirs, files in os.walk(audio_dir):\n        for f in files:\n            if f.endswith(\'.wav\') or f.endswith(\'.flac\'):\n                all_audio_files.append(os.path.join(root, f))\n\n    selected_table = select_stratified_audio_files(\n        all_audio_files,\n        metadata_df=metadata_df,\n        fraction=DATASET_FRACTION,\n        seed=DATASET_RANDOM_SEED,\n    )\n\n    print(f"\\n📊 Total file audio: {len(all_audio_files)}")\n    print(f"📊 File yang dipakai ({DATASET_FRACTION:.0%}): {len(selected_table)}")\n    print(f"📊 Sample rate target: {TARGET_SR} Hz")\n    print(f"📊 Random seed: {DATASET_RANDOM_SEED} | Stratified by composer + instrument")\n    print(f"📊 Gap durations: {GAP_DURATIONS_MS} ms")\n    if {"composer", "instrument"}.issubset(selected_table.columns):\n        print("\\n📊 Distribusi strata terpilih (top 10):")\n        print(selected_table.groupby(["composer", "instrument"]).size().sort_values(ascending=False).head(10).to_string())\n\n    segment_metadata = []\n    segment_id = 0\n\n    print("\\n🔄 Memulai preprocessing...")\n    for file_idx, (_, file_row) in enumerate(tqdm(selected_table.iterrows(), total=len(selected_table), desc="Preprocessing files")):\n        filepath = file_row["audio_path"]\n        try:\n            audio = preprocess_audio(filepath)\n            segments = split_into_segments(audio, file_seed=DATASET_RANDOM_SEED + file_idx)\n\n            for segment in segments:\n                clean_filename = f"seg_{segment_id:05d}.wav"\n                clean_path = os.path.join(PATHS["preprocessed"], clean_filename)\n                sf.write(clean_path, segment, TARGET_SR)\n\n                for gap_ms in GAP_DURATIONS_MS:\n                    masked_audio, mask, gap_start, gap_end = apply_gap_mask(segment, gap_ms)\n                    masked_dir = os.path.join(PATHS["masked"], f"gap_{gap_ms}ms")\n                    os.makedirs(masked_dir, exist_ok=True)\n                    sf.write(os.path.join(masked_dir, clean_filename), masked_audio, TARGET_SR)\n\n                composer = str(file_row.get("composer", "unknown"))\n                instrument = str(file_row.get("instrument", "unknown"))\n                segment_metadata.append({\n                    "segment_id": segment_id,\n                    "source_file": os.path.basename(filepath),\n                    "track_id": file_row.get("track_id", _track_id_from_path(filepath)),\n                    "composer": composer,\n                    "instrument": instrument,\n                    "stratify_key": f"{composer}__{instrument}",\n                    "clean_path": clean_path,\n                    "duration_s": SEGMENT_DURATION,\n                    "sample_rate": TARGET_SR,\n                    "n_samples": len(segment),\n                })\n                segment_id += 1\n\n        except Exception as e:\n            print(f"\\n⚠️ Gagal memproses {filepath}: {e}")\n            continue\n\n    meta_df = pd.DataFrame(segment_metadata)\n    meta_path = os.path.join(PATHS["preprocessed"], "metadata.csv")\n    meta_df.to_csv(meta_path, index=False)\n\n    with open(preprocessed_flag, \'w\') as f:\n        f.write("done")\n    with open(preprocessed_config, "w", encoding="utf-8") as f:\n        json.dump(_current_preprocessing_config(), f, indent=2)\n\n    print(f"\\n✅ Preprocessing selesai! Total segmen: {segment_id}")\n    print(f"   Metadata: {meta_path}")\n\n# ---\n# ## CELL 4 — Definisi FiLM Layer\n\n\n# ============================================================\n# CELL 4: DEFINISI FILM LAYER\n# ============================================================\n# FiLM (Feature-wise Linear Modulation) adalah jembatan antara\n# encoder SSL dan decoder diffusion.\n#\n# Cara kerja:\n#   output = gamma * fitur_decoder + beta\n#   gamma dan beta dihasilkan dari latent encoder\n#\n# CATATAN: FiLM hanya digunakan oleh Cell 8-11 (kombinasi hybrid).\n# Cell 7 (baseline) tidak menggunakan FiLM sama sekali.\n# ============================================================\n\nimport torch\nimport torch.nn as nn\n\n\nclass FiLMLayer(nn.Module):\n    """\n    Feature-wise Linear Modulation Layer.\n\n    Menyuntikkan informasi encoder ke dalam decoder\n    dengan memodulasi fitur-fitur internal decoder.\n\n    Support 2D (B, D) dan 3D (B, T, D) decoder features.\n    Untuk 3D, gamma/beta di-broadcast ke semua timestep.\n\n    Init: gamma=1, beta=0 (identity transform) tapi gradien non-zero.\n    """\n\n    def __init__(self, encoder_dim: int, decoder_feature_dim: int, hidden_dim: int = None):\n        super(FiLMLayer, self).__init__()\n\n        if hidden_dim is None:\n            hidden_dim = (encoder_dim + decoder_feature_dim) // 2\n\n        self.decoder_feature_dim = decoder_feature_dim\n\n        self.proj = nn.Sequential(\n            nn.Linear(encoder_dim, hidden_dim),\n            nn.SiLU(),\n            nn.Linear(hidden_dim, 2 * decoder_feature_dim),\n        )\n\n        nn.init.zeros_(self.proj[-1].weight)\n        with torch.no_grad():\n            self.proj[-1].bias[:decoder_feature_dim].fill_(1.0)\n            self.proj[-1].bias[decoder_feature_dim:].fill_(0.0)\n\n    def forward(self, encoder_latent: torch.Tensor, decoder_features: torch.Tensor):\n        """\n        Args:\n            encoder_latent  : (B, encoder_dim)\n            decoder_features: (B, D) atau (B, T, D) — last dim = decoder_feature_dim\n        Returns:\n            Fitur decoder yang sudah dimodulasi, shape sama dengan input\n        """\n        # gamma, beta: (B, decoder_feature_dim)\n        gamma, beta = self.proj(encoder_latent).chunk(2, dim=-1)\n\n        # Broadcast ke sequence dimension kalau decoder_features 3D\n        if decoder_features.dim() == 3:\n            gamma = gamma.unsqueeze(1)  # (B, 1, D)\n            beta = beta.unsqueeze(1)    # (B, 1, D)\n\n        return gamma * decoder_features + beta\n\n\n# Konfigurasi dimensi FiLM per kombinasi\n# encoder_dim        : ukuran output latent encoder\n# decoder_feature_dim: ukuran fitur internal decoder yang dimodulasi\nFILM_CONFIGS = {\n    "baseline_cqtdiff_finetuned": {"encoder_dim": 512, "decoder_feature_dim": 256},\n    "clap_cqtdiff":     {"encoder_dim": 512, "decoder_feature_dim": 256},\n    "clap_maid":        {"encoder_dim": 512, "decoder_feature_dim": 512},\n    "audiomae_cqtdiff": {"encoder_dim": 768, "decoder_feature_dim": 256},\n    "audiomae_maid":    {"encoder_dim": 768, "decoder_feature_dim": 512},\n    # baseline_cqtdiff tidak ada di sini karena tidak pakai FiLM\n}\n\nprint("✅ FiLMLayer berhasil didefinisikan!")\nprint("\\n📋 Konfigurasi FiLM per kombinasi:")\nfor combo, cfg in FILM_CONFIGS.items():\n    note = " (zero encoder — ablation)" if combo == "baseline_cqtdiff_finetuned" else ""\n    print(f"   {combo}: encoder_dim={cfg[\'encoder_dim\']}, "\n          f"decoder_feature_dim={cfg[\'decoder_feature_dim\']}{note}")\nprint("   baseline_cqtdiff: tidak menggunakan FiLM (pretrained only)")\n\n# ---\n# ## CELL 5 — Fungsi Evaluasi (LSD, FAD, VISQOL_ODG)\n\n\n# ============================================================\n# CELL 5: FUNGSI EVALUASI\n# ============================================================\n# Mendefinisikan 3 metrik evaluasi:\n#\n# 1. LSD (Log Spectral Distance)\n#    - Mengukur perbedaan spektral antara audio asli vs rekonstruksi\n#    - Lebih rendah = lebih baik\n#    - Range: 0 (sempurna) hingga ~5 (buruk)\n#\n# 2. FAD (Frechet Audio Distance)\n#    - Mengukur jarak distribusi audio asli vs rekonstruksi\n#    - Lebih rendah = lebih baik\n#    - Dihitung per set, bukan per sample\n#\n# 3. VISQOL_ODG (ViSQOL Objective Difference Grade)\n#    - Mengukur kualitas perseptual berdasarkan reference vs degraded audio\n#    - Skala ODG-like: 0 (imperceptible) hingga -4 (sangat buruk)\n#    - Lebih tinggi (mendekati 0) = lebih baik\n# ============================================================\n\nimport numpy as np\nimport librosa\nimport pandas as pd\n\n_VISQOL_FALLBACK_WARNED = False\n# Aktifkan akselerasi GPU untuk metrik yang sudah punya implementasi torch.\n# FAD/ViSQOL/GstPEAQ tetap memakai backend resmi CPU agar definisi metrik tidak berubah.\nEVAL_USE_GPU = os.environ.get("EVAL_USE_GPU", "1").lower() in {"1", "true", "yes", "on"}\nEVAL_VISQOL_BACKEND = os.environ.get("EVAL_VISQOL_BACKEND", "visqol")\nEVAL_USE_GSTPEAQ = os.environ.get("EVAL_USE_GSTPEAQ", "1").lower() in {"1", "true", "yes"}\nFAD_USE_VGGISH_PCA = os.environ.get("FAD_USE_VGGISH_PCA", "0").lower() in {"1", "true", "yes", "on"}\nFAD_DEBUG_STATS = os.environ.get("FAD_DEBUG_STATS", "1").lower() in {"1", "true", "yes", "on"}\nGSTPEAQ_DIR = os.environ.get("GSTPEAQ_DIR", os.path.join(EXTERNAL_DIR, "gstpeaq"))\nGSTPEAQ_BIN = os.environ.get("GSTPEAQ_BIN", "")\nGSTPEAQ_PLUGIN = os.environ.get("GSTPEAQ_PLUGIN", "")\nGSTPEAQ_ADVANCED = os.environ.get("GSTPEAQ_ADVANCED", "0").lower() in {"1", "true", "yes"}\n\n\ndef compute_lsd(original: np.ndarray, reconstructed: np.ndarray,\n                sr: int = TARGET_SR, n_fft: int = 2048, hop_length: int = 512,\n                gap_start: int = None, gap_end: int = None, frame_pad: int = 2):\n    """\n    Hitung Log Spectral Distance (LSD) dalam dB.\n\n    LSD dihitung hanya pada gap region (+ frame_pad frame di tiap sisi)\n    agar metrik benar-benar mengukur kualitas inpainting, bukan bagian\n    non-gap yang sudah diketahui.\n    """\n    n = min(len(original), len(reconstructed))\n    o, r = original[:n], reconstructed[:n]\n\n    O = np.abs(librosa.stft(o, n_fft=n_fft, hop_length=hop_length)) ** 2\n    R = np.abs(librosa.stft(r, n_fft=n_fft, hop_length=hop_length)) ** 2\n\n    eps = max(1e-10, 1e-6 * O.max())\n    log_diff = 10.0 * (np.log10(O + eps) - np.log10(R + eps))  # dB\n\n    if gap_start is not None and gap_end is not None:\n        f_start = max(0, gap_start // hop_length - frame_pad)\n        f_end = min(O.shape[1], gap_end // hop_length + frame_pad + 1)\n        log_diff = log_diff[:, f_start:f_end]\n\n    lsd = np.mean(np.sqrt(np.mean(log_diff ** 2, axis=0)))\n    return float(lsd)\n\n\ndef _metric_gap_bounds(audio_len, gap_ms, sr=TARGET_SR, region=None):\n    if region is not None:\n        return int(region["gap_start"]), int(region["gap_end"])\n    return compute_gap_bounds(audio_len, gap_ms, sr=sr)\n\n\ndef _slice_metric_region(original, reconstructed, gap_start, gap_end):\n    n = min(len(original), len(reconstructed))\n    gap_start = max(0, min(int(gap_start), n))\n    gap_end = max(gap_start, min(int(gap_end), n))\n    ref = np.asarray(original[:n], dtype=np.float64)[gap_start:gap_end]\n    est = np.asarray(reconstructed[:n], dtype=np.float64)[gap_start:gap_end]\n    return ref, est\n\n\ndef compute_gap_snr(original, reconstructed, gap_start, gap_end):\n    ref, est = _slice_metric_region(original, reconstructed, gap_start, gap_end)\n    if len(ref) == 0:\n        return np.nan\n    noise = ref - est\n    return float(10.0 * np.log10((np.sum(ref ** 2) + 1e-12) / (np.sum(noise ** 2) + 1e-12)))\n\n\ndef compute_gap_si_sdr(original, reconstructed, gap_start, gap_end):\n    ref, est = _slice_metric_region(original, reconstructed, gap_start, gap_end)\n    if len(ref) == 0:\n        return np.nan\n    ref = ref - np.mean(ref)\n    est = est - np.mean(est)\n    ref_energy = np.sum(ref ** 2) + 1e-12\n    target = (np.sum(est * ref) / ref_energy) * ref\n    error = est - target\n    return float(10.0 * np.log10((np.sum(target ** 2) + 1e-12) / (np.sum(error ** 2) + 1e-12)))\n\n\ndef compute_gap_mel_distance(original, reconstructed, sr=TARGET_SR, gap_start=None, gap_end=None,\n                             n_fft=1024, hop_length=256, n_mels=64):\n    ref, est = _slice_metric_region(original, reconstructed, gap_start, gap_end)\n    if len(ref) == 0:\n        return np.nan\n    if len(ref) < n_fft:\n        pad = n_fft - len(ref)\n        ref = np.pad(ref, (0, pad))\n        est = np.pad(est, (0, pad))\n    ref_mel = librosa.feature.melspectrogram(\n        y=ref.astype(np.float32), sr=sr, n_fft=n_fft, hop_length=hop_length,\n        n_mels=n_mels, power=2.0\n    )\n    est_mel = librosa.feature.melspectrogram(\n        y=est.astype(np.float32), sr=sr, n_fft=n_fft, hop_length=hop_length,\n        n_mels=n_mels, power=2.0\n    )\n    ref_peak = max(float(np.max(ref_mel)), 1e-10)\n    ref_db = librosa.power_to_db(ref_mel, ref=ref_peak)\n    est_db = librosa.power_to_db(est_mel, ref=ref_peak)\n    frames = min(ref_db.shape[-1], est_db.shape[-1])\n    return float(np.mean(np.abs(ref_db[..., :frames] - est_db[..., :frames])))\n\n\ndef compute_gap_window_visqol_odg(original, reconstructed, sr=TARGET_SR, gap_start=None, gap_end=None,\n                                  pad_ms=EVAL_GAP_WINDOW_PAD_MS):\n    pad = int(round(sr * pad_ms / 1000))\n    n = min(len(original), len(reconstructed))\n    start = max(0, int(gap_start) - pad)\n    end = min(n, int(gap_end) + pad)\n    if end <= start:\n        return np.nan\n    try:\n        return compute_visqol_odg(original[start:end], reconstructed[start:end], sr)\n    except Exception as exc:\n        print(f"    Gap-window ViSQOL gagal ({exc}); nilai diisi NaN.")\n        return np.nan\n\n\ndef extract_fad_features(audio_list: list, sr: int = TARGET_SR):\n    """\n    Ekstrak fitur untuk Frechet Audio Distance.\n\n    FAD selalu menggunakan VGGish embeddings di CPU agar skala metrik konsisten\n    dengan pipeline FAD legacy. Tidak ada fallback log-mel untuk FAD.\n    Semua patch embedding VGGish dari seluruh set dipakai sebagai sampel FAD;\n    embedding tidak dirata-rata per file.\n    """\n    features = []\n    try:\n        import inspect\n        import torch as _torch\n        import torchvggish\n        try:\n            from torchvggish import vggish_input as _vggish_input\n        except Exception:\n            _vggish_input = getattr(torchvggish, "vggish_input", None)\n\n        _device = _torch.device("cpu")\n        vggish_kwargs = {}\n        try:\n            sig = inspect.signature(torchvggish.vggish)\n            if "postprocess" in sig.parameters:\n                # FAD libraries commonly use raw VGGish embeddings\n                # (equivalent to use_pca=False). The PCA+8-bit YouTube-8M\n                # postprocess space has a 0..255 scale and can inflate FAD\n                # into tens of thousands for otherwise reasonable audio.\n                vggish_kwargs["postprocess"] = bool(FAD_USE_VGGISH_PCA)\n        except (TypeError, ValueError):\n            pass\n\n        _vggish = torchvggish.vggish(**vggish_kwargs).to(_device).eval()\n        postprocess_active = bool(vggish_kwargs.get("postprocess", False))\n        postprocess_active = postprocess_active or bool(getattr(_vggish, "postprocess", False))\n        postprocess_active = postprocess_active or bool(getattr(_vggish, "pproc", None) is not None)\n        if postprocess_active and not FAD_USE_VGGISH_PCA:\n            raise RuntimeError(\n                "torchvggish tetap mengaktifkan VGGish post-processing walau "\n                "FAD_USE_VGGISH_PCA=0. Ini berpotensi membuat skala FAD meledak. "\n                "Gunakan package torchvggish yang mendukung vggish(postprocess=False), "\n                "atau set FAD_USE_VGGISH_PCA=1 hanya jika memang ingin FAD pada "\n                "embedding PCA+8-bit legacy."\n            )\n        if _vggish_input is None:\n            raise RuntimeError("torchvggish.vggish_input tidak tersedia")\n\n        with _torch.inference_mode():\n            for audio in audio_list:\n                audio = np.asarray(audio, dtype=np.float32)\n                if audio.ndim > 1:\n                    audio = audio.mean(axis=1, dtype=np.float32)\n                audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)\n                audio_16k = librosa.resample(\n                    audio,\n                    orig_sr=sr,\n                    target_sr=16000,\n                )\n                examples = _vggish_input.waveform_to_examples(audio_16k, 16000)\n                if len(examples) == 0:\n                    raise RuntimeError("VGGish tidak menghasilkan example untuk salah satu audio.")\n                examples = _torch.as_tensor(examples, dtype=_torch.float32, device=_device)\n                emb = _vggish(examples)\n                emb_np = emb.detach().cpu().numpy()\n                if emb_np.dtype == np.uint8:\n                    emb_np = emb_np.astype(np.float32)\n                features.append(emb_np)\n    except Exception as exc:\n        raise RuntimeError(\n            "FAD wajib memakai VGGish CPU. Install/konfigurasi torchvggish sebelum evaluasi FAD. "\n            f"Detail: {exc}"\n        ) from exc\n\n    features = np.concatenate(features, axis=0).astype(np.float64, copy=False)\n    if features.ndim != 2 or features.shape[0] < 2:\n        raise RuntimeError(f"Embedding VGGish FAD tidak cukup: shape={features.shape}")\n    if not np.isfinite(features).all():\n        raise RuntimeError("Embedding VGGish FAD berisi NaN/Inf.")\n    if FAD_DEBUG_STATS:\n        print(\n            "    FAD VGGish features: "\n            f"shape={features.shape}, mean={features.mean():.4f}, std={features.std():.4f}, "\n            f"min={features.min():.4f}, max={features.max():.4f}"\n        )\n    return features\n\n\ndef compute_fad(original_audios: list, reconstructed_audios: list, sr: int = TARGET_SR):\n    """\n    Hitung Frechet Audio Distance (FAD) sebagai metrik distribusional per-set.\n\n    FAD tetap terpisah dari VISQOL_ODG: FAD menjawab kemiripan distribusi\n    embedding VGGish, sedangkan VISQOL_ODG menjawab kualitas perseptual per\n    pasangan audio.\n    """\n    orig_features = extract_fad_features(original_audios, sr)\n    recon_features = extract_fad_features(reconstructed_audios, sr)\n\n    if orig_features.shape[1] != recon_features.shape[1]:\n        raise RuntimeError(\n            f"Dimensi embedding FAD tidak cocok: original={orig_features.shape}, recon={recon_features.shape}"\n        )\n\n    mu1 = np.mean(orig_features, axis=0)\n    mu2 = np.mean(recon_features, axis=0)\n    d = orig_features.shape[1]\n\n    if len(orig_features) < 2 or len(recon_features) < 2:\n        raise RuntimeError(\n            f"FAD butuh minimal 2 embedding per set: original={len(orig_features)}, recon={len(recon_features)}"\n        )\n\n    sigma1 = np.cov(orig_features, rowvar=False) + 1e-6 * np.eye(d)\n    sigma2 = np.cov(recon_features, rowvar=False) + 1e-6 * np.eye(d)\n    sigma1 = (sigma1 + sigma1.T) * 0.5\n    sigma2 = (sigma2 + sigma2.T) * 0.5\n\n    diff = mu1 - mu2\n    mean_diff = np.dot(diff, diff)\n    if FAD_DEBUG_STATS:\n        if len(orig_features) <= d or len(recon_features) <= d:\n            print(\n                "    ⚠️ FAD sample count lebih kecil/sama dari dimensi embedding "\n                f"(orig={len(orig_features)}, recon={len(recon_features)}, dim={d}); "\n                "covariance FAD bisa sangat noisy. Naikkan N_EVAL_SAMPLES untuk hasil final."\n            )\n        print(f"    FAD mean term: {mean_diff:.4f}")\n\n    def _psd_matrix_sqrt(mat, eps=1e-10):\n        mat = (mat + mat.T) * 0.5\n        vals, vecs = np.linalg.eigh(mat)\n        vals = np.clip(vals, eps, None)\n        return (vecs * np.sqrt(vals)) @ vecs.T\n\n    try:\n        sqrt_sigma1 = _psd_matrix_sqrt(sigma1)\n        covmean = _psd_matrix_sqrt(sqrt_sigma1 @ sigma2 @ sqrt_sigma1)\n    except np.linalg.LinAlgError:\n        offset = np.eye(d) * 1e-5\n        sqrt_sigma1 = _psd_matrix_sqrt(sigma1 + offset)\n        covmean = _psd_matrix_sqrt(sqrt_sigma1 @ (sigma2 + offset) @ sqrt_sigma1)\n\n    fad = mean_diff + np.trace(sigma1 + sigma2 - 2 * covmean)\n    if not np.isfinite(fad):\n        raise RuntimeError("FAD menghasilkan NaN/Inf.")\n    return float(max(0.0, np.real(fad)))\n\n\ndef _eval_device():\n    import torch\n    return torch.device("cuda" if EVAL_USE_GPU and torch.cuda.is_available() else "cpu")\n\n\ndef _stack_audio_gpu(audio_list, device):\n    import torch\n    min_len = min(len(audio) for audio in audio_list)\n    arr = np.stack([np.asarray(audio[:min_len], dtype=np.float32) for audio in audio_list], axis=0)\n    return torch.as_tensor(arr, dtype=torch.float32, device=device), min_len\n\n\ndef _torch_logmel(audio, sr, n_mels=128, n_fft=2048, hop_length=512):\n    import torch\n    import torchaudio\n\n    window = torch.hann_window(n_fft, device=audio.device)\n    spec = torch.stft(\n        audio.float(),\n        n_fft=n_fft,\n        hop_length=hop_length,\n        window=window,\n        return_complex=True,\n    )\n    power = spec.abs().pow(2.0)\n    mel_basis = torchaudio.functional.melscale_fbanks(\n        n_freqs=n_fft // 2 + 1,\n        f_min=0.0,\n        f_max=sr / 2,\n        n_mels=n_mels,\n        sample_rate=sr,\n        norm="slaney",\n        mel_scale="slaney",\n    ).to(device=audio.device, dtype=torch.float32).T.contiguous()\n    mel_power = torch.einsum("mf,bft->bmt", mel_basis, power).clamp_min(1e-10)\n    mel_db = 10.0 * torch.log10(mel_power)\n    mel_db = mel_db - mel_db.amax(dim=(1, 2), keepdim=True)\n    return torch.clamp(mel_db, min=-80.0)\n\n\ndef compute_lsd_batch_gpu(original_audios, reconstructed_audios, gap_ms, sr=TARGET_SR,\n                          n_fft=2048, hop_length=512, frame_pad=2):\n    import torch\n\n    device = _eval_device()\n    if device.type != "cuda":\n        raise RuntimeError("GPU tidak tersedia untuk compute_lsd_batch_gpu")\n\n    originals, n = _stack_audio_gpu(original_audios, device)\n    recons, _ = _stack_audio_gpu(reconstructed_audios, device)\n    originals = originals[:, :n]\n    recons = recons[:, :n]\n\n    window = torch.hann_window(n_fft, device=device)\n    o_power = torch.stft(originals, n_fft=n_fft, hop_length=hop_length, window=window, return_complex=True).abs().pow(2)\n    r_power = torch.stft(recons, n_fft=n_fft, hop_length=hop_length, window=window, return_complex=True).abs().pow(2)\n    eps = torch.clamp(1e-6 * o_power.amax(dim=(1, 2), keepdim=True), min=1e-10)\n    log_diff = 10.0 * (torch.log10(o_power + eps) - torch.log10(r_power + eps))\n\n    gap_samples = int(round(sr * gap_ms / 1000))\n    center = n // 2\n    gap_start = center - gap_samples // 2\n    gap_end = gap_start + gap_samples\n    f_start = max(0, gap_start // hop_length - frame_pad)\n    f_end = min(o_power.shape[-1], gap_end // hop_length + frame_pad + 1)\n    log_diff = log_diff[:, :, f_start:f_end]\n\n    scores = torch.sqrt(torch.mean(log_diff.pow(2), dim=1)).mean(dim=1)\n    return scores.detach().cpu().numpy().astype(np.float64)\n\n\ndef extract_fad_features_gpu(audio_list, sr=TARGET_SR):\n    raise RuntimeError("FAD tidak memakai fitur GPU/log-mel; gunakan VGGish CPU via extract_fad_features().")\n\n\ndef compute_fad_gpu(original_audios, reconstructed_audios, sr=TARGET_SR):\n    """Backward-compatible wrapper: FAD tetap dihitung dengan VGGish di CPU."""\n    return compute_fad(original_audios, reconstructed_audios, sr)\n\n\ndef compute_nsim_odg_batch_gpu(original_audios, reconstructed_audios, sr=TARGET_SR):\n    import torch\n\n    device = _eval_device()\n    if device.type != "cuda":\n        raise RuntimeError("GPU tidak tersedia untuk compute_nsim_odg_batch_gpu")\n    originals, n = _stack_audio_gpu(original_audios, device)\n    recons, _ = _stack_audio_gpu(reconstructed_audios, device)\n    originals = originals[:, :n]\n    recons = recons[:, :n]\n\n    with torch.inference_mode():\n        orig_log = _torch_logmel(originals, sr)\n        recon_log = _torch_logmel(recons, sr)\n        frames = min(orig_log.shape[-1], recon_log.shape[-1])\n        orig_log = orig_log[..., :frames]\n        recon_log = recon_log[..., :frames]\n\n        c1, c2 = 1e-4, 1e-4\n        mu_o = orig_log.mean(dim=1)\n        mu_r = recon_log.mean(dim=1)\n        sig_o = orig_log.std(dim=1, unbiased=False)\n        sig_r = recon_log.std(dim=1, unbiased=False)\n        sig_or = ((orig_log - mu_o.unsqueeze(1)) * (recon_log - mu_r.unsqueeze(1))).mean(dim=1)\n        ssim = ((2 * mu_o * mu_r + c1) * (2 * sig_or + c2)) / (\n            (mu_o.pow(2) + mu_r.pow(2) + c1) * (sig_o.pow(2) + sig_r.pow(2) + c2)\n        )\n        mean_ssim = torch.clamp(ssim, 0, 1).mean(dim=1)\n        odg = torch.clamp(-4.0 * (1.0 - torch.sqrt(mean_ssim)), min=-4.0, max=0.0)\n    return odg.detach().cpu().numpy().astype(np.float64)\n\n\ndef _moslqo_to_odg(moslqo):\n    return float(np.clip(float(moslqo) - 5.0, -4.0, 0.0))\n\n\ndef _compute_visqol_python_odg(original, reconstructed, sr):\n    """ViSQOL fallback via visqol-python pure Python API."""\n    import visqol\n    from visqol import VisqolApi\n\n    orig_48k = librosa.resample(original, orig_sr=sr, target_sr=48000).astype(np.float64)\n    recon_48k = librosa.resample(reconstructed, orig_sr=sr, target_sr=48000).astype(np.float64)\n    n = min(len(orig_48k), len(recon_48k))\n    orig_48k, recon_48k = orig_48k[:n], recon_48k[:n]\n\n    try:\n        api = VisqolApi()\n        api.create(mode="audio")\n        result = api.measure_from_arrays(orig_48k, recon_48k, sample_rate=48000)\n    except Exception as exc:\n        raise RuntimeError(\n            "visqol-python gagal. Kemungkinan ada stale/mixed ViSQOL files di venv. "\n            f"visqol={getattr(visqol, \'__file__\', None)}"\n        ) from exc\n    return _moslqo_to_odg(result.moslqo)\n\n\ndef _compute_visqol_odg(original, reconstructed, sr):\n    """Fallback perceptual ODG via ViSQOL audio mode."""\n    return _compute_visqol_python_odg(original, reconstructed, sr)\n\n\ndef _compute_nsim_odg(original, reconstructed, sr):\n    """Last-resort fallback: NSIM pada log-mel, dipetakan ke ODG-like scale."""\n    orig_mel = librosa.feature.melspectrogram(\n        y=original, sr=sr, n_mels=128, n_fft=2048, hop_length=512\n    )\n    recon_mel = librosa.feature.melspectrogram(\n        y=reconstructed, sr=sr, n_mels=128, n_fft=2048, hop_length=512\n    )\n\n    orig_log = librosa.power_to_db(orig_mel, ref=np.max)\n    recon_log = librosa.power_to_db(recon_mel, ref=np.max)\n\n    C1, C2 = 1e-4, 1e-4\n    mu_o = np.mean(orig_log, axis=0)\n    mu_r = np.mean(recon_log, axis=0)\n    sig_o = np.std(orig_log, axis=0)\n    sig_r = np.std(recon_log, axis=0)\n    sig_or = np.mean((orig_log - mu_o) * (recon_log - mu_r), axis=0)\n\n    ssim = ((2 * mu_o * mu_r + C1) * (2 * sig_or + C2)) / \\\n           ((mu_o**2 + mu_r**2 + C1) * (sig_o**2 + sig_r**2 + C2))\n    mean_ssim = float(np.mean(np.clip(ssim, 0, 1)))\n\n    odg = -4.0 * (1.0 - mean_ssim ** 0.5)\n    return float(np.clip(odg, -4.0, 0.0))\n\n\ndef compute_visqol_odg(original: np.ndarray, reconstructed: np.ndarray, sr: int = TARGET_SR):\n    """\n    Hitung ViSQOL perceptual quality sebagai Objective Difference Grade-like score.\n\n    Primary path memakai ViSQOL audio mode. Untuk hasil paper, kegagalan ViSQOL\n    harus fail-fast; NSIM hanya boleh dipakai untuk smoke test eksplisit.\n    """\n    global _VISQOL_FALLBACK_WARNED\n\n    min_len = min(len(original), len(reconstructed))\n    original = np.asarray(original[:min_len], dtype=np.float64)\n    reconstructed = np.asarray(reconstructed[:min_len], dtype=np.float64)\n\n    visqol_sr = sr\n    if visqol_sr not in (44100, 48000):\n        original = librosa.resample(original, orig_sr=sr, target_sr=44100)\n        reconstructed = librosa.resample(reconstructed, orig_sr=sr, target_sr=44100)\n        visqol_sr = 44100\n\n    try:\n        return _compute_visqol_odg(original, reconstructed, visqol_sr)\n    except Exception as exc:\n        allow_nsim = os.environ.get("ALLOW_NSIM_VISQOL_FALLBACK", "0").lower() in {"1", "true", "yes"}\n        if allow_nsim:\n            if not _VISQOL_FALLBACK_WARNED:\n                print(f"⚠️ ViSQOL gagal ({exc}); fallback ke NSIM ODG-like untuk smoke test.")\n                _VISQOL_FALLBACK_WARNED = True\n            return _compute_nsim_odg(original, reconstructed, visqol_sr)\n        raise RuntimeError(\n            "ViSQOL gagal, sehingga VISQOL_ODG tidak boleh diisi dengan proxy NSIM untuk hasil paper. "\n            "Perbaiki instalasi ViSQOL atau set ALLOW_NSIM_VISQOL_FALLBACK=1 hanya untuk smoke test."\n        ) from exc\n\n\ndef _find_gstpeaq_binary():\n    import shutil\n\n    candidates = [\n        GSTPEAQ_BIN,\n        os.path.join(GSTPEAQ_DIR, "src", "peaq"),\n        os.path.join(GSTPEAQ_DIR, "src", "peaq.exe"),\n        shutil.which("peaq"),\n    ]\n    for path in candidates:\n        if path and os.path.exists(path):\n            return os.path.abspath(path)\n    raise FileNotFoundError(\n        "Executable GstPEAQ \'peaq\' tidak ditemukan. Build external/gstpeaq terlebih dahulu "\n        "atau set GSTPEAQ_BIN=/path/to/peaq. Set EVAL_USE_GSTPEAQ=0 hanya jika ingin skip PEAQ_ODG."\n    )\n\n\ndef _find_gstpeaq_plugin():\n    candidates = [\n        GSTPEAQ_PLUGIN,\n        os.path.join(GSTPEAQ_DIR, "src", ".libs", "libgstpeaq.so"),\n        os.path.join(GSTPEAQ_DIR, "src", ".libs", "libgstpeaq.dylib"),\n        os.path.join(GSTPEAQ_DIR, "src", ".libs", "gstpeaq.dll"),\n    ]\n    for path in candidates:\n        if path and os.path.exists(path):\n            return os.path.abspath(path)\n    return None\n\n\ndef _write_peaq_wav(path, audio, sr):\n    audio = np.asarray(audio, dtype=np.float32)\n    if audio.ndim > 1:\n        audio = np.mean(audio, axis=-1)\n    audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)\n    if sr != 48000:\n        audio = librosa.resample(audio, orig_sr=sr, target_sr=48000).astype(np.float32)\n    audio = np.clip(audio, -1.0, 1.0)\n    sf.write(path, audio, 48000, subtype="PCM_16")\n\n\ndef compute_gstpeaq_odg(original: np.ndarray, reconstructed: np.ndarray, sr: int = TARGET_SR):\n    """Hitung ODG PEAQ asli via GstPEAQ CLI. Ini metrik terpisah dari VISQOL_ODG."""\n    import re\n    import tempfile\n\n    peaq_bin = _find_gstpeaq_binary()\n    peaq_plugin = _find_gstpeaq_plugin()\n\n    min_len = min(len(original), len(reconstructed))\n    original = np.asarray(original[:min_len], dtype=np.float32)\n    reconstructed = np.asarray(reconstructed[:min_len], dtype=np.float32)\n\n    with tempfile.TemporaryDirectory(prefix="gstpeaq_", dir=PATHS["outputs"]) as tmpdir:\n        ref_path = os.path.join(tmpdir, "ref.wav")\n        test_path = os.path.join(tmpdir, "test.wav")\n        _write_peaq_wav(ref_path, original, sr)\n        _write_peaq_wav(test_path, reconstructed, sr)\n\n        cmd = [peaq_bin, "--gst-disable-segtrap"]\n        if peaq_plugin:\n            cmd.append(f"--gst-plugin-load={peaq_plugin}")\n        if GSTPEAQ_ADVANCED:\n            cmd.append("--advanced")\n        cmd.extend([ref_path, test_path])\n\n        env = os.environ.copy()\n        env["LC_ALL"] = "C"\n        proc = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=120)\n        output = f"{proc.stdout}\\n{proc.stderr}"\n        if proc.returncode != 0:\n            raise RuntimeError(f"GstPEAQ gagal dengan exit code {proc.returncode}: {output.strip()}")\n\n    match = re.search(r"Objective Difference Grade:\\s*([-+]?\\d+(?:\\.\\d+)?)", output)\n    if not match:\n        raise RuntimeError(f"Output GstPEAQ tidak memuat ODG: {output.strip()}")\n    odg = float(match.group(1))\n    if not np.isfinite(odg):\n        raise RuntimeError("GstPEAQ menghasilkan ODG NaN/Inf.")\n    return odg\n\n\n# Backward-compatible aliases for the actual GstPEAQ metric.\ncompute_peaq_odg = compute_gstpeaq_odg\ncompute_peaq = compute_gstpeaq_odg\n\n\ndef _regions_are_centered(regions, audio_len, gap_ms, sr=TARGET_SR):\n    if not regions:\n        return True\n    center_start, center_end = compute_gap_bounds(audio_len, gap_ms, sr=sr)\n    return all(\n        int(region.get("gap_start", -1)) == center_start\n        and int(region.get("gap_end", -1)) == center_end\n        for region in regions\n    )\n\n\ndef evaluate_all_gaps(original_audios: list, reconstructed_dict: dict, sr: int = TARGET_SR,\n                      gap_regions_by_gap: dict = None):\n    """\n    Evaluasi semua gap duration untuk satu model.\n\n    Returns:\n        DataFrame dengan full-clip metrics dan gap-focused metrics.\n    """\n    results = []\n    use_gpu_metrics = _eval_device().type == "cuda"\n    if use_gpu_metrics:\n        print(\n            "  Fast GPU evaluation aktif: LSD batch di CUDA; "\n            f"VISQOL_ODG backend={EVAL_VISQOL_BACKEND}; FAD tetap VGGish CPU."\n        )\n\n    for gap_ms, recon_audios in reconstructed_dict.items():\n        print(f"  Evaluating gap {gap_ms}ms...")\n\n        regions = (gap_regions_by_gap or {}).get(gap_ms)\n        if not regions:\n            regions = [\n                {"gap_start": compute_gap_bounds(len(orig), gap_ms, sr=sr)[0],\n                 "gap_end": compute_gap_bounds(len(orig), gap_ms, sr=sr)[1],\n                 "gap_position": "center"}\n                for orig in original_audios\n            ]\n        centered_regions = _regions_are_centered(regions, len(original_audios[0]), gap_ms, sr)\n\n        # Hitung gap indices (konsisten dengan apply_gap_mask)\n        gap_samples = int(round(sr * gap_ms / 1000))\n        fad_score = compute_fad(original_audios, recon_audios, sr)\n\n        if use_gpu_metrics and centered_regions:\n            try:\n                lsd_scores = compute_lsd_batch_gpu(original_audios, recon_audios, gap_ms, sr)\n                lsd_gap_only_scores = compute_lsd_batch_gpu(\n                    original_audios, recon_audios, gap_ms, sr, frame_pad=0\n                )\n                if EVAL_VISQOL_BACKEND in {"fast_gpu", "gpu", "nsim"}:\n                    visqol_odg_scores = compute_nsim_odg_batch_gpu(original_audios, recon_audios, sr)\n                else:\n                    visqol_odg_scores = [compute_visqol_odg(orig, recon, sr) for orig, recon in zip(original_audios, recon_audios)]\n            except Exception as exc:\n                print(f"⚠️ Fast GPU evaluation gagal ({exc}); fallback ke CPU metric path.")\n                lsd_scores = []\n                lsd_gap_only_scores = []\n                visqol_odg_scores = []\n                for orig, recon in zip(original_audios, recon_audios):\n                    center = len(orig) // 2\n                    gap_start = center - gap_samples // 2\n                    gap_end = gap_start + gap_samples\n\n                    lsd_scores.append(compute_lsd(orig, recon, sr,\n                                                  gap_start=gap_start, gap_end=gap_end))\n                    lsd_gap_only_scores.append(compute_lsd(\n                        orig, recon, sr, gap_start=gap_start, gap_end=gap_end, frame_pad=0\n                    ))\n                    visqol_odg_scores.append(compute_visqol_odg(orig, recon, sr))\n        else:\n            lsd_scores = []\n            lsd_gap_only_scores = []\n            visqol_odg_scores = []\n            for idx, (orig, recon) in enumerate(zip(original_audios, recon_audios)):\n                gap_start, gap_end = _metric_gap_bounds(\n                    len(orig), gap_ms, sr=sr, region=regions[idx] if idx < len(regions) else None\n                )\n\n                lsd_scores.append(compute_lsd(orig, recon, sr,\n                                              gap_start=gap_start, gap_end=gap_end))\n                lsd_gap_only_scores.append(compute_lsd(\n                    orig, recon, sr, gap_start=gap_start, gap_end=gap_end, frame_pad=0\n                ))\n                visqol_odg_scores.append(compute_visqol_odg(orig, recon, sr))\n\n        gap_snr_scores = []\n        gap_si_sdr_scores = []\n        gap_mel_scores = []\n        gap_window_visqol_scores = []\n        for idx, (orig, recon) in enumerate(zip(original_audios, recon_audios)):\n            gap_start, gap_end = _metric_gap_bounds(\n                len(orig), gap_ms, sr=sr, region=regions[idx] if idx < len(regions) else None\n            )\n            gap_snr_scores.append(compute_gap_snr(orig, recon, gap_start, gap_end))\n            gap_si_sdr_scores.append(compute_gap_si_sdr(orig, recon, gap_start, gap_end))\n            gap_mel_scores.append(compute_gap_mel_distance(\n                orig, recon, sr=sr, gap_start=gap_start, gap_end=gap_end\n            ))\n            if EVAL_GAP_WINDOW_PERCEPTUAL:\n                gap_window_visqol_scores.append(compute_gap_window_visqol_odg(\n                    orig, recon, sr=sr, gap_start=gap_start, gap_end=gap_end\n                ))\n\n        peaq_odg_scores = []\n        if EVAL_USE_GSTPEAQ:\n            print("    Computing PEAQ_ODG via GstPEAQ...")\n            for orig, recon in zip(original_audios, recon_audios):\n                peaq_odg_scores.append(compute_gstpeaq_odg(orig, recon, sr))\n\n        results.append({\n            "gap_ms": gap_ms,\n            "gap_position": EVAL_GAP_POSITION,\n            "LSD": round(np.mean(lsd_scores), 4),\n            "LSD_GAP_ONLY": round(np.mean(lsd_gap_only_scores), 4),\n            "GAP_LSD": round(np.mean(lsd_gap_only_scores), 4),\n            "GAP_SI_SDR": round(np.nanmean(gap_si_sdr_scores), 4),\n            "GAP_SNR": round(np.nanmean(gap_snr_scores), 4),\n            "GAP_MEL_DISTANCE": round(np.nanmean(gap_mel_scores), 4),\n            "FAD": round(fad_score, 4),\n            "VISQOL_ODG": round(np.mean(visqol_odg_scores), 4),\n            "PEAQ_ODG": round(np.mean(peaq_odg_scores), 4) if peaq_odg_scores else np.nan,\n            "GAP_WINDOW_VISQOL_ODG": round(np.nanmean(gap_window_visqol_scores), 4)\n            if gap_window_visqol_scores else np.nan,\n        })\n\n    return pd.DataFrame(results)\n\n\nprint("✅ Fungsi evaluasi (LSD, FAD, VISQOL_ODG, PEAQ_ODG) berhasil didefinisikan!")\n\n# ---\n# ## CELL 6 — Helper Functions\n\n\n# ============================================================\n# CELL 6: HELPER FUNCTIONS\n# ============================================================\n# Fungsi pendukung untuk:\n# - Monitoring penggunaan VRAM\n# - Membersihkan memori GPU setelah selesai satu model\n# - Menyimpan hasil ke Google Drive\n# - Mengecek apakah model sudah pernah dijalankan\n# - Loading data yang sudah dipreprocess\n# ============================================================\n\nimport torch\nimport gc\nimport os\nimport pandas as pd\nimport soundfile as sf\nimport random\nimport json\nimport logging\nfrom datetime import datetime\n\n\ndef print_gpu_usage(label: str = ""):\n    """\n    Tampilkan penggunaan VRAM saat ini.\n\n    Cara pakai:\n        print_gpu_usage("Sebelum load encoder")\n        # load model...\n        print_gpu_usage("Sesudah load encoder")\n    """\n    if torch.cuda.is_available():\n        allocated = torch.cuda.memory_allocated() / 1e9\n        reserved  = torch.cuda.memory_reserved() / 1e9\n        total     = torch.cuda.get_device_properties(0).total_memory / 1e9\n        free      = total - reserved\n        label_str = f"[{label}] " if label else ""\n        print(f"🖥️  GPU {label_str}| Terpakai: {allocated:.2f}GB | "\n              f"Reserved: {reserved:.2f}GB | Bebas: {free:.2f}GB / {total:.2f}GB")\n\n\ndef clear_gpu_memory(*models):\n    """\n    Bebaskan VRAM setelah selesai menggunakan model.\n\n    Cara pakai:\n        clear_gpu_memory(encoder, decoder, film_layer)\n        # atau untuk baseline:\n        clear_gpu_memory(decoder)\n    """\n    print_gpu_usage("Sebelum clear")\n    for model in models:\n        if model is not None:\n            del model\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n        torch.cuda.synchronize()\n    print_gpu_usage("Sesudah clear")\n    print("✅ GPU memory berhasil dibersihkan\\n")\n\n\ndef save_results(results_df: pd.DataFrame, model_name: str):\n    """\n    Simpan hasil evaluasi ke Google Drive.\n\n    Menyimpan dua file:\n    1. File per model: {model_name}_results.csv\n    2. Master file: all_results.csv (gabungan semua model)\n\n    Args:\n        results_df : DataFrame hasil evaluasi\n        model_name : Nama model (misal: "baseline_cqtdiff", "clap_cqtdiff")\n    """\n    results_df = results_df.copy()\n    results_df["model"] = model_name\n    results_df["experiment_config_id"] = EXPERIMENT_CONFIG_ID\n    results_df["target_sr"] = TARGET_SR\n    results_df["segment_samples"] = SEGMENT_SAMPLES\n    if "gap_position" not in results_df.columns:\n        results_df["gap_position"] = EVAL_GAP_POSITION\n    results_df["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")\n\n    # Simpan file per model\n    result_artifact_name = evaluation_artifact_name(model_name)\n    model_path = os.path.join(PATHS["results"], f"{result_artifact_name}_results.csv")\n    results_df.to_csv(model_path, index=False)\n    print(f"💾 Hasil {model_name} disimpan: {model_path}")\n\n    # Update master file\n    master_path = os.path.join(PATHS["results"], "all_results.csv")\n    if os.path.exists(master_path):\n        existing = pd.read_csv(master_path)\n        if "gap_position" not in existing.columns:\n            existing["gap_position"] = "center"\n        existing = existing[\n            ~((existing["model"] == model_name) & (existing["gap_position"] == EVAL_GAP_POSITION))\n        ]  # Hapus hasil lama untuk model + eval gap position yang sama\n        combined = pd.concat([existing, results_df], ignore_index=True)\n    else:\n        combined = results_df\n\n    combined.to_csv(master_path, index=False)\n    print(f"💾 Master file diupdate: {master_path}")\n    update_experiment_summary()\n\n\n\nEXPECTED_MODEL_CONFIGS = [\n    "baseline_cqtdiff",\n    "baseline_cqtdiff_finetuned",\n    "clap_cqtdiff",\n    "clap_maid",\n    "audiomae_cqtdiff",\n    "audiomae_maid",\n]\nTRAINING_TIMING_SUMMARY = []\n\n\ndef format_duration(seconds):\n    """Format durasi detik menjadi string ringkas."""\n    seconds = float(seconds or 0.0)\n    hours, rem = divmod(int(seconds), 3600)\n    minutes, secs = divmod(rem, 60)\n    if hours:\n        return f"{hours:d}h {minutes:02d}m {secs:02d}s"\n    if minutes:\n        return f"{minutes:d}m {secs:02d}s"\n    return f"{seconds:.1f}s"\n\n\n\n\n\ndef safe_float(value, default=None):\n    try:\n        if value is None or pd.isna(value):\n            return default\n        return float(value)\n    except Exception:\n        return default\n\n\ndef make_json_safe(value):\n    """Convert numpy/pandas scalars so experiment summaries can be dumped as JSON."""\n    if isinstance(value, dict):\n        return {str(k): make_json_safe(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [make_json_safe(v) for v in value]\n    if isinstance(value, np.ndarray):\n        return make_json_safe(value.tolist())\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating,)):\n        return float(value) if np.isfinite(value) else None\n    if isinstance(value, (np.bool_,)):\n        return bool(value)\n    if isinstance(value, (pd.Timestamp, datetime)):\n        return value.isoformat()\n    if value is pd.NA:\n        return None\n    if isinstance(value, float) and not np.isfinite(value):\n        return None\n    return value\n\n\ndef get_peak_vram_gb():\n    if torch.cuda.is_available():\n        return torch.cuda.max_memory_allocated() / (1024 ** 3)\n    return 0.0\n\n\ndef reset_peak_vram_stats():\n    if torch.cuda.is_available():\n        torch.cuda.reset_peak_memory_stats()\n\n\ndef _read_single_timing_seconds(path, column):\n    if os.path.exists(path):\n        try:\n            df = pd.read_csv(path)\n            if not df.empty and column in df.columns:\n                return safe_float(df[column].iloc[-1], 0.0)\n        except Exception:\n            return 0.0\n    return 0.0\n\n\ndef model_name_from_label(label):\n    raw = str(label).strip()\n    if raw in EXPECTED_MODEL_CONFIGS or raw in ALL_MODELS:\n        return raw\n    text = raw.lower().replace("+", " ").replace("-", " ")\n    text = " ".join(text.split())\n    explicit = {\n        "baseline cqtdiff": "baseline_cqtdiff",\n        "baseline cqt diff": "baseline_cqtdiff",\n        "baseline fine tuned no ssl": "baseline_cqtdiff_finetuned",\n        "baseline finetuned no ssl": "baseline_cqtdiff_finetuned",\n        "fine tuned baseline no ssl": "baseline_cqtdiff_finetuned",\n        "finetuned baseline no ssl": "baseline_cqtdiff_finetuned",\n        "clap cqtdiff": "clap_cqtdiff",\n        "clap cqt diff": "clap_cqtdiff",\n        "clap maid": "clap_maid",\n        "audiomae cqtdiff": "audiomae_cqtdiff",\n        "audiomae cqt diff": "audiomae_cqtdiff",\n        "audiomae maid": "audiomae_maid",\n    }\n    if text in explicit:\n        return explicit[text]\n    if "baseline" in text and ("fine tuned" in text or "finetuned" in text or "no ssl" in text):\n        return "baseline_cqtdiff_finetuned"\n    if "baseline" in text:\n        return "baseline_cqtdiff"\n    if "clap" in text and "maid" in text:\n        return "clap_maid"\n    if "clap" in text and "cqt" in text:\n        return "clap_cqtdiff"\n    if "audiomae" in text and "maid" in text:\n        return "audiomae_maid"\n    if "audiomae" in text and "cqt" in text:\n        return "audiomae_cqtdiff"\n    return str(label).strip().lower().replace(" ", "_")\n\n\ndef save_training_history_artifacts(model_name, history):\n    """Persist train/validation loss history plus PNG plot for each model."""\n    if not history:\n        return None\n\n    os.makedirs(PATHS["logs"], exist_ok=True)\n    os.makedirs(PATHS["plots"], exist_ok=True)\n    hist_df = pd.DataFrame(history)\n    history_path = os.path.join(PATHS["logs"], f"{model_name}_training_history.csv")\n    hist_df.to_csv(history_path, index=False)\n\n    try:\n        import matplotlib.pyplot as plt\n        fig, ax1 = plt.subplots(figsize=(8, 5))\n        if "train_loss" in hist_df.columns:\n            ax1.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="train_loss")\n        if "val_loss" in hist_df.columns and hist_df["val_loss"].notna().any():\n            val_df = hist_df.dropna(subset=["val_loss"])\n            ax1.plot(val_df["epoch"], val_df["val_loss"], marker="s", label="val_loss")\n        ax1.set_title(f"Training History - {model_name}")\n        ax1.set_xlabel("Epoch")\n        ax1.set_ylabel("Loss")\n        ax1.grid(True, alpha=0.3)\n        ax1.legend(loc="best")\n        fig.tight_layout()\n        plot_path = os.path.join(PATHS["plots"], f"{model_name}_training_history.png")\n        fig.savefig(plot_path, dpi=150, bbox_inches="tight")\n        plt.close(fig)\n    except Exception as exc:\n        plot_path = None\n        print(f"Could not save training plot for {model_name}: {exc}")\n\n    logging.getLogger("music_inpainting").info("Training history saved for %s: %s", model_name, history_path)\n    return {"history_csv": history_path, "plot_png": plot_path}\n\n\ndef update_experiment_summary():\n    """Write CSV/JSON summary for all configured baselines and hybrids."""\n    rows = []\n    timing_path = os.path.join(PATHS["results"], "training_timing_summary.csv")\n    eval_timing_path = os.path.join(PATHS["results"], "evaluation_timing_summary.csv")\n    results_path = os.path.join(PATHS["results"], "all_results.csv")\n    preprocessing_path = os.path.join(PATHS["results"], "preprocessing_timing.csv")\n\n    timing_df = pd.read_csv(timing_path) if os.path.exists(timing_path) else pd.DataFrame()\n    eval_df = pd.read_csv(eval_timing_path) if os.path.exists(eval_timing_path) else pd.DataFrame()\n    results_df = pd.read_csv(results_path) if os.path.exists(results_path) else pd.DataFrame()\n    preprocessing_seconds = _read_single_timing_seconds(preprocessing_path, "preprocessing_seconds")\n\n    for model_name in EXPECTED_MODEL_CONFIGS:\n        row = {\n            "stage": globals().get("PIPELINE_STAGE_NAME", "code_v3"),\n            "model": model_name,\n            "dataset_fraction": globals().get("DATASET_FRACTION", None),\n            "experiment_config_id": EXPERIMENT_CONFIG_ID,\n            "target_sr": TARGET_SR,\n            "segment_samples": SEGMENT_SAMPLES,\n            "segment_duration": SEGMENT_DURATION,\n            "eval_gap_position": EVAL_GAP_POSITION,\n            "batch_size": None,\n            "epochs": None,\n            "training_seconds": None,\n            "training_time": None,\n            "preprocessing_seconds": preprocessing_seconds,\n            "evaluation_seconds": None,\n            "evaluation_time": None,\n            "peak_vram_gb": None,\n            "checkpoint_path": None,\n            "final_LSD_mean": None,\n            "final_LSD_GAP_ONLY_mean": None,\n            "final_GAP_LSD_mean": None,\n            "final_GAP_SI_SDR_mean": None,\n            "final_GAP_SNR_mean": None,\n            "final_GAP_MEL_DISTANCE_mean": None,\n            "final_FAD_mean": None,\n            "final_VISQOL_ODG_mean": None,\n            "final_PEAQ_ODG_mean": None,\n            "status": "pending",\n            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),\n        }\n\n        if not timing_df.empty and "model" in timing_df.columns:\n            m = timing_df[timing_df["model"] == model_name]\n            if not m.empty:\n                last = m.iloc[-1]\n                row.update({\n                    "batch_size": last.get("batch_size"),\n                    "epochs": last.get("num_epochs"),\n                    "training_seconds": safe_float(last.get("total_seconds")),\n                    "training_time": last.get("total_time"),\n                    "peak_vram_gb": safe_float(last.get("peak_vram_gb")),\n                    "checkpoint_path": last.get("checkpoint_path"),\n                    "status": last.get("status", "trained"),\n                })\n\n        if not eval_df.empty and "model" in eval_df.columns:\n            e = eval_df[eval_df["model"] == model_name]\n            if "eval_gap_position" in e.columns:\n                e = e[e["eval_gap_position"].fillna("center") == EVAL_GAP_POSITION]\n            if not e.empty:\n                last = e.iloc[-1]\n                row["evaluation_seconds"] = safe_float(last.get("evaluation_seconds"))\n                row["evaluation_time"] = last.get("evaluation_time")\n                row["status"] = "evaluated"\n\n        if not results_df.empty and "model" in results_df.columns:\n            r = results_df[results_df["model"] == model_name]\n            if "gap_position" in r.columns:\n                r = r[r["gap_position"].fillna("center") == EVAL_GAP_POSITION]\n            if not r.empty:\n                for metric in [\n                    "LSD", "LSD_GAP_ONLY", "GAP_LSD", "GAP_SI_SDR", "GAP_SNR",\n                    "GAP_MEL_DISTANCE", "FAD", "VISQOL_ODG", "PEAQ_ODG",\n                    "GAP_WINDOW_VISQOL_ODG",\n                ]:\n                    if metric in r.columns:\n                        row[f"final_{metric}_mean"] = safe_float(r[metric].mean())\n                row["status"] = "evaluated"\n\n        rows.append(row)\n\n    summary_df = pd.DataFrame(rows)\n    csv_path = os.path.join(PATHS["results"], "experiment_summary.csv")\n    json_path = os.path.join(PATHS["results"], "experiment_summary.json")\n    summary_df.to_csv(csv_path, index=False)\n    with open(json_path, "w", encoding="utf-8") as f:\n        json.dump(make_json_safe(rows), f, indent=2)\n    print(f"Experiment summary saved: {csv_path}")\n    print(f"Experiment summary JSON saved: {json_path}")\n    return summary_df\n\n\ndef record_evaluation_timing(model_name, total_seconds, n_eval_samples=None):\n    row = {\n        "stage": globals().get("PIPELINE_STAGE_NAME", "code_v3"),\n        "model": model_name,\n        "n_eval_samples": n_eval_samples,\n        "experiment_config_id": EXPERIMENT_CONFIG_ID,\n        "target_sr": TARGET_SR,\n        "segment_samples": SEGMENT_SAMPLES,\n        "eval_gap_position": EVAL_GAP_POSITION,\n        "evaluation_seconds": float(total_seconds or 0.0),\n        "evaluation_time": format_duration(total_seconds or 0.0),\n        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    timing_path = os.path.join(PATHS["results"], "evaluation_timing_summary.csv")\n    if os.path.exists(timing_path):\n        timing_df = pd.read_csv(timing_path)\n        if "eval_gap_position" not in timing_df.columns:\n            timing_df["eval_gap_position"] = "center"\n        timing_df = timing_df[\n            ~((timing_df["model"] == model_name) & (timing_df["eval_gap_position"] == EVAL_GAP_POSITION))\n        ]\n        timing_df = pd.concat([timing_df, pd.DataFrame([row])], ignore_index=True)\n    else:\n        timing_df = pd.DataFrame([row])\n    timing_df.to_csv(timing_path, index=False)\n    logging.getLogger("music_inpainting").info("Evaluation timing saved for %s: %.2fs", model_name, total_seconds)\n    update_experiment_summary()\n    return row\n\ndef check_batch_size_memory(batch_size, min_expected_vram_gb=8.0):\n    """Cetak warning jika VRAM runtime terlihat terlalu kecil untuk batch size stage."""\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA/GPU wajib aktif untuk pipeline ini. Aktifkan GPU runtime sebelum menjalankan notebook.")\n\n    props = torch.cuda.get_device_properties(0)\n    total_gb = props.total_memory / (1024 ** 3)\n    print(f"GPU check: {props.name} | VRAM {total_gb:.2f} GiB | batch_size={batch_size}")\n    if total_gb < min_expected_vram_gb:\n        print(\n            f"VRAM terdeteksi {total_gb:.2f} GiB, di bawah estimasi aman "\n            f"{min_expected_vram_gb:.1f} GiB untuk batch size {batch_size}. "\n            "Jika OOM, gunakan smoke stage atau GPU dengan VRAM lebih besar."\n        )\n        return False\n    print(f"? VRAM memenuhi estimasi minimum batch size {batch_size}.")\n    return True\n\n\ndef record_training_timing(model_name, total_seconds, num_epochs, lr, batch_size=None,\n                           dataset_fraction=None, best_val_loss=None, status="trained",\n                           checkpoint_path=None, epoch_times=None, peak_vram_gb=None):\n    """Simpan ringkasan waktu training per model ke memory dan CSV stage."""\n    batch_size = batch_size if batch_size is not None else globals().get("BATCH_SIZE")\n    dataset_fraction = dataset_fraction if dataset_fraction is not None else globals().get("DATASET_FRACTION")\n    epoch_times = list(epoch_times or [])\n    avg_epoch_seconds = float(np.mean(epoch_times)) if epoch_times else 0.0\n    row = {\n        "stage": globals().get("PIPELINE_STAGE_NAME", "code_v3"),\n        "model": model_name,\n        "status": status,\n        "experiment_config_id": EXPERIMENT_CONFIG_ID,\n        "target_sr": TARGET_SR,\n        "segment_samples": SEGMENT_SAMPLES,\n        "segment_duration": SEGMENT_DURATION,\n        "dataset_fraction": dataset_fraction,\n        "batch_size": batch_size,\n        "num_epochs": num_epochs,\n        "learning_rate": lr,\n        "total_seconds": float(total_seconds or 0.0),\n        "total_time": format_duration(total_seconds or 0.0),\n        "avg_epoch_seconds": avg_epoch_seconds,\n        "avg_epoch_time": format_duration(avg_epoch_seconds),\n        "epoch_times_json": json.dumps(epoch_times),\n        "best_val_loss": best_val_loss,\n        "peak_vram_gb": peak_vram_gb if peak_vram_gb is not None else get_peak_vram_gb(),\n        "checkpoint_path": checkpoint_path,\n        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    TRAINING_TIMING_SUMMARY.append(row)\n\n    timing_path = os.path.join(PATHS["results"], "training_timing_summary.csv")\n    if os.path.exists(timing_path):\n        timing_df = pd.read_csv(timing_path)\n        timing_df = timing_df[timing_df["model"] != model_name]\n        timing_df = pd.concat([timing_df, pd.DataFrame([row])], ignore_index=True)\n    else:\n        timing_df = pd.DataFrame([row])\n    timing_df.to_csv(timing_path, index=False)\n    logging.getLogger("music_inpainting").info("Training timing saved for %s: %.2fs", model_name, total_seconds)\n    print(f"Timing summary updated: {timing_path}")\n    update_experiment_summary()\n\n\ndef print_training_timing_summary(expected_models=None):\n    """Print timing summary untuk baseline + semua kombinasi hybrid."""\n    expected_models = expected_models or EXPECTED_MODEL_CONFIGS\n    timing_path = os.path.join(PATHS["results"], "training_timing_summary.csv")\n    if os.path.exists(timing_path):\n        timing_df = pd.read_csv(timing_path)\n    else:\n        timing_df = pd.DataFrame(TRAINING_TIMING_SUMMARY)\n\n    print("\\n" + "="*70)\n    print("TRAINING TIME SUMMARY")\n    print("="*70)\n    print(f"Total expected configurations: {len(expected_models)}")\n    print("Baseline models: baseline_cqtdiff")\n    print("Hybrid combinations: clap_cqtdiff, clap_maid, audiomae_cqtdiff, audiomae_maid")\n\n    if timing_df.empty:\n        print("Belum ada timing training yang tercatat.")\n        return\n\n    cols = ["model", "status", "dataset_fraction", "batch_size", "num_epochs", "total_time", "best_val_loss"]\n    available_cols = [c for c in cols if c in timing_df.columns]\n    print(timing_df[available_cols].to_string(index=False))\n\n    completed = set(timing_df["model"].tolist()) if "model" in timing_df.columns else set()\n    missing = [m for m in expected_models if m not in completed]\n    if missing:\n        print(f"Timing belum ada untuk: {missing}")\n    else:\n        print("? Timing tersedia untuk semua 5 konfigurasi.")\n\n\ndef print_final_training_summary(available_models=None):\n    """Ringkasan akhir untuk baseline + 4 kombinasi model."""\n    available_models = set(list(available_models or []))\n    print("\\n" + "="*70)\n    print("FINAL TRAINING SUMMARY")\n    print("="*70)\n    print(f"Stage: {globals().get(\'PIPELINE_STAGE_NAME\', \'code_v3\')}")\n    print(f"Dataset fraction: {globals().get(\'DATASET_FRACTION\', \'unknown\')}")\n    print(f"Total configurations: {len(EXPECTED_MODEL_CONFIGS)}")\n    for model_name in EXPECTED_MODEL_CONFIGS:\n        kind = "baseline" if model_name.startswith("baseline_") else "hybrid"\n        status = "evaluated" if model_name in available_models else "pending/no results yet"\n        print(f"  - {model_name:<20} | {kind:<8} | {status}")\n    print_training_timing_summary(EXPECTED_MODEL_CONFIGS)\n\ndef check_if_done(model_name: str):\n    """\n    Cek apakah model ini sudah pernah dijalankan.\n\n    Berguna saat Colab crash: model yang sudah selesai\n    tidak perlu diulang.\n\n    Returns:\n        True  : sudah selesai, bisa di-skip\n        False : belum selesai, perlu dijalankan\n    """\n    result_path = os.path.join(PATHS["results"], f"{evaluation_artifact_name(model_name)}_results.csv")\n    if os.path.exists(result_path):\n        print(f"✅ {model_name} sudah selesai sebelumnya.")\n        print(f"   Untuk menjalankan ulang, hapus: {result_path}")\n        return True\n    return False\n\n\ndef _stratified_sample_table(df, n_samples, seed=DATASET_RANDOM_SEED,\n                             stratify_cols=("composer", "instrument")):\n    """Sampling deterministic dengan proporsi composer + instrument sebisa mungkin terjaga."""\n    if len(df) == 0 or n_samples <= 0:\n        return df.iloc[0:0].copy()\n\n    work = df.copy().reset_index(drop=True)\n    work["_sample_row_id"] = np.arange(len(work))\n    n_samples = min(int(n_samples), len(work))\n    for col in stratify_cols:\n        if col not in work.columns:\n            work[col] = "unknown"\n        work[col] = work[col].fillna("unknown").astype(str)\n\n    rng = np.random.default_rng(seed)\n    grouped = list(work.groupby(list(stratify_cols), dropna=False, sort=True))\n    quotas = []\n    for _, group in grouped:\n        expected = len(group) * n_samples / len(work)\n        base = int(np.floor(expected))\n        quotas.append({\n            "group": group,\n            "quota": min(base, len(group)),\n            "fractional": expected - base,\n            "tie": rng.random(),\n        })\n\n    remaining = n_samples - sum(q["quota"] for q in quotas)\n    for q in sorted(quotas, key=lambda item: (-item["fractional"], item["tie"])):\n        if remaining <= 0:\n            break\n        capacity = len(q["group"]) - q["quota"]\n        if capacity > 0:\n            q["quota"] += 1\n            remaining -= 1\n\n    parts = []\n    for offset, q in enumerate(quotas):\n        if q["quota"] > 0:\n            parts.append(q["group"].sample(q["quota"], random_state=seed + offset))\n\n    sampled = pd.concat(parts, ignore_index=True) if parts else work.iloc[0:0].copy()\n    if len(sampled) < n_samples:\n        missing = n_samples - len(sampled)\n        sampled_ids = set(sampled["_sample_row_id"]) if "_sample_row_id" in sampled.columns else set()\n        unsampled = work[~work["_sample_row_id"].isin(sampled_ids)]\n        if len(unsampled) > 0:\n            sampled = pd.concat([\n                sampled,\n                unsampled.sample(min(missing, len(unsampled)), random_state=seed + 999)\n            ], ignore_index=True)\n\n    return sampled.sample(frac=1.0, random_state=seed).drop(columns=["_sample_row_id"], errors="ignore").reset_index(drop=True)\n\n\ndef get_data_splits(meta_df=None):\n    """\n    Buat group-aware train/val/test split berdasarkan source_file.\n\n    Split dilakukan pada level source_file (lagu), bukan segment,\n    agar tidak ada segment dari lagu yang sama muncul di train dan test\n    (mencegah data leakage). Pemilihan source_file memakai seed 42 dan\n    stratified sampling berdasarkan composer + instrument.\n\n    Returns:\n        dict: {"train": DataFrame, "val": DataFrame, "test": DataFrame}\n    """\n    if meta_df is None:\n        meta_path = os.path.join(PATHS["preprocessed"], "metadata.csv")\n        meta_df = pd.read_csv(meta_path)\n\n    source_cols = ["source_file"]\n    for col in ["composer", "instrument"]:\n        if col in meta_df.columns:\n            source_cols.append(col)\n\n    source_df = meta_df[source_cols].drop_duplicates("source_file").reset_index(drop=True)\n    for col in ["composer", "instrument"]:\n        if col not in source_df.columns:\n            source_df[col] = "unknown"\n        source_df[col] = source_df[col].fillna("unknown").astype(str)\n\n    n_test = max(1, int(round(len(source_df) * 0.15)))\n    n_val = max(1, int(round(len(source_df) * 0.15)))\n\n    test_sources_df = _stratified_sample_table(source_df, n_test, seed=DATASET_RANDOM_SEED)\n    remaining_sources = source_df[~source_df["source_file"].isin(test_sources_df["source_file"])].reset_index(drop=True)\n    val_sources_df = _stratified_sample_table(remaining_sources, n_val, seed=DATASET_RANDOM_SEED + 1)\n    train_sources_df = remaining_sources[~remaining_sources["source_file"].isin(val_sources_df["source_file"])].reset_index(drop=True)\n\n    test_files = set(test_sources_df["source_file"])\n    val_files = set(val_sources_df["source_file"])\n    train_files = set(train_sources_df["source_file"])\n\n    splits = {\n        "train": meta_df[meta_df["source_file"].isin(train_files)].reset_index(drop=True),\n        "val": meta_df[meta_df["source_file"].isin(val_files)].reset_index(drop=True),\n        "test": meta_df[meta_df["source_file"].isin(test_files)].reset_index(drop=True),\n    }\n\n    for name, df in splits.items():\n        n_sources = df["source_file"].nunique()\n        print(f"  {name:5s}: {len(df)} segments from {n_sources} source files")\n\n    print(f"  split seed: {DATASET_RANDOM_SEED} | stratified by composer + instrument")\n    return splits\n\ndef load_preprocessed_data(n_samples: int = 50, split: str = "test", gap_position: str = None):\n    """\n    Load data yang sudah dipreprocess dari Drive.\n\n    Menggunakan group-aware split untuk mencegah data leakage.\n    Default menggunakan split "test" untuk evaluasi.\n\n    Args:\n        n_samples: Jumlah sampel untuk evaluasi.\n        split: "train", "val", atau "test"\n\n    Returns:\n        original_audios : List audio ground truth\n        masked_by_gap   : {gap_ms: [list audio masked]}\n        gap_regions     : {gap_ms: [{"gap_start", "gap_end", "gap_position"}]}\n    """\n    meta_path = os.path.join(PATHS["preprocessed"], "metadata.csv")\n    if not os.path.exists(meta_path):\n        raise FileNotFoundError(\n            "Metadata tidak ditemukan! Jalankan Cell 3 (preprocessing) terlebih dahulu."\n        )\n\n    meta_df = pd.read_csv(meta_path)\n    splits = get_data_splits(meta_df)\n    split_df = splits[split]\n\n    selected = _stratified_sample_table(split_df, min(n_samples, len(split_df)), seed=DATASET_RANDOM_SEED + 2)\n\n    gap_position = (gap_position or EVAL_GAP_POSITION).strip().lower()\n    if gap_position not in {"center", "random"}:\n        raise ValueError("gap_position harus \'center\' atau \'random\'.")\n\n    print(f"📂 Loading {len(selected)} sampel dari Drive (split={split}, gap_position={gap_position})...")\n\n    original_audios = []\n    masked_by_gap = {gap_ms: [] for gap_ms in GAP_DURATIONS_MS}\n    gap_regions = {gap_ms: [] for gap_ms in GAP_DURATIONS_MS}\n\n    for sample_index, (_, row) in enumerate(selected.iterrows()):\n        orig_audio, sr_loaded = sf.read(row["clean_path"])\n        if int(sr_loaded) != int(TARGET_SR):\n            raise RuntimeError(f"SR preprocessed tidak cocok: {sr_loaded} != {TARGET_SR} pada {row[\'clean_path\']}")\n        orig_audio = np.asarray(orig_audio, dtype=np.float32)\n        original_audios.append(orig_audio)\n\n        filename = os.path.basename(row["clean_path"])\n        for gap_ms in GAP_DURATIONS_MS:\n            if gap_position == "center":\n                masked_path = os.path.join(PATHS["masked"], f"gap_{gap_ms}ms", filename)\n                masked_audio, sr_masked = sf.read(masked_path)\n                if int(sr_masked) != int(TARGET_SR):\n                    raise RuntimeError(f"SR masked tidak cocok: {sr_masked} != {TARGET_SR} pada {masked_path}")\n                mask, gap_start, gap_end = make_gap_mask(len(orig_audio), gap_ms)\n                masked_audio = np.asarray(masked_audio, dtype=np.float32)\n            else:\n                mask, gap_start, gap_end = make_eval_gap_mask(\n                    len(orig_audio), gap_ms, sample_index=sample_index, sr=TARGET_SR\n                )\n                masked_audio = orig_audio.copy()\n                masked_audio[gap_start:gap_end] = 0.0\n            masked_by_gap[gap_ms].append(masked_audio)\n            gap_regions[gap_ms].append({\n                "gap_start": int(gap_start),\n                "gap_end": int(gap_end),\n                "gap_position": gap_position,\n            })\n\n    print(f"✅ {len(original_audios)} sampel siap dievaluasi.")\n    return original_audios, masked_by_gap, gap_regions\n\n\nprint("✅ Helper functions berhasil didefinisikan!")\n\n# ---\n# ## CELL 6.5 — Dataset, DataLoader & Shared Utilities\n# \n# Definisi Dataset/DataLoader untuk training, serta fungsi utilitas\n# yang dipakai di semua cell inference (mask creation, boundary cross-fade).\n\n\n# ============================================================\n# CELL 6.5: DATASET, DATALOADER & SHARED UTILITIES\n# ============================================================\n# 1. MusicGapDataset: PyTorch Dataset untuk training\n# 2. make_gap_mask(): fungsi mask yang konsisten antara\n#    preprocessing dan inference (menghindari off-by-one)\n# 3. crossfade_boundary(): half-cosine crossfade untuk\n#    menghilangkan click artifact di boundary gap\n# 4. DataLoader optimized untuk Vast.ai GPU training\n# ============================================================\n\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nimport numpy as np\nimport soundfile as sf\nimport os\nimport time\nimport random\nfrom tqdm import tqdm\n\n# Vast.ai/Linux: use a small worker pool. SSL encoder precompute stays single-process\n# because AudioMAE/timm/torchaudio can fan out OpenMP threads inside each worker.\nCPU_COUNT = os.cpu_count() or 4\nAUTO_NUM_WORKERS = int(os.environ.get(\n    "PIPELINE_NUM_WORKERS",\n    0 if os.name == "nt" else min(2, max(1, CPU_COUNT // 4)),\n))\nENCODER_PRECOMPUTE_NUM_WORKERS = int(os.environ.get("ENCODER_PRECOMPUTE_NUM_WORKERS", "0"))\nDATALOADER_PREFETCH_FACTOR = int(os.environ.get("PIPELINE_PREFETCH_FACTOR", "2"))\nCACHE_AUDIO_IN_MEMORY = True\nPRECOMPUTE_ENCODER_LATENTS = True\n\ntry:\n    torch.set_num_threads(int(os.environ.get("PIPELINE_TORCH_THREADS", CPU_THREAD_LIMIT)))\n    torch.set_num_interop_threads(1)\nexcept Exception:\n    pass\n\n\ndef make_gap_mask(audio_length, gap_ms, sr=TARGET_SR):\n    return build_gap_mask_array(audio_length, gap_ms, sr=sr)\n\n\ndef make_eval_gap_mask(audio_length, gap_ms, sample_index, sr=TARGET_SR):\n    """Center gap by default; deterministic non-center random gap for robustness eval."""\n    if EVAL_GAP_POSITION == "center":\n        return make_gap_mask(audio_length, gap_ms, sr=sr)\n\n    gap_samples = int(round(sr * gap_ms / 1000))\n    min_context = int(round(sr * EVAL_RANDOM_GAP_MIN_CONTEXT_MS / 1000))\n    min_start = min_context\n    max_start = audio_length - gap_samples - min_context\n    if max_start < min_start:\n        min_start = 0\n        max_start = audio_length - gap_samples\n    if max_start < min_start:\n        raise ValueError(\n            f"Random gap {gap_ms}ms tidak valid untuk audio_length={audio_length}, sr={sr}."\n        )\n    rng = np.random.default_rng(DATASET_RANDOM_SEED + 100_000 + int(sample_index) * 997 + int(gap_ms))\n    gap_start = int(rng.integers(min_start, max_start + 1))\n    return build_gap_mask_array(audio_length, gap_ms, sr=sr, gap_start=gap_start)\n\n\ndef crossfade_boundary(original, reconstructed, gap_start, gap_end,\n                       sr=TARGET_SR, fade_ms=30):\n    fade_n = int(fade_ms * 1e-3 * sr)\n    if fade_n <= 0:\n        return reconstructed\n\n    output = reconstructed.copy()\n    fade = 0.5 * (1 - np.cos(np.pi * np.linspace(0, 1, fade_n)))\n\n    left_start = max(0, gap_start - fade_n)\n    left_len = gap_start - left_start\n    if left_len > 0:\n        f = fade[-left_len:]\n        output[left_start:gap_start] = (\n            original[left_start:gap_start] * (1 - f)\n            + reconstructed[left_start:gap_start] * f\n        )\n\n    right_end = min(len(output), gap_end + fade_n)\n    right_len = right_end - gap_end\n    if right_len > 0:\n        f = fade[:right_len]\n        output[gap_end:right_end] = (\n            reconstructed[gap_end:right_end] * (1 - f)\n            + original[gap_end:right_end] * f\n        )\n\n    return output\n\n\ndef evaluation_artifact_name(model_name: str):\n    """Separate optional robustness-eval artifacts from main center-gap artifacts."""\n    if EVAL_GAP_POSITION == "center":\n        return model_name\n    return f"{model_name}_{EVAL_GAP_POSITION}gap"\n\n\ndef prepare_reconstructed_outputs(model_name: str, clear: bool = False):\n    """Siapkan folder reconstructed audio; hapus hanya jika diminta eksplisit."""\n    import shutil\n\n    out_dir = os.path.join(PATHS["outputs"], evaluation_artifact_name(model_name))\n    if clear and os.path.isdir(out_dir):\n        shutil.rmtree(out_dir)\n    os.makedirs(out_dir, exist_ok=True)\n    return out_dir\n\n\ndef reset_reconstructed_outputs(model_name: str):\n    """Backward-compatible helper untuk mengosongkan reconstructed audio."""\n    return prepare_reconstructed_outputs(model_name, clear=True)\n\n\ndef reconstructed_output_path(model_name: str, gap_ms: int, sample_index: int):\n    artifact_name = evaluation_artifact_name(model_name)\n    out_dir = os.path.join(PATHS["outputs"], artifact_name, f"gap_{gap_ms}ms")\n    filename = f"{artifact_name}_gap{gap_ms}ms_sample{sample_index:04d}_reconstructed.wav"\n    return os.path.join(out_dir, filename)\n\n\nCURRENT_RECONSTRUCTION_CACHE_TAG = None\nEVAL_CACHE_CODE_VERSION = "eval_cache_v4_musicnet_cqtdiffplus_gapaware"\n\n\ndef _file_fingerprint(path: str):\n    if not path or not os.path.exists(path):\n        return None\n    st = os.stat(path)\n    return {\n        "path": os.path.abspath(path),\n        "size": int(st.st_size),\n        "mtime_ns": int(st.st_mtime_ns),\n    }\n\n\ndef _decoder_cache_identity(decoder):\n    return {\n        "class": decoder.__class__.__name__,\n        "architecture": getattr(decoder, "architecture_name", None),\n        "official_weights_path": getattr(decoder, "official_weights_path", None),\n        "official_weights_type": getattr(decoder, "official_weights_type", None),\n        "official_weights_fingerprint": _file_fingerprint(getattr(decoder, "official_weights_path", None)),\n    }\n\n\ndef set_reconstruction_cache_context(model_name: str, decoder, checkpoint_path: str = None):\n    """Cache tag supaya evaluator tidak memakai WAV lama dari checkpoint/arsitektur berbeda."""\n    global CURRENT_RECONSTRUCTION_CACHE_TAG\n    context = {\n        "version": EVAL_CACHE_CODE_VERSION,\n        "stage": globals().get("PIPELINE_STAGE_NAME", "code_v3"),\n        "experiment_config_id": EXPERIMENT_CONFIG_ID,\n        "model": model_name,\n        "checkpoint": _file_fingerprint(checkpoint_path),\n        "decoder": _decoder_cache_identity(decoder),\n        "target_sr": int(TARGET_SR),\n        "segment_samples": int(SEGMENT_SAMPLES),\n        "gap_durations_ms": list(GAP_DURATIONS_MS),\n        "eval_gap_position": EVAL_GAP_POSITION,\n        "eval_random_gap_min_context_ms": int(EVAL_RANDOM_GAP_MIN_CONTEXT_MS),\n    }\n    CURRENT_RECONSTRUCTION_CACHE_TAG = json.dumps(context, sort_keys=True)\n    return CURRENT_RECONSTRUCTION_CACHE_TAG\n\n\ndef _reconstructed_sidecar_path(path: str):\n    return f"{path}.meta.json"\n\n\ndef save_reconstructed_output(model_name: str, gap_ms: int, sample_index: int,\n                              reconstructed, sr: int = TARGET_SR):\n    """Simpan final reconstructed waveform yang dipakai oleh metrik evaluasi."""\n    path = reconstructed_output_path(model_name, gap_ms, sample_index)\n    out_dir = os.path.dirname(path)\n    os.makedirs(out_dir, exist_ok=True)\n\n    audio = np.asarray(reconstructed, dtype=np.float32)\n    if audio.ndim > 1:\n        audio = audio.mean(axis=-1, dtype=np.float32)\n    audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)\n\n    sf.write(path, audio, sr, subtype="FLOAT")\n    sidecar = {\n        "model": model_name,\n        "gap_ms": int(gap_ms),\n        "sample_index": int(sample_index),\n        "sr": int(sr),\n        "n_samples": int(len(audio)),\n        "experiment_config_id": EXPERIMENT_CONFIG_ID,\n        "eval_gap_position": EVAL_GAP_POSITION,\n        "cache_tag": CURRENT_RECONSTRUCTION_CACHE_TAG,\n        "saved_at": pd.Timestamp.now().isoformat(),\n    }\n    with open(_reconstructed_sidecar_path(path), "w", encoding="utf-8") as f:\n        json.dump(sidecar, f, indent=2)\n    return path\n\n\ndef load_reconstructed_output_if_available(model_name: str, gap_ms: int, sample_index: int,\n                                           expected_n_samples: int = None, sr: int = TARGET_SR):\n    """Load reconstructed WAV yang sudah ada; return (audio, path) atau (None, path)."""\n    path = reconstructed_output_path(model_name, gap_ms, sample_index)\n    if not EVAL_REUSE_RECONSTRUCTIONS or not os.path.exists(path):\n        return None, path\n\n    try:\n        if CURRENT_RECONSTRUCTION_CACHE_TAG is not None:\n            sidecar_path = _reconstructed_sidecar_path(path)\n            if not os.path.exists(sidecar_path):\n                print(f"    Reconstruct cache ignored (missing cache metadata): {path}")\n                return None, path\n            with open(sidecar_path, "r", encoding="utf-8") as f:\n                sidecar = json.load(f)\n            if sidecar.get("cache_tag") != CURRENT_RECONSTRUCTION_CACHE_TAG:\n                print(f"    Reconstruct cache ignored (checkpoint/model cache tag mismatch): {path}")\n                return None, path\n            if sidecar.get("model") != model_name or int(sidecar.get("gap_ms", -1)) != int(gap_ms):\n                print(f"    Reconstruct cache ignored (model/gap metadata mismatch): {path}")\n                return None, path\n            if int(sidecar.get("sample_index", -1)) != int(sample_index):\n                print(f"    Reconstruct cache ignored (sample metadata mismatch): {path}")\n                return None, path\n        info = sf.info(path)\n        if int(info.samplerate) != int(sr):\n            print(f"    Reconstruct cache ignored (SR mismatch): {path}")\n            return None, path\n        if expected_n_samples is not None and int(info.frames) != int(expected_n_samples):\n            print(f"    Reconstruct cache ignored (length mismatch): {path}")\n            return None, path\n        audio = _read_audio_float32(path)\n        if not np.isfinite(audio).all():\n            print(f"    Reconstruct cache ignored (non-finite audio): {path}")\n            return None, path\n        return audio, path\n    except Exception as exc:\n        print(f"    Reconstruct cache ignored ({exc}): {path}")\n        return None, path\n\n\ndef summarize_reconstruction_cache(model_name: str, n_eval_samples: int):\n    """Print ringkasan cache rekonstruksi yang akan dipakai evaluator."""\n    total_expected = len(GAP_DURATIONS_MS) * int(n_eval_samples)\n    existing = 0\n    missing_examples = []\n\n    for gap_ms in GAP_DURATIONS_MS:\n        for sample_index in range(int(n_eval_samples)):\n            path = reconstructed_output_path(model_name, gap_ms, sample_index)\n            if os.path.exists(path):\n                existing += 1\n            elif len(missing_examples) < 3:\n                missing_examples.append(path)\n\n    print(\n        f"  Reconstruction cache probe: {existing}/{total_expected} WAV ditemukan "\n        f"di {os.path.join(PATHS[\'outputs\'], evaluation_artifact_name(model_name))}"\n    )\n    if missing_examples:\n        print("  Contoh path yang belum ada:")\n        for path in missing_examples:\n            print(f"   - {path}")\n    return existing, total_expected\n\n\ndef save_reconstruction_manifest(model_name: str, rows: list):\n    out_dir = os.path.join(PATHS["outputs"], evaluation_artifact_name(model_name))\n    os.makedirs(out_dir, exist_ok=True)\n    manifest_path = os.path.join(out_dir, "manifest.csv")\n    pd.DataFrame(rows).to_csv(manifest_path, index=False)\n    print(f"💾 Reconstructed output manifest: {manifest_path}")\n    return manifest_path\n\n\ndef validate_masked_gap_alignment(original, masked, mask, gap_ms, sr=TARGET_SR,\n                                  atol=2e-4, expected_gap_start=None, expected_gap_end=None):\n    """Fail-fast jika file masked tidak memiliki gap di posisi yang sama dengan mask evaluasi."""\n    original = np.asarray(original, dtype=np.float32)\n    masked = np.asarray(masked, dtype=np.float32)\n    mask = np.asarray(mask, dtype=bool)\n    n = min(len(original), len(masked), len(mask))\n    original = original[:n]\n    masked = masked[:n]\n    mask = mask[:n]\n\n    gap_samples = int(round(sr * gap_ms / 1000))\n    if expected_gap_start is None or expected_gap_end is None:\n        expected_start, expected_end = compute_gap_bounds(n, gap_ms, sr=sr)\n    else:\n        expected_start, expected_end = int(expected_gap_start), int(expected_gap_end)\n    gap_idx = np.flatnonzero(mask)\n    if len(gap_idx) != gap_samples or gap_idx[0] != expected_start or gap_idx[-1] + 1 != expected_end:\n        raise RuntimeError(\n            f"Mask gap tidak sesuai untuk {gap_ms}ms: expected=({expected_start},{expected_end}), "\n            f"actual=({gap_idx[0] if len(gap_idx) else None},{gap_idx[-1] + 1 if len(gap_idx) else None}), "\n            f"gap_samples={len(gap_idx)}"\n        )\n\n    gap_abs = float(np.max(np.abs(masked[mask]))) if np.any(mask) else 0.0\n    known_abs = float(np.max(np.abs(masked[~mask] - original[~mask]))) if np.any(~mask) else 0.0\n    if gap_abs > atol:\n        raise RuntimeError(f"Masked audio gap {gap_ms}ms tidak nol penuh: max_abs_gap={gap_abs:.6g}")\n    if known_abs > atol:\n        raise RuntimeError(f"Masked audio non-gap berubah dari original: max_abs_known_diff={known_abs:.6g}")\n\n    return {\n        "gap_start": int(expected_start),\n        "gap_end": int(expected_end),\n        "gap_samples": int(gap_samples),\n        "masked_gap_max_abs": gap_abs,\n        "known_region_max_abs_diff": known_abs,\n    }\n\n\ndef _gap_region_stats(original, reconstructed, gap_ms, sr=TARGET_SR, gap_start=None, gap_end=None):\n    n = min(len(original), len(reconstructed))\n    original = np.asarray(original[:n], dtype=np.float32)\n    reconstructed = np.asarray(reconstructed[:n], dtype=np.float32)\n    if gap_start is None or gap_end is None:\n        gap_start, gap_end = compute_gap_bounds(n, gap_ms, sr=sr)\n    gap_start, gap_end = int(gap_start), int(gap_end)\n    ref_gap = original[gap_start:gap_end]\n    rec_gap = reconstructed[gap_start:gap_end]\n    eps = 1e-12\n    ref_rms = float(np.sqrt(np.mean(ref_gap ** 2) + eps))\n    rec_rms = float(np.sqrt(np.mean(rec_gap ** 2) + eps))\n    return {\n        "gap_start": int(gap_start),\n        "gap_end": int(gap_end),\n        "ref_gap_rms": ref_rms,\n        "recon_gap_rms": rec_rms,\n        "gap_gain_db": float(20.0 * np.log10((rec_rms + eps) / (ref_rms + eps))),\n        "ref_full_rms": float(np.sqrt(np.mean(original ** 2) + eps)),\n        "recon_full_rms": float(np.sqrt(np.mean(reconstructed ** 2) + eps)),\n        "gap_peak_abs": float(np.max(np.abs(rec_gap))) if len(rec_gap) else 0.0,\n        "gap_zero_fraction": float(np.mean(np.abs(rec_gap) < 1e-6)) if len(rec_gap) else 1.0,\n    }\n\n\ndef save_reconstruction_diagnostics(model_name: str, original_audios: list, reconstructed_dict: dict,\n                                    alignment_rows: list = None, conditioning_rows: list = None,\n                                    gap_regions_by_gap: dict = None):\n    """Simpan diagnostik gain/posisi/conditioning untuk audit hasil evaluasi."""\n    out_dir = os.path.join(PATHS["outputs"], evaluation_artifact_name(model_name))\n    os.makedirs(out_dir, exist_ok=True)\n\n    rows = []\n    for gap_ms, recon_audios in reconstructed_dict.items():\n        regions = (gap_regions_by_gap or {}).get(gap_ms, [])\n        per_sample = []\n        for idx, (orig, recon) in enumerate(zip(original_audios, recon_audios)):\n            region = regions[idx] if idx < len(regions) else {}\n            per_sample.append(\n                _gap_region_stats(\n                    orig, recon, gap_ms, TARGET_SR,\n                    gap_start=region.get("gap_start"),\n                    gap_end=region.get("gap_end"),\n                )\n            )\n        row = {"model": model_name, "gap_ms": int(gap_ms), "n_samples": int(len(per_sample))}\n        for key in per_sample[0].keys():\n            values = np.asarray([item[key] for item in per_sample], dtype=np.float64)\n            row[f"{key}_mean"] = float(np.mean(values))\n            row[f"{key}_std"] = float(np.std(values))\n        rows.append(row)\n\n    gain_path = os.path.join(out_dir, "diagnostics_gain_by_gap.csv")\n    pd.DataFrame(rows).to_csv(gain_path, index=False)\n    print(f"💾 Gain diagnostics: {gain_path}")\n\n    if alignment_rows:\n        align_path = os.path.join(out_dir, "diagnostics_gap_alignment.csv")\n        pd.DataFrame(alignment_rows).to_csv(align_path, index=False)\n        print(f"💾 Gap alignment diagnostics: {align_path}")\n\n    if conditioning_rows:\n        cond_path = os.path.join(out_dir, "diagnostics_conditioning.csv")\n        pd.DataFrame(conditioning_rows).to_csv(cond_path, index=False)\n        print(f"💾 Conditioning diagnostics: {cond_path}")\n\n\ndef _read_audio_float32(path):\n    audio, _ = sf.read(path, dtype="float32", always_2d=False)\n    if audio.ndim > 1:\n        audio = audio.mean(axis=1, dtype=np.float32)\n    return np.ascontiguousarray(audio, dtype=np.float32)\n\n\nclass MusicGapDataset(Dataset):\n    """\n    Dataset optimized for GPU training:\n    - metadata columns are lists, avoiding pandas iloc per sample\n    - clean audio can be cached in RAM once, removing repeated disk reads\n    - mask creation is simple slicing and matches apply_gap_mask semantics\n    """\n    def __init__(self, meta_df, gap_ms_choices, sr=TARGET_SR, cache_audio=CACHE_AUDIO_IN_MEMORY):\n        self.meta = meta_df.reset_index(drop=True)\n        self.clean_paths = self.meta["clean_path"].astype(str).tolist()\n        self.gap_ms_choices = list(gap_ms_choices)\n        self.sr = sr\n        self.cache_audio = bool(cache_audio)\n        self._audio_cache = None\n\n        if self.cache_audio:\n            cache_start = time.perf_counter()\n            self._audio_cache = [_read_audio_float32(path) for path in self.clean_paths]\n            mb = sum(audio.nbytes for audio in self._audio_cache) / (1024 ** 2)\n            print(f"   Cached {len(self._audio_cache)} audio segments ({mb:.1f} MB) in {time.perf_counter() - cache_start:.1f}s")\n\n    def __len__(self):\n        return len(self.clean_paths) * len(self.gap_ms_choices)\n\n    def _get_clean_audio(self, seg_i):\n        if self._audio_cache is not None:\n            return self._audio_cache[seg_i]\n        return _read_audio_float32(self.clean_paths[seg_i])\n\n    def __getitem__(self, idx):\n        seg_i, g_i = divmod(int(idx), len(self.gap_ms_choices))\n        gap_ms = self.gap_ms_choices[g_i]\n\n        clean = self._get_clean_audio(seg_i)\n        mask, gs, ge = make_gap_mask(len(clean), gap_ms, self.sr)\n        masked = clean.copy()\n        masked[gs:ge] = 0.0\n\n        return {\n            "clean": torch.from_numpy(clean),\n            "masked": torch.from_numpy(masked),\n            "mask": torch.from_numpy(mask),\n            "gap_start": gs,\n            "gap_end": ge,\n            "gap_ms": gap_ms,\n        }\n\n\nclass EncoderCachedDataset(Dataset):\n    """Dataset wrapper that adds precomputed frozen SSL encoder latents."""\n    def __init__(self, base_dataset, encoder_latents):\n        self.base_dataset = base_dataset\n        self.encoder_latents = encoder_latents.cpu().float()\n        if len(self.encoder_latents) != len(self.base_dataset):\n            raise ValueError("encoder_latents length must match base_dataset length")\n\n    def __len__(self):\n        return len(self.base_dataset)\n\n    def __getitem__(self, idx):\n        item = self.base_dataset[idx]\n        item["encoder_latent"] = self.encoder_latents[int(idx)]\n        return item\n\n\ndef _seed_worker(worker_id):\n    worker_seed = DATASET_RANDOM_SEED + worker_id\n    np.random.seed(worker_seed)\n    random.seed(worker_seed)\n    torch.set_num_threads(1)\n\n\ndef build_audio_loader(ds, batch_size, shuffle, num_workers=AUTO_NUM_WORKERS):\n    kwargs = {\n        "batch_size": batch_size,\n        "shuffle": shuffle,\n        "num_workers": int(num_workers),\n        "pin_memory": torch.cuda.is_available(),\n        "persistent_workers": (int(num_workers) > 0),\n        "worker_init_fn": _seed_worker if int(num_workers) > 0 else None,\n    }\n    if int(num_workers) > 0:\n        kwargs["prefetch_factor"] = DATALOADER_PREFETCH_FACTOR\n    return DataLoader(ds, **kwargs)\n\n\ndef make_dataloaders(batch_size=32, num_workers=AUTO_NUM_WORKERS, cache_audio=CACHE_AUDIO_IN_MEMORY):\n    """Buat DataLoader untuk train/val/test dengan group-aware split."""\n    meta_path = os.path.join(PATHS["preprocessed"], "metadata.csv")\n    meta_df = pd.read_csv(meta_path)\n    splits = get_data_splits(meta_df)\n\n    if num_workers is None:\n        num_workers = AUTO_NUM_WORKERS\n    print(f"DataLoader config: batch_size={batch_size}, num_workers={num_workers}, pin_memory={torch.cuda.is_available()}, prefetch={DATALOADER_PREFETCH_FACTOR if int(num_workers) > 0 else 0}, cache_audio={cache_audio}")\n\n    loaders = {}\n    for name, df in splits.items():\n        ds = MusicGapDataset(df, GAP_DURATIONS_MS, sr=TARGET_SR, cache_audio=cache_audio)\n        loaders[name] = build_audio_loader(\n            ds,\n            batch_size=batch_size,\n            shuffle=(name == "train"),\n            num_workers=num_workers,\n        )\n    return loaders\n\n\ndef _artifact_safe_key(value):\n    """Return a filesystem-safe key while preserving model-level uniqueness."""\n    raw = str(value).strip()\n    safe = "".join(ch if ch.isalnum() or ch in {"_", "-"} else "_" for ch in raw)\n    return safe or "unknown"\n\n\ndef _encoder_cache_path(model_name, split_name, dataset_len):\n    encoder_key = _artifact_safe_key(model_name)\n    gaps = "-".join(str(g) for g in GAP_DURATIONS_MS)\n    config_key = EXPERIMENT_CONFIG_ID.replace(".", "p")\n    cache_dir = os.path.join(PATHS["preprocessed"], "encoder_latents")\n    os.makedirs(cache_dir, exist_ok=True)\n    return os.path.join(\n        cache_dir,\n        f"{encoder_key}_{split_name}_n{dataset_len}_{config_key}_gaps{gaps}.pt",\n    )\n\n\ndef precompute_encoder_latents_for_dataset(base_dataset, encoder_fn, device, batch_size, num_workers,\n                                           model_name, split_name):\n    """Precompute frozen CLAP/AudioMAE latents so training loop no longer runs CPU preprocessing."""\n    num_workers = ENCODER_PRECOMPUTE_NUM_WORKERS if num_workers is None else min(\n        int(num_workers),\n        ENCODER_PRECOMPUTE_NUM_WORKERS,\n    )\n    cache_path = _encoder_cache_path(model_name, split_name, len(base_dataset))\n    if os.path.exists(cache_path):\n        print(f"   Loading cached encoder latents [{split_name}]: {cache_path}")\n        return torch.load(cache_path, map_location="cpu", weights_only=False)\n\n    print(f"   Precomputing encoder latents [{split_name}] ({len(base_dataset)} items, num_workers={num_workers})...")\n    precompute_loader = build_audio_loader(\n        base_dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=num_workers,\n    )\n    latents = []\n    start = time.perf_counter()\n    with torch.inference_mode():\n        for batch in tqdm(precompute_loader, desc=f"Precompute {model_name}:{split_name}", leave=False):\n            masked = batch["masked"].to(device, non_blocking=True)\n            z = encoder_fn(masked)\n            latents.append(z.detach().cpu().float())\n    latents = torch.cat(latents, dim=0)\n    torch.save(latents, cache_path)\n    print(f"   Saved encoder latents [{split_name}] in {format_duration(time.perf_counter() - start)}: {cache_path}")\n    return latents\n\n\ndef add_encoder_cache_to_loaders(loaders, encoder_fn, device, model_name, num_workers=AUTO_NUM_WORKERS,\n                                 splits=("train", "val")):\n    if not PRECOMPUTE_ENCODER_LATENTS:\n        return loaders\n\n    cached = dict(loaders)\n    for split_name in splits:\n        if split_name not in loaders:\n            continue\n        loader = loaders[split_name]\n        latents = precompute_encoder_latents_for_dataset(\n            loader.dataset,\n            encoder_fn,\n            device,\n            batch_size=loader.batch_size,\n            num_workers=num_workers,\n            model_name=model_name,\n            split_name=split_name,\n        )\n        cached_ds = EncoderCachedDataset(loader.dataset, latents)\n        cached[split_name] = build_audio_loader(\n            cached_ds,\n            batch_size=loader.batch_size,\n            shuffle=(split_name == "train"),\n            num_workers=num_workers,\n        )\n    return cached\n\n\nprint("✅ Dataset, DataLoader & shared utilities berhasil didefinisikan!")\nprint(f"   Gap durations: {GAP_DURATIONS_MS} ms")\nprint(f"   AUTO_NUM_WORKERS: {AUTO_NUM_WORKERS}")\nprint(f"   ENCODER_PRECOMPUTE_NUM_WORKERS: {ENCODER_PRECOMPUTE_NUM_WORKERS}")\nprint(f"   CPU thread env limit: {CPU_THREAD_LIMIT}")\n\n\n# ---\n# ## CELL 6.6 — Training Loop (Reconstruction-Based)\n# \n# **Perubahan penting dari versi sebelumnya:**\n# Versi lama memakai DDPM epsilon prediction untuk training, tapi saat inference\n# langsung pakai output model sebagai audio. Ini menyebabkan **mismatch fatal**:\n# model dilatih memprediksi noise, tapi output-nya dipakai sebagai rekonstruksi.\n# \n# Versi baru ini membedakan objective per decoder:\n# - CQT-Diff hybrid: SSL latent menjadi conditioning denoiser diffusion\n# - Fallback/STFT decoder: prediksi complex STFT clean audio pada gap frames\n# - Training dan inference dijaga selaras per decoder\n# \n# Fitur:\n# - Gap-only reconstruction loss (L1 pada complex STFT real+imag di gap region)\n# - Full audio auxiliary loss (0.1x bobot, buat stabilitas)\n# - Classifier-free guidance dropout (CFG)\n# - Mixed precision training (AMP) + gradient clipping\n# - Separate trainer untuk baseline (tanpa encoder) dan hybrid (dengan encoder)\n\n\n# ============================================================\n# CELL 6.6: TRAINING LOOP (CONDITIONED DIFFUSION / RECONSTRUCTION)\n# ============================================================\n# Training untuk pipeline hybrid SSL + Decoder.\n#\n# PERBAIKAN UTAMA dari versi sebelumnya:\n# - CQT-Diff hybrid memakai diffusion_loss sehingga SSL conditioning dipelajari\n#   di dalam denoiser/sampler diffusion.\n# - Decoder fallback tetap memakai rekonstruksi STFT yang selaras dengan\n#   inference masing-masing.\n# - Loss dihitung pada gap region.\n# - CFG dropout tetap dipertahankan\n# ============================================================\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport numpy as np\nimport random\nimport os\nfrom tqdm import tqdm\nimport time\n\n\ndef compute_stft_target(clean_audio, n_fft=2048, hop_length=512):\n    """Hitung target complex STFT real+imag dari clean audio."""\n    window = _get_hann_window(n_fft, clean_audio.device)\n    spec = torch.stft(\n        clean_audio, n_fft=n_fft, hop_length=hop_length, window=window,\n        return_complex=True\n    )\n    spec_ri = torch.view_as_real(spec).permute(0, 1, 3, 2).contiguous()\n    return spec_ri.reshape(spec.shape[0], spec.shape[1] * 2, spec.shape[2])  # (B, 2F, T)\n\n\ndef compute_frame_mask(sample_mask, n_frames, hop_length=512):\n    """Konversi sample-level mask -> frame-level mask dengan satu GPU op."""\n    mask_f = sample_mask.float().unsqueeze(1)\n    pooled = F.avg_pool1d(mask_f, kernel_size=hop_length, stride=hop_length, ceil_mode=True).squeeze(1)\n    if pooled.shape[1] < n_frames:\n        pooled = F.pad(pooled, (0, n_frames - pooled.shape[1]))\n    elif pooled.shape[1] > n_frames:\n        pooled = pooled[:, :n_frames]\n    return pooled > 0.5  # (B, T) boolean\n\n\nCQT_WAVEFORM_GAP_LOSS_WEIGHT = float(os.environ.get("CQT_WAVEFORM_GAP_LOSS_WEIGHT", "0.1"))\nCQT_ENERGY_LOSS_WEIGHT = float(os.environ.get("CQT_ENERGY_LOSS_WEIGHT", "0.05"))\n\n\ndef spec_features_to_waveform(pred_spec, audio_length, n_fft=2048, hop_length=512):\n    """Convert predicted real+imag STFT features (B, T, 2F) back to waveform."""\n    with torch.autocast(device_type="cuda" if pred_spec.device.type == "cuda" else "cpu", enabled=False):\n        pred_spec = pred_spec.float()\n        batch_size, n_frames, two_freq = pred_spec.shape\n        freq_bins = two_freq // 2\n        pred_pairs = pred_spec.reshape(batch_size, n_frames, freq_bins, 2).permute(0, 2, 1, 3).contiguous()\n        complex_spec = torch.view_as_complex(pred_pairs)\n        window = _get_hann_window(n_fft, pred_spec.device)\n        return torch.istft(\n            complex_spec,\n            n_fft=n_fft,\n            hop_length=hop_length,\n            window=window,\n            length=audio_length,\n        )\n\n\ndef compute_waveform_gap_losses(pred_spec, clean_audio, sample_mask):\n    """Waveform-domain gap loss plus log-RMS energy loss to discourage silent gaps."""\n    pred_wave = spec_features_to_waveform(pred_spec, clean_audio.shape[-1])\n    mask = sample_mask.bool()\n    waveform_gap_loss = F.l1_loss(pred_wave[mask], clean_audio[mask])\n\n    mask_f = mask.float()\n    denom = mask_f.sum(dim=1).clamp_min(1.0)\n    pred_rms = torch.sqrt(((pred_wave * mask_f).pow(2).sum(dim=1) / denom).clamp_min(1e-10))\n    target_rms = torch.sqrt(((clean_audio * mask_f).pow(2).sum(dim=1) / denom).clamp_min(1e-10))\n    energy_loss = F.l1_loss(torch.log(pred_rms + 1e-5), torch.log(target_rms + 1e-5))\n    return pred_wave, waveform_gap_loss, energy_loss\n\n\nPROFILE_EVERY_N_STEPS = 25\n\n\nclass EpochProfiler:\n    """Lightweight timing for data wait, H2D, preprocessing/encoder, and GPU compute."""\n    def __init__(self, profile_every=PROFILE_EVERY_N_STEPS):\n        self.profile_every = max(1, int(profile_every))\n        self.data_load_seconds = 0.0\n        self.h2d_seconds = 0.0\n        self.preprocess_seconds = 0.0\n        self.gpu_compute_seconds = 0.0\n        self.profiled_steps = 0\n        self.total_steps = 0\n\n    def add_data_wait(self, seconds):\n        self.data_load_seconds += float(seconds)\n        self.total_steps += 1\n\n    def should_profile(self, step_idx):\n        return (step_idx % self.profile_every) == 0\n\n    def add_step_profile(self, profile):\n        if not profile:\n            return\n        self.profiled_steps += 1\n        self.h2d_seconds += float(profile.get("h2d_seconds", 0.0))\n        self.preprocess_seconds += float(profile.get("preprocess_seconds", 0.0))\n        self.gpu_compute_seconds += float(profile.get("gpu_compute_seconds", 0.0))\n\n    def summary(self):\n        scale = (self.total_steps / self.profiled_steps) if self.profiled_steps else 0.0\n        h2d = self.h2d_seconds * scale\n        preprocess = self.preprocess_seconds * scale\n        gpu_compute = self.gpu_compute_seconds * scale\n        active = h2d + preprocess + gpu_compute\n        total = self.data_load_seconds + active\n        return {\n            "steps": self.total_steps,\n            "profiled_steps": self.profiled_steps,\n            "data_load_seconds": self.data_load_seconds,\n            "h2d_seconds_est": h2d,\n            "preprocess_seconds_est": preprocess,\n            "gpu_compute_seconds_est": gpu_compute,\n            "gpu_active_share": (gpu_compute / total) if total > 0 else 0.0,\n        }\n\n\ndef _sync_if_profile(profile_step):\n    if profile_step and torch.cuda.is_available():\n        torch.cuda.synchronize()\n\n\ndef get_gpu_utilization_snapshot():\n    try:\n        import subprocess\n        out = subprocess.check_output([\n            "nvidia-smi",\n            "--query-gpu=utilization.gpu,utilization.memory,memory.used,memory.total",\n            "--format=csv,noheader,nounits",\n        ], text=True, timeout=2).strip().splitlines()[0]\n        gpu_util, mem_util, mem_used, mem_total = [part.strip() for part in out.split(",")]\n        return f"nvidia-smi gpu={gpu_util}% mem_util={mem_util}% mem={mem_used}/{mem_total}MB"\n    except Exception:\n        return "nvidia-smi unavailable"\n\n\ndef print_epoch_profile(model_name, epoch, summary):\n    print(\n        f"  Profile {model_name} epoch {epoch}: "\n        f"data_wait={summary[\'data_load_seconds\']:.2f}s | "\n        f"h2d~={summary[\'h2d_seconds_est\']:.2f}s | "\n        f"preproc/encoder~={summary[\'preprocess_seconds_est\']:.2f}s | "\n        f"gpu_compute~={summary[\'gpu_compute_seconds_est\']:.2f}s | "\n        f"gpu_active_share~={summary[\'gpu_active_share\']:.1%} | "\n        f"peak_vram={get_peak_vram_gb():.2f}GB | {get_gpu_utilization_snapshot()}"\n    )\n\n\nTRAIN_DIAGNOSTICS_ENABLED = os.environ.get("TRAIN_DIAGNOSTICS_ENABLED", "1").lower() in {\n    "1",\n    "true",\n    "yes",\n}\nTRAIN_DIAGNOSTICS_EVERY_EPOCHS = int(os.environ.get("TRAIN_DIAGNOSTICS_EVERY_EPOCHS", "1"))\nTRAIN_DIAGNOSTICS_MAX_UPDATE_ELEMENTS = int(os.environ.get("TRAIN_DIAGNOSTICS_MAX_UPDATE_ELEMENTS", "20000000"))\n\n\ndef _named_trainable_parameters(module_items):\n    named_params = []\n    for prefix, module in module_items:\n        if module is None:\n            continue\n        for name, param in module.named_parameters():\n            if param.requires_grad:\n                named_params.append((f"{prefix}.{name}", param))\n    return named_params\n\n\ndef _module_param_count(module):\n    total = sum(p.numel() for p in module.parameters())\n    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)\n    return total, trainable\n\n\ndef make_trainable_optimizer(model_name, module_items, lr, weight_decay=1e-4):\n    """Create AdamW over parameters that can actually receive gradients."""\n    named_params = _named_trainable_parameters(module_items)\n    if not named_params:\n        raise RuntimeError(\n            f"{model_name}: tidak ada parameter trainable. "\n            "Cek requires_grad pada decoder/FiLM sebelum training."\n        )\n\n    total_trainable = sum(param.numel() for _, param in named_params)\n    print(f"Trainable setup {model_name}:")\n    for prefix, module in module_items:\n        if module is None:\n            continue\n        total, trainable = _module_param_count(module)\n        print(f"  - {prefix}: trainable={trainable:,}/{total:,} params")\n    preview = ", ".join(name for name, _ in named_params[:6])\n    if len(named_params) > 6:\n        preview += ", ..."\n    print(f"  Optimizer receives {len(named_params)} tensors / {total_trainable:,} trainable params")\n    print(f"  Trainable parameter preview: {preview}")\n\n    optimizer = torch.optim.AdamW([param for _, param in named_params], lr=lr, weight_decay=weight_decay)\n    return optimizer, named_params\n\n\ndef _clone_params_for_update(named_params):\n    total_elements = sum(param.numel() for _, param in named_params)\n    if total_elements > TRAIN_DIAGNOSTICS_MAX_UPDATE_ELEMENTS:\n        return None, total_elements\n    return [(name, param.detach().float().clone()) for name, param in named_params], total_elements\n\n\ndef _gradient_diagnostics(named_params):\n    grad_sq = 0.0\n    max_abs = 0.0\n    none_count = 0\n    zero_count = 0\n    tensor_count = 0\n    for _, param in named_params:\n        tensor_count += 1\n        if param.grad is None:\n            none_count += 1\n            continue\n        grad = param.grad.detach().float()\n        grad_norm = float(torch.linalg.vector_norm(grad).item())\n        grad_sq += grad_norm ** 2\n        max_abs = max(max_abs, float(grad.abs().max().item()))\n        if grad_norm == 0.0:\n            zero_count += 1\n    return {\n        "grad_l2": grad_sq ** 0.5,\n        "grad_max_abs": max_abs,\n        "grad_none_tensors": none_count,\n        "grad_zero_tensors": zero_count,\n        "grad_tensor_count": tensor_count,\n    }\n\n\ndef _update_diagnostics(snapshot, named_params, total_elements):\n    if snapshot is None:\n        return {\n            "param_update_l2": None,\n            "param_l2": None,\n            "param_update_ratio": None,\n            "param_update_skipped_elements": total_elements,\n        }\n\n    current_by_name = {name: param for name, param in named_params}\n    update_sq = 0.0\n    param_sq = 0.0\n    for name, before in snapshot:\n        param = current_by_name[name].detach().float()\n        before = before.to(param.device)\n        update_sq += float(torch.sum((param - before) ** 2).item())\n        param_sq += float(torch.sum(param ** 2).item())\n    update_l2 = update_sq ** 0.5\n    param_l2 = param_sq ** 0.5\n    return {\n        "param_update_l2": update_l2,\n        "param_l2": param_l2,\n        "param_update_ratio": update_l2 / max(param_l2, 1e-12),\n        "param_update_skipped_elements": 0,\n    }\n\n\ndef _should_collect_train_diagnostics(epoch, step_idx):\n    if not TRAIN_DIAGNOSTICS_ENABLED:\n        return False\n    if step_idx != 0:\n        return False\n    return (epoch % max(1, TRAIN_DIAGNOSTICS_EVERY_EPOCHS)) == 0\n\n\ndef _add_epoch_metric(epoch_metrics, metrics):\n    for key, value in metrics.items():\n        if key.startswith("_"):\n            continue\n        if isinstance(value, (int, float, np.floating)) and np.isfinite(value):\n            epoch_metrics.setdefault(key, []).append(float(value))\n\n\ndef _mean_epoch_metrics(epoch_metrics):\n    return {\n        f"avg_{key}": float(np.mean(values))\n        for key, values in epoch_metrics.items()\n        if values\n    }\n\n\ndef print_epoch_loss_breakdown(model_name, epoch, metric_means):\n    keys = [\n        "avg_gap_loss",\n        "avg_full_loss",\n        "avg_waveform_gap_loss",\n        "avg_energy_loss",\n        "avg_condition_gate_mean",\n        "avg_conditioned_residual_rms",\n    ]\n    parts = []\n    for key in keys:\n        value = metric_means.get(key)\n        if value is not None and np.isfinite(value):\n            parts.append(f"{key.replace(\'avg_\', \'\')}={value:.6g}")\n    if parts:\n        print(f"  Loss parts {model_name} epoch {epoch}: " + " | ".join(parts))\n\n\ndef print_train_diagnostics(model_name, epoch, diagnostics):\n    if not diagnostics:\n        return\n    msg = (\n        f"  Train diagnostics {model_name} epoch {epoch}: "\n        f"grad_l2={diagnostics.get(\'grad_l2\', 0.0):.3e} | "\n        f"grad_max={diagnostics.get(\'grad_max_abs\', 0.0):.3e} | "\n        f"none_grad={diagnostics.get(\'grad_none_tensors\', 0)}/{diagnostics.get(\'grad_tensor_count\', 0)} | "\n        f"zero_grad={diagnostics.get(\'grad_zero_tensors\', 0)}/{diagnostics.get(\'grad_tensor_count\', 0)}"\n    )\n    if diagnostics.get("param_update_l2") is not None:\n        msg += (\n            f" | update_l2={diagnostics[\'param_update_l2\']:.3e} "\n            f"(ratio={diagnostics[\'param_update_ratio\']:.3e})"\n        )\n    else:\n        msg += f" | update_l2=skipped({diagnostics.get(\'param_update_skipped_elements\', 0):,} elems)"\n    print(msg)\n\n\ndef trainable_parameter_fingerprint(module_items):\n    """Small numeric fingerprint for checking whether saved trainable weights move."""\n    l2_sq = 0.0\n    checksum = 0.0\n    count = 0\n    with torch.no_grad():\n        for _, param in _named_trainable_parameters(module_items):\n            data = param.detach().float()\n            l2_sq += float(torch.sum(data ** 2).item())\n            checksum += float(torch.sum(data).item())\n            count += int(data.numel())\n    return {\n        "trainable_param_count": count,\n        "trainable_param_l2": l2_sq ** 0.5,\n        "trainable_param_checksum": checksum,\n    }\n\n\ndef train_step_reconstruction(decoder, encoder_fn, film, batch, optimizer, scaler,\n                              cfg_drop=0.1, device="cuda", profile_step=False,\n                              diagnostic_named_params=None, collect_diagnostics=False):\n    """Satu step training hybrid: CQT uses conditioned diffusion loss, fallback uses STFT loss."""\n    decoder.train()\n    film.train()\n    profile = {}\n\n    _sync_if_profile(profile_step)\n    t0 = time.perf_counter()\n    clean = batch["clean"].to(device, non_blocking=True)\n    masked = batch["masked"].to(device, non_blocking=True)\n    mask = batch["mask"].to(device, non_blocking=True)\n    cached_z = batch.get("encoder_latent")\n    if cached_z is not None:\n        cached_z = cached_z.to(device, non_blocking=True)\n    _sync_if_profile(profile_step)\n    if profile_step:\n        profile["h2d_seconds"] = time.perf_counter() - t0\n\n    B = clean.size(0)\n\n    _sync_if_profile(profile_step)\n    t_pre = time.perf_counter()\n    with torch.no_grad():\n        z = cached_z if cached_z is not None else encoder_fn(masked)\n    _sync_if_profile(profile_step)\n    if profile_step:\n        profile["preprocess_seconds"] = time.perf_counter() - t_pre\n\n    drop = (torch.rand(B, device=device) < cfg_drop).view(-1, *([1] * (z.dim() - 1)))\n    z = torch.where(drop, torch.zeros_like(z), z)\n\n    _sync_if_profile(profile_step)\n    t_gpu = time.perf_counter()\n    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=scaler.is_enabled()):\n        # SSL conditioning dipakai oleh diffusion denoiser untuk belajar,\n        # bukan sebagai post-processing/refinement setelah hasil CQT-Diff keluar.\n        features = decoder.get_features(masked, mask)\n        cond_features = film(z.float(), features)\n        if hasattr(decoder, "diffusion_loss"):\n            loss_parts = decoder.diffusion_loss(\n                clean,\n                masked,\n                mask=mask,\n                conditioning=cond_features,\n            )\n            loss = loss_parts["loss"]\n            gap_loss = loss_parts.get("gap_loss", loss)\n            full_loss = loss_parts.get("full_loss", loss)\n            waveform_gap_loss = loss_parts.get("waveform_gap_loss", torch.zeros_like(loss))\n            energy_loss = loss_parts.get("energy_loss", torch.zeros_like(loss))\n        else:\n            pred_spec = decoder.decode_features(cond_features)\n            target_spec = compute_stft_target(clean).permute(0, 2, 1)\n\n            T_min = min(pred_spec.shape[1], target_spec.shape[1])\n            pred_spec = pred_spec[:, :T_min, :]\n            target_spec = target_spec[:, :T_min, :]\n\n            frame_mask = compute_frame_mask(mask, T_min).unsqueeze(-1).expand_as(pred_spec)\n            gap_loss = F.l1_loss(pred_spec[frame_mask], target_spec[frame_mask])\n            full_loss = F.l1_loss(pred_spec, target_spec)\n            _, waveform_gap_loss, energy_loss = compute_waveform_gap_losses(pred_spec, clean, mask)\n            loss = (\n                gap_loss\n                + 0.1 * full_loss\n                + CQT_WAVEFORM_GAP_LOSS_WEIGHT * waveform_gap_loss\n                + CQT_ENERGY_LOSS_WEIGHT * energy_loss\n            )\n\n    diagnostics = {}\n    update_snapshot = None\n    update_elements = 0\n    if collect_diagnostics and diagnostic_named_params is not None:\n        update_snapshot, update_elements = _clone_params_for_update(diagnostic_named_params)\n\n    optimizer.zero_grad(set_to_none=True)\n    scaler.scale(loss).backward()\n    scaler.unscale_(optimizer)\n    clip_params = (\n        [param for _, param in diagnostic_named_params]\n        if diagnostic_named_params is not None\n        else [p for p in list(decoder.parameters()) + list(film.parameters()) if p.requires_grad]\n    )\n    if collect_diagnostics and diagnostic_named_params is not None:\n        diagnostics.update(_gradient_diagnostics(diagnostic_named_params))\n    torch.nn.utils.clip_grad_norm_(clip_params, max_norm=1.0)\n    scaler.step(optimizer)\n    scaler.update()\n    if collect_diagnostics and diagnostic_named_params is not None:\n        diagnostics.update(_update_diagnostics(update_snapshot, diagnostic_named_params, update_elements))\n    _sync_if_profile(profile_step)\n    if profile_step:\n        profile["gpu_compute_seconds"] = time.perf_counter() - t_gpu\n\n    result = {\n        "loss": loss.item(),\n        "gap_loss": gap_loss.item(),\n        "full_loss": full_loss.item(),\n        "waveform_gap_loss": waveform_gap_loss.item(),\n        "energy_loss": energy_loss.item(),\n        "_profile": profile,\n    }\n    if collect_diagnostics:\n        result["_diagnostics"] = diagnostics\n    for key in ("condition_gate_mean", "conditioned_residual_rms", "conditioning_wave_rms"):\n        if "loss_parts" in locals() and key in loss_parts:\n            value = loss_parts[key]\n            result[key] = value.item() if isinstance(value, torch.Tensor) else float(value)\n    return result\n\n\ndef train_step_baseline(decoder, batch, optimizer, scaler, device="cuda", profile_step=False,\n                        diagnostic_named_params=None, collect_diagnostics=False):\n    """Training step buat baseline (tanpa encoder, tanpa FiLM)."""\n    decoder.train()\n    profile = {}\n\n    _sync_if_profile(profile_step)\n    t0 = time.perf_counter()\n    clean = batch["clean"].to(device, non_blocking=True)\n    masked = batch["masked"].to(device, non_blocking=True)\n    mask = batch["mask"].to(device, non_blocking=True)\n    _sync_if_profile(profile_step)\n    if profile_step:\n        profile["h2d_seconds"] = time.perf_counter() - t0\n        profile["preprocess_seconds"] = 0.0\n\n    _sync_if_profile(profile_step)\n    t_gpu = time.perf_counter()\n    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=scaler.is_enabled()):\n        features = decoder.get_features(masked, mask)\n        pred_spec = decoder.decode_features(features)\n        target_spec = compute_stft_target(clean).permute(0, 2, 1)\n\n        T_min = min(pred_spec.shape[1], target_spec.shape[1])\n        pred_spec = pred_spec[:, :T_min, :]\n        target_spec = target_spec[:, :T_min, :]\n\n        frame_mask = compute_frame_mask(mask, T_min).unsqueeze(-1).expand_as(pred_spec)\n        gap_loss = F.l1_loss(pred_spec[frame_mask], target_spec[frame_mask])\n        full_loss = F.l1_loss(pred_spec, target_spec)\n        _, waveform_gap_loss, energy_loss = compute_waveform_gap_losses(pred_spec, clean, mask)\n        loss = (\n            gap_loss\n            + 0.1 * full_loss\n            + CQT_WAVEFORM_GAP_LOSS_WEIGHT * waveform_gap_loss\n            + CQT_ENERGY_LOSS_WEIGHT * energy_loss\n        )\n\n    diagnostics = {}\n    update_snapshot = None\n    update_elements = 0\n    if collect_diagnostics and diagnostic_named_params is not None:\n        update_snapshot, update_elements = _clone_params_for_update(diagnostic_named_params)\n\n    optimizer.zero_grad(set_to_none=True)\n    scaler.scale(loss).backward()\n    scaler.unscale_(optimizer)\n    clip_params = (\n        [param for _, param in diagnostic_named_params]\n        if diagnostic_named_params is not None\n        else [p for p in decoder.parameters() if p.requires_grad]\n    )\n    if collect_diagnostics and diagnostic_named_params is not None:\n        diagnostics.update(_gradient_diagnostics(diagnostic_named_params))\n    torch.nn.utils.clip_grad_norm_(clip_params, max_norm=1.0)\n    scaler.step(optimizer)\n    scaler.update()\n    if collect_diagnostics and diagnostic_named_params is not None:\n        diagnostics.update(_update_diagnostics(update_snapshot, diagnostic_named_params, update_elements))\n    _sync_if_profile(profile_step)\n    if profile_step:\n        profile["gpu_compute_seconds"] = time.perf_counter() - t_gpu\n\n    result = {\n        "loss": loss.item(),\n        "gap_loss": gap_loss.item(),\n        "full_loss": full_loss.item(),\n        "waveform_gap_loss": waveform_gap_loss.item(),\n        "energy_loss": energy_loss.item(),\n        "_profile": profile,\n    }\n    if collect_diagnostics:\n        result["_diagnostics"] = diagnostics\n    return result\n\n\ndef get_training_checkpoint_paths(checkpoint_dir, model_name):\n    os.makedirs(checkpoint_dir, exist_ok=True)\n    return {\n        "latest": os.path.join(checkpoint_dir, f"{model_name}_latest.pt"),\n        "best": os.path.join(checkpoint_dir, f"{model_name}_best.pt"),\n    }\n\n\ndef atomic_torch_save(payload, path):\n    tmp_path = f"{path}.tmp"\n    torch.save(payload, tmp_path)\n    os.replace(tmp_path, path)\n\n\ndef save_training_checkpoint(decoder, optimizer, epoch, metric, checkpoint_dir, model_name,\n                             film=None, scheduler=None, scaler=None, history=None,\n                             best_val_loss=None, is_best=False, metric_name="val_loss"):\n    paths = get_training_checkpoint_paths(checkpoint_dir, model_name)\n    trainable_fingerprint = trainable_parameter_fingerprint(\n        [("decoder", decoder), ("film", film)] if film is not None else [("decoder", decoder)]\n    )\n    payload = {\n        "model_name": model_name,\n        "epoch": int(epoch),\n        "decoder_state": decoder.state_dict(),\n        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,\n        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,\n        "scaler_state": scaler.state_dict() if scaler is not None else None,\n        metric_name: float(metric) if metric is not None and np.isfinite(metric) else None,\n        "best_val_loss": float(best_val_loss) if best_val_loss is not None and np.isfinite(best_val_loss) else None,\n        "best_metrics": {"val_loss": float(best_val_loss)} if best_val_loss is not None and np.isfinite(best_val_loss) else {},\n        "history": history or [],\n        "saved_at": pd.Timestamp.now().isoformat(),\n        "stage": globals().get("PIPELINE_STAGE_NAME", "code_v3"),\n        "architecture": getattr(decoder, "architecture_name", decoder.__class__.__name__),\n        **trainable_fingerprint,\n    }\n    if film is not None:\n        payload["film_state"] = film.state_dict()\n\n    atomic_torch_save(payload, paths["latest"])\n    if is_best:\n        atomic_torch_save(payload, paths["best"])\n        print(f"  Best checkpoint saved: {paths[\'best\']}")\n    print(f"  Latest checkpoint saved: {paths[\'latest\']}")\n    print(\n        "  Checkpoint trainable fingerprint: "\n        f"count={trainable_fingerprint[\'trainable_param_count\']:,} | "\n        f"l2={trainable_fingerprint[\'trainable_param_l2\']:.6e} | "\n        f"checksum={trainable_fingerprint[\'trainable_param_checksum\']:.6e}"\n    )\n    return paths\n\n\ndef load_training_checkpoint_if_available(decoder, optimizer, scheduler, scaler, checkpoint_dir,\n                                          model_name, device, film=None):\n    if not checkpoint_dir:\n        return 0, float("inf"), []\n\n    paths = get_training_checkpoint_paths(checkpoint_dir, model_name)\n    latest_path = paths["latest"]\n    if not os.path.exists(latest_path):\n        return 0, float("inf"), []\n\n    payload = torch.load(latest_path, map_location=device, weights_only=False)\n    expected_arch = getattr(decoder, "architecture_name", None)\n    checkpoint_arch = payload.get("architecture")\n    if expected_arch is not None and checkpoint_arch != expected_arch:\n        print(\n            f"Checkpoint {latest_path} memakai arsitektur lama "\n            f"({checkpoint_arch or \'unknown\'}), training {model_name} dimulai ulang "\n            f"dengan {expected_arch}."\n        )\n        return 0, float("inf"), []\n    decoder.load_state_dict(payload["decoder_state"])\n    if film is not None and payload.get("film_state") is not None:\n        film.load_state_dict(payload["film_state"])\n    if optimizer is not None and payload.get("optimizer_state") is not None:\n        try:\n            optimizer.load_state_dict(payload["optimizer_state"])\n        except ValueError as exc:\n            print(\n                f"Optimizer state checkpoint tidak kompatibel untuk {model_name}; "\n                f"optimizer dibuat ulang dari parameter trainable saat ini. Detail: {exc}"\n            )\n    if scheduler is not None and payload.get("scheduler_state") is not None:\n        scheduler.load_state_dict(payload["scheduler_state"])\n    if scaler is not None and payload.get("scaler_state") is not None:\n        scaler.load_state_dict(payload["scaler_state"])\n\n    start_epoch = int(payload.get("epoch", -1)) + 1\n    best_val_loss = payload.get("best_val_loss")\n    if best_val_loss is None:\n        best_val_loss = payload.get("best_metrics", {}).get("val_loss", payload.get("val_loss", float("inf")))\n    if best_val_loss is None:\n        best_val_loss = float("inf")\n    history = payload.get("history", []) or []\n    print(f"Resuming {model_name} from epoch {start_epoch + 1} using {latest_path}")\n    return start_epoch, float(best_val_loss), history\n\n\nEARLY_STOPPING_ENABLED = os.environ.get("EARLY_STOPPING_ENABLED", "1").lower() in {"1", "true", "yes"}\nEARLY_STOPPING_PATIENCE = int(os.environ.get("EARLY_STOPPING_PATIENCE", "5"))\nEARLY_STOPPING_MIN_DELTA = float(os.environ.get("EARLY_STOPPING_MIN_DELTA", "1e-4"))\nEARLY_STOPPING_MIN_EPOCHS = int(os.environ.get("EARLY_STOPPING_MIN_EPOCHS", "10"))\nVAL_EVERY_EPOCHS = int(os.environ.get("VAL_EVERY_EPOCHS", "1"))\n\n\ndef _count_stale_validations(history, best_val_loss,\n                             min_delta=EARLY_STOPPING_MIN_DELTA):\n    stale = 0\n    for row in reversed(history or []):\n        val = row.get("val_loss")\n        if val is None or not np.isfinite(val):\n            continue\n        if float(val) <= float(best_val_loss) + float(min_delta):\n            break\n        stale += 1\n    return stale\n\n\ndef should_early_stop(history, best_val_loss, current_epoch, model_name):\n    if not EARLY_STOPPING_ENABLED:\n        return False\n    if current_epoch < EARLY_STOPPING_MIN_EPOCHS:\n        return False\n    stale = _count_stale_validations(history, best_val_loss)\n    if stale >= EARLY_STOPPING_PATIENCE:\n        print(\n            f"  Early stopping {model_name}: tidak ada improvement val_loss > "\n            f"{EARLY_STOPPING_MIN_DELTA:g} selama {stale} validasi "\n            f"(patience={EARLY_STOPPING_PATIENCE})."\n        )\n        return True\n    return False\n\n\ndef train_model(decoder, encoder_fn, film, train_loader, val_loader=None,\n                num_epochs=30, lr=1e-4, device="cuda", checkpoint_dir=None,\n                model_name="model", batch_size=None, dataset_fraction=None):\n    """\n    Training loop lengkap untuk hybrid model (encoder + FiLM + decoder).\n    Saves latest checkpoints every epoch and resumes from *_latest.pt.\n    """\n    module_items = [("decoder", decoder), ("film", film)]\n    optimizer, diagnostic_named_params = make_trainable_optimizer(\n        model_name, module_items, lr=lr, weight_decay=1e-4\n    )\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)\n    scaler = torch.amp.GradScaler("cuda", enabled=True)\n\n    reset_peak_vram_stats()\n    start_epoch, best_val_loss, history = load_training_checkpoint_if_available(\n        decoder, optimizer, scheduler, scaler, checkpoint_dir, model_name, device, film=film\n    )\n\n    if start_epoch >= num_epochs:\n        print(f"{model_name} already reached {num_epochs} epochs. No additional training needed.")\n        save_training_history_artifacts(model_name, history)\n        return decoder, film\n\n    training_start = time.perf_counter()\n\n    for epoch in range(start_epoch, num_epochs):\n        epoch_start = time.perf_counter()\n        decoder.train()\n        film.train()\n        epoch_losses = []\n        epoch_metrics = {}\n        epoch_diagnostics = None\n\n        profiler = EpochProfiler()\n        data_wait_start = time.perf_counter()\n        for step_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)):\n            profiler.add_data_wait(time.perf_counter() - data_wait_start)\n            collect_diag = _should_collect_train_diagnostics(epoch + 1, step_idx)\n            metrics = train_step_reconstruction(\n                decoder, encoder_fn, film, batch, optimizer, scaler, device=device,\n                profile_step=profiler.should_profile(step_idx),\n                diagnostic_named_params=diagnostic_named_params,\n                collect_diagnostics=collect_diag,\n            )\n            profiler.add_step_profile(metrics.get("_profile"))\n            epoch_losses.append(metrics["loss"])\n            _add_epoch_metric(epoch_metrics, metrics)\n            if metrics.get("_diagnostics"):\n                epoch_diagnostics = metrics["_diagnostics"]\n            data_wait_start = time.perf_counter()\n\n        avg_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")\n        metric_means = _mean_epoch_metrics(epoch_metrics)\n        epoch_seconds = time.perf_counter() - epoch_start\n        profile_summary = profiler.summary()\n        lr_now = optimizer.param_groups[0]["lr"]\n        print(f"  Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.6f} | LR: {lr_now:.2e} | Time: {format_duration(epoch_seconds)}")\n        print_epoch_loss_breakdown(model_name, epoch + 1, metric_means)\n        print_train_diagnostics(model_name, epoch + 1, epoch_diagnostics)\n        print_epoch_profile(model_name, epoch + 1, profile_summary)\n        scheduler.step()\n\n        val_loss = None\n        is_best = False\n        if val_loader is not None and (epoch + 1) % VAL_EVERY_EPOCHS == 0:\n            val_loss = validate_model(decoder, encoder_fn, film, val_loader, device)\n            print(f"  Val Loss: {val_loss:.6f}")\n            logging.getLogger("music_inpainting").info("%s epoch=%s train_loss=%.6f val_loss=%.6f", model_name, epoch + 1, avg_loss, val_loss)\n            if val_loss < best_val_loss:\n                best_val_loss = val_loss\n                is_best = True\n        else:\n            logging.getLogger("music_inpainting").info("%s epoch=%s train_loss=%.6f", model_name, epoch + 1, avg_loss)\n\n        history.append({\n            "epoch": epoch + 1,\n            "train_loss": avg_loss,\n            "val_loss": val_loss,\n            "epoch_seconds": epoch_seconds,\n            "epoch_time": format_duration(epoch_seconds),\n            "learning_rate": lr_now,\n            "peak_vram_gb": get_peak_vram_gb(),\n            **metric_means,\n            **(epoch_diagnostics or {}),\n            **profile_summary,\n        })\n\n        if checkpoint_dir:\n            save_training_checkpoint(\n                decoder, optimizer, epoch, val_loss if val_loss is not None else avg_loss,\n                checkpoint_dir, model_name, film=film, scheduler=scheduler, scaler=scaler,\n                history=history, best_val_loss=best_val_loss, is_best=is_best,\n            )\n        save_training_history_artifacts(model_name, history)\n        if val_loss is not None and should_early_stop(history, best_val_loss, epoch + 1, model_name):\n            break\n\n    if checkpoint_dir:\n        paths = get_training_checkpoint_paths(checkpoint_dir, model_name)\n        if not os.path.exists(paths["best"]):\n            save_training_checkpoint(\n                decoder, optimizer, num_epochs - 1, history[-1]["train_loss"],\n                checkpoint_dir, model_name, film=film, scheduler=scheduler, scaler=scaler,\n                history=history, best_val_loss=best_val_loss, is_best=True,\n            )\n\n    elapsed_this_run = time.perf_counter() - training_start\n    total_seconds = float(sum(row.get("epoch_seconds", 0.0) for row in history))\n    avg_epoch_seconds = float(np.mean([row.get("epoch_seconds", 0.0) for row in history])) if history else 0.0\n    print(f"\\nTraining time this run {model_name}: {format_duration(elapsed_this_run)}")\n    print(f"Total recorded training time {model_name}: {format_duration(total_seconds)}")\n    print(f"Average epoch time {model_name}: {format_duration(avg_epoch_seconds)}")\n    record_training_timing(\n        model_name, total_seconds, num_epochs, lr,\n        batch_size=batch_size, dataset_fraction=dataset_fraction,\n        best_val_loss=best_val_loss,\n        checkpoint_path=os.path.join(checkpoint_dir, f"{model_name}_best.pt") if checkpoint_dir else None,\n        epoch_times=[row.get("epoch_seconds", 0.0) for row in history],\n        peak_vram_gb=get_peak_vram_gb(),\n    )\n    print(f"\\nTraining complete. Best val loss: {best_val_loss:.6f}")\n    return decoder, film\n\n\ndef train_baseline_model(decoder, train_loader, val_loader=None,\n                         num_epochs=100, lr=1e-4, device="cuda",\n                         checkpoint_dir=None, model_name="baseline_cqtdiff",\n                         batch_size=None, dataset_fraction=None):\n    """\n    Training loop buat baseline (tanpa encoder, tanpa FiLM).\n    Saves latest checkpoints every epoch and resumes from *_latest.pt.\n    """\n    module_items = [("decoder", decoder)]\n    optimizer, diagnostic_named_params = make_trainable_optimizer(\n        model_name, module_items, lr=lr, weight_decay=1e-4\n    )\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)\n    scaler = torch.amp.GradScaler("cuda", enabled=True)\n\n    reset_peak_vram_stats()\n    start_epoch, best_val_loss, history = load_training_checkpoint_if_available(\n        decoder, optimizer, scheduler, scaler, checkpoint_dir, model_name, device, film=None\n    )\n\n    if start_epoch >= num_epochs:\n        print(f"{model_name} already reached {num_epochs} epochs. No additional training needed.")\n        save_training_history_artifacts(model_name, history)\n        return decoder\n\n    training_start = time.perf_counter()\n\n    for epoch in range(start_epoch, num_epochs):\n        epoch_start = time.perf_counter()\n        decoder.train()\n        epoch_losses = []\n        epoch_metrics = {}\n        epoch_diagnostics = None\n\n        profiler = EpochProfiler()\n        data_wait_start = time.perf_counter()\n        for step_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)):\n            profiler.add_data_wait(time.perf_counter() - data_wait_start)\n            collect_diag = _should_collect_train_diagnostics(epoch + 1, step_idx)\n            metrics = train_step_baseline(\n                decoder, batch, optimizer, scaler, device=device,\n                profile_step=profiler.should_profile(step_idx),\n                diagnostic_named_params=diagnostic_named_params,\n                collect_diagnostics=collect_diag,\n            )\n            profiler.add_step_profile(metrics.get("_profile"))\n            epoch_losses.append(metrics["loss"])\n            _add_epoch_metric(epoch_metrics, metrics)\n            if metrics.get("_diagnostics"):\n                epoch_diagnostics = metrics["_diagnostics"]\n            data_wait_start = time.perf_counter()\n\n        avg_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")\n        metric_means = _mean_epoch_metrics(epoch_metrics)\n        epoch_seconds = time.perf_counter() - epoch_start\n        profile_summary = profiler.summary()\n        lr_now = optimizer.param_groups[0]["lr"]\n        print(f"  Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.6f} | LR: {lr_now:.2e} | Time: {format_duration(epoch_seconds)}")\n        print_epoch_loss_breakdown(model_name, epoch + 1, metric_means)\n        print_train_diagnostics(model_name, epoch + 1, epoch_diagnostics)\n        print_epoch_profile(model_name, epoch + 1, profile_summary)\n        scheduler.step()\n\n        val_loss = None\n        is_best = False\n        if val_loader is not None and (epoch + 1) % VAL_EVERY_EPOCHS == 0:\n            val_loss = validate_baseline(decoder, val_loader, device)\n            print(f"  Val Loss: {val_loss:.6f}")\n            logging.getLogger("music_inpainting").info("%s epoch=%s train_loss=%.6f val_loss=%.6f", model_name, epoch + 1, avg_loss, val_loss)\n            if val_loss < best_val_loss:\n                best_val_loss = val_loss\n                is_best = True\n        else:\n            logging.getLogger("music_inpainting").info("%s epoch=%s train_loss=%.6f", model_name, epoch + 1, avg_loss)\n\n        history.append({\n            "epoch": epoch + 1,\n            "train_loss": avg_loss,\n            "val_loss": val_loss,\n            "epoch_seconds": epoch_seconds,\n            "epoch_time": format_duration(epoch_seconds),\n            "learning_rate": lr_now,\n            "peak_vram_gb": get_peak_vram_gb(),\n            **metric_means,\n            **(epoch_diagnostics or {}),\n            **profile_summary,\n        })\n\n        if checkpoint_dir:\n            save_training_checkpoint(\n                decoder, optimizer, epoch, val_loss if val_loss is not None else avg_loss,\n                checkpoint_dir, model_name, scheduler=scheduler, scaler=scaler,\n                history=history, best_val_loss=best_val_loss, is_best=is_best,\n            )\n        save_training_history_artifacts(model_name, history)\n        if val_loss is not None and should_early_stop(history, best_val_loss, epoch + 1, model_name):\n            break\n\n    if checkpoint_dir:\n        paths = get_training_checkpoint_paths(checkpoint_dir, model_name)\n        if not os.path.exists(paths["best"]):\n            save_training_checkpoint(\n                decoder, optimizer, num_epochs - 1, history[-1]["train_loss"],\n                checkpoint_dir, model_name, scheduler=scheduler, scaler=scaler,\n                history=history, best_val_loss=best_val_loss, is_best=True,\n            )\n\n    elapsed_this_run = time.perf_counter() - training_start\n    total_seconds = float(sum(row.get("epoch_seconds", 0.0) for row in history))\n    avg_epoch_seconds = float(np.mean([row.get("epoch_seconds", 0.0) for row in history])) if history else 0.0\n    print(f"\\nTraining time this run {model_name}: {format_duration(elapsed_this_run)}")\n    print(f"Total recorded training time {model_name}: {format_duration(total_seconds)}")\n    print(f"Average epoch time {model_name}: {format_duration(avg_epoch_seconds)}")\n    record_training_timing(\n        model_name, total_seconds, num_epochs, lr,\n        batch_size=batch_size, dataset_fraction=dataset_fraction,\n        best_val_loss=best_val_loss,\n        checkpoint_path=os.path.join(checkpoint_dir, f"{model_name}_best.pt") if checkpoint_dir else None,\n        epoch_times=[row.get("epoch_seconds", 0.0) for row in history],\n        peak_vram_gb=get_peak_vram_gb(),\n    )\n    print(f"\\nBaseline training complete. Best val loss: {best_val_loss:.6f}")\n    return decoder\n\n\ndef validate_model(decoder, encoder_fn, film, val_loader, device):\n    """Validasi hybrid model (dengan encoder + FiLM)."""\n    decoder.eval()\n    film.eval()\n    val_losses = []\n\n    with torch.inference_mode():\n        for batch in val_loader:\n            clean = batch["clean"].to(device, non_blocking=True)\n            masked = batch["masked"].to(device, non_blocking=True)\n            mask = batch["mask"].to(device, non_blocking=True)\n            cached_z = batch.get("encoder_latent")\n            z = cached_z.to(device, non_blocking=True) if cached_z is not None else encoder_fn(masked)\n            features = decoder.get_features(masked, mask)\n            cond_features = film(z.float(), features)\n            if hasattr(decoder, "diffusion_loss"):\n                loss_parts = decoder.diffusion_loss(\n                    clean,\n                    masked,\n                    mask=mask,\n                    conditioning=cond_features,\n                    deterministic_sigma=True,\n                )\n                val_loss = loss_parts["loss"]\n            else:\n                pred_spec = decoder.decode_features(cond_features)\n\n                target_spec = compute_stft_target(clean).permute(0, 2, 1)\n                T_min = min(pred_spec.shape[1], target_spec.shape[1])\n                pred_spec = pred_spec[:, :T_min, :]\n                target_spec = target_spec[:, :T_min, :]\n\n                frame_mask = compute_frame_mask(mask, T_min).unsqueeze(-1).expand_as(pred_spec)\n                gap_loss = F.l1_loss(pred_spec[frame_mask], target_spec[frame_mask])\n                full_loss = F.l1_loss(pred_spec, target_spec)\n                _, waveform_gap_loss, energy_loss = compute_waveform_gap_losses(pred_spec, clean, mask)\n                val_loss = (\n                    gap_loss\n                    + 0.1 * full_loss\n                    + CQT_WAVEFORM_GAP_LOSS_WEIGHT * waveform_gap_loss\n                    + CQT_ENERGY_LOSS_WEIGHT * energy_loss\n                )\n            val_losses.append(val_loss.item())\n\n    return float(np.mean(val_losses))\n\n\ndef validate_baseline(decoder, val_loader, device):\n    """Validasi baseline (tanpa encoder)."""\n    decoder.eval()\n    val_losses = []\n\n    with torch.inference_mode():\n        for batch in val_loader:\n            clean = batch["clean"].to(device, non_blocking=True)\n            masked = batch["masked"].to(device, non_blocking=True)\n            mask = batch["mask"].to(device, non_blocking=True)\n\n            features = decoder.get_features(masked, mask)\n            pred_spec = decoder.decode_features(features)\n\n            target_spec = compute_stft_target(clean).permute(0, 2, 1)\n            T_min = min(pred_spec.shape[1], target_spec.shape[1])\n            pred_spec = pred_spec[:, :T_min, :]\n            target_spec = target_spec[:, :T_min, :]\n\n            frame_mask = compute_frame_mask(mask, T_min).unsqueeze(-1).expand_as(pred_spec)\n            gap_loss = F.l1_loss(pred_spec[frame_mask], target_spec[frame_mask])\n            full_loss = F.l1_loss(pred_spec, target_spec)\n            _, waveform_gap_loss, energy_loss = compute_waveform_gap_losses(pred_spec, clean, mask)\n            val_loss = (\n                gap_loss\n                + 0.1 * full_loss\n                + CQT_WAVEFORM_GAP_LOSS_WEIGHT * waveform_gap_loss\n                + CQT_ENERGY_LOSS_WEIGHT * energy_loss\n            )\n            val_losses.append(val_loss.item())\n\n    return float(np.mean(val_losses))\n\n\ndef save_checkpoint(decoder, film, optimizer, epoch, val_loss, checkpoint_dir, model_name):\n    """Simpan checkpoint model."""\n    os.makedirs(checkpoint_dir, exist_ok=True)\n    ckpt_path = os.path.join(checkpoint_dir, f"{model_name}_best.pt")\n    torch.save({\n        "epoch": epoch,\n        "decoder_state": decoder.state_dict(),\n        "film_state": film.state_dict(),\n        "optimizer_state": optimizer.state_dict(),\n        "val_loss": val_loss,\n    }, ckpt_path)\n    print(f"  💾 Best checkpoint saved: {ckpt_path}")\n\n\nprint("✅ Training loop (conditioned diffusion / reconstruction) berhasil didefinisikan!")\nprint("   Komponen: train_step_reconstruction, train_step_baseline,")\nprint("             train_model, train_baseline_model, validate_model")\n\n\n# ============================================================\n# CELL 6.7: SHARED HYBRID TRAINING HELPERS\n# ============================================================\n# Helper bersama untuk:\n# - builder encoder batch-capable (CLAP, AudioMAE)\n# - builder decoder via adapters (CQT-Diff+ original, MAID original DDPM-Midi2Performance)\n# - save/load checkpoint hybrid\n# - trainer minimal-kompatibel untuk MAID\n# - helper evaluasi hybrid yang selalu load checkpoint terlatih\n#\n# PERBAIKAN UTAMA:\n# - proxy/replika decoder dinonaktifkan untuk final pipeline\n# - Training & inference aligned sesuai objective decoder\n# - Mask-aware: model tahu lokasi dan ukuran gap\n# ============================================================\n\nimport os\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport librosa\nimport torchaudio\nimport time\n\nHYBRID_CKPT_SUFFIX = "_best.pt"\n\n\ndef get_model_checkpoint_dir(model_name: str):\n    ckpt_dir = os.path.join(PATHS["checkpoints"], model_name)\n    os.makedirs(ckpt_dir, exist_ok=True)\n    return ckpt_dir\n\n\ndef get_model_checkpoint_path(model_name: str):\n    return os.path.join(get_model_checkpoint_dir(model_name), f"{model_name}{HYBRID_CKPT_SUFFIX}")\n\n\ndef hybrid_checkpoint_exists(model_name: str):\n    return os.path.exists(get_model_checkpoint_path(model_name))\n\n\ndef reset_training_checkpoints_if_requested(model_name: str, force_retrain: bool):\n    """Remove latest/best checkpoints so FORCE_RETRAIN starts from epoch 1."""\n    if not force_retrain:\n        return\n\n    ckpt_dir = get_model_checkpoint_dir(model_name)\n    paths = get_training_checkpoint_paths(ckpt_dir, model_name)\n    removed = []\n    for path in paths.values():\n        if os.path.exists(path):\n            os.remove(path)\n            removed.append(path)\n\n    if removed:\n        print(f"♻️ FORCE_RETRAIN aktif. Checkpoint lama dihapus untuk {model_name}:")\n        for path in removed:\n            print(f"   - {path}")\n\n\ndef load_hybrid_checkpoint(model_name: str, decoder: nn.Module, film_layer: nn.Module, device):\n    ckpt_path = get_model_checkpoint_path(model_name)\n    if not os.path.exists(ckpt_path):\n        raise FileNotFoundError(\n            f"Checkpoint untuk {model_name} belum ada: {ckpt_path}. Jalankan cell training-nya terlebih dahulu."\n        )\n\n    payload = torch.load(ckpt_path, map_location=device, weights_only=False)\n    expected_arch = getattr(decoder, "architecture_name", None)\n    checkpoint_arch = payload.get("architecture")\n    if expected_arch is not None and checkpoint_arch != expected_arch:\n        raise RuntimeError(\n            f"Checkpoint {ckpt_path} memakai arsitektur lama "\n            f"({checkpoint_arch or \'unknown\'}), sedangkan model sekarang {expected_arch}. "\n            "Retrain model ini agar SSL menjadi conditioning diffusion, bukan refinement lama."\n        )\n    if expected_arch is not None and checkpoint_arch is None:\n        decoder.load_state_dict(payload["decoder_state"], strict=False)\n    else:\n        decoder.load_state_dict(payload["decoder_state"])\n    film_layer.load_state_dict(payload["film_state"])\n    print(f"✅ Checkpoint diload: {ckpt_path}")\n    return payload\n\n\ndef load_baseline_checkpoint(decoder: nn.Module, device):\n    """Load checkpoint baseline (tanpa FiLM)."""\n    ckpt_dir = get_model_checkpoint_dir("baseline_cqtdiff")\n    ckpt_path = os.path.join(ckpt_dir, "baseline_cqtdiff_best.pt")\n    if not os.path.exists(ckpt_path):\n        raise FileNotFoundError(f"Baseline checkpoint belum ada: {ckpt_path}")\n    payload = torch.load(ckpt_path, map_location=device, weights_only=False)\n    expected_arch = getattr(decoder, "architecture_name", None)\n    checkpoint_arch = payload.get("architecture")\n    if expected_arch is not None and checkpoint_arch not in {None, expected_arch}:\n        raise RuntimeError(\n            f"Baseline checkpoint {ckpt_path} memakai arsitektur {checkpoint_arch}, "\n            f"sedangkan model sekarang {expected_arch}."\n        )\n    if expected_arch is not None and checkpoint_arch is None:\n        decoder.load_state_dict(payload["decoder_state"], strict=False)\n    else:\n        decoder.load_state_dict(payload["decoder_state"])\n    print(f"✅ Baseline checkpoint diload: {ckpt_path}")\n    return payload\n\n\n_MEL_FILTER_CACHE = {}\n_WINDOW_CACHE = {}\n_INVERSE_MEL_CACHE = {}\n_GRIFFINLIM_CACHE = {}\n\n\ndef _get_hann_window(n_fft, device):\n    key = (n_fft, str(device))\n    window = _WINDOW_CACHE.get(key)\n    if window is None or window.device != device:\n        window = torch.hann_window(n_fft, device=device)\n        _WINDOW_CACHE[key] = window\n    return window\n\n\ndef _get_mel_filter(sr, n_fft, n_mels, device):\n    key = (sr, n_fft, n_mels, str(device))\n    mel = _MEL_FILTER_CACHE.get(key)\n    if mel is None or mel.device != device:\n        # TorchAudio returns (freq, mel); transpose once and cache on-device.\n        mel = torchaudio.functional.melscale_fbanks(\n            n_freqs=n_fft // 2 + 1,\n            f_min=0.0,\n            f_max=sr / 2,\n            n_mels=n_mels,\n            sample_rate=sr,\n            norm="slaney",\n            mel_scale="slaney",\n        ).to(device=device, dtype=torch.float32).transpose(0, 1).contiguous()\n        _MEL_FILTER_CACHE[key] = mel\n    return mel\n\n\ndef _module_device(module, fallback):\n    try:\n        return next(module.parameters()).device\n    except StopIteration:\n        try:\n            return next(module.buffers()).device\n        except StopIteration:\n            return fallback\n\n\ndef _get_inverse_mel_scale(sr, n_fft, n_mels, device):\n    key = (sr, n_fft, n_mels, str(device))\n    inverse = _INVERSE_MEL_CACHE.get(key)\n    if inverse is None or _module_device(inverse, device) != device:\n        inverse = torchaudio.transforms.InverseMelScale(\n            n_stft=n_fft // 2 + 1,\n            n_mels=n_mels,\n            sample_rate=sr,\n            norm="slaney",\n            mel_scale="slaney",\n        ).to(device)\n        _INVERSE_MEL_CACHE[key] = inverse\n    return inverse\n\n\ndef _get_griffinlim(n_fft, hop_length, device):\n    key = (n_fft, hop_length, str(device))\n    griffinlim = _GRIFFINLIM_CACHE.get(key)\n    if griffinlim is None or _module_device(griffinlim, device) != device:\n        griffinlim = torchaudio.transforms.GriffinLim(\n            n_fft=n_fft,\n            n_iter=64,\n            win_length=n_fft,\n            hop_length=hop_length,\n            power=1.0,\n            momentum=0.99,\n        ).to(device)\n        _GRIFFINLIM_CACHE[key] = griffinlim\n    return griffinlim\n\n\ndef torch_audio_to_mel_batch(audio_batch, sr: int = TARGET_SR, n_mels: int = 128,\n                             n_fft: int = 2048, hop_length: int = 512, device=None):\n    """GPU mel-power/log-mel path matching librosa power_to_db(ref=np.max) semantics."""\n    if isinstance(audio_batch, torch.Tensor):\n        if device is None:\n            device = audio_batch.device\n        audio = audio_batch.to(device=device, dtype=torch.float32, non_blocking=True)\n    else:\n        audio = torch.as_tensor(np.stack(ensure_audio_list(audio_batch), axis=0), dtype=torch.float32, device=device)\n\n    if audio.dim() == 1:\n        audio = audio.unsqueeze(0)\n    if audio.dim() == 3:\n        audio = audio.squeeze(1)\n\n    autocast_device = "cuda" if audio.device.type == "cuda" else "cpu"\n    with torch.autocast(device_type=autocast_device, enabled=False):\n        audio = audio.float()\n        window = _get_hann_window(n_fft, audio.device)\n        spec = torch.stft(\n            audio,\n            n_fft=n_fft,\n            hop_length=hop_length,\n            window=window,\n            return_complex=True,\n        )\n        power = spec.abs().pow(2.0)\n        mel_basis = _get_mel_filter(sr, n_fft, n_mels, audio.device)\n        mel_power = torch.einsum("mf,bft->bmt", mel_basis, power).clamp_min(1e-10)\n\n        mel_db_raw = 10.0 * torch.log10(mel_power)\n        ref_db = mel_db_raw.amax(dim=(1, 2), keepdim=True)\n        mel_db = torch.clamp(mel_db_raw - ref_db, min=-80.0)\n        mel_norm = (mel_db + 40.0) / 40.0\n    return mel_db, mel_norm\n\n\n\n\n\ndef ensure_audio_list(audio_batch):\n    if isinstance(audio_batch, torch.Tensor):\n        audio_batch = audio_batch.detach().cpu().numpy()\n\n    if isinstance(audio_batch, np.ndarray):\n        if audio_batch.ndim == 1:\n            return [audio_batch.astype(np.float32)]\n        return [sample.astype(np.float32) for sample in audio_batch]\n\n    if isinstance(audio_batch, (list, tuple)):\n        return [np.asarray(sample, dtype=np.float32) for sample in audio_batch]\n\n    raise TypeError(f"Tipe audio batch tidak didukung: {type(audio_batch)}")\n\n\n\ndef build_film_layer(model_name: str, device):\n    cfg = FILM_CONFIGS[model_name]\n    return FiLMLayer(\n        encoder_dim=cfg["encoder_dim"],\n        decoder_feature_dim=cfg["decoder_feature_dim"],\n    ).to(device)\n\n\ndef _load_official_adapter(module_name: str, builder_name: str, model_label: str):\n    """Load an official-model adapter and fail loudly if it is not configured."""\n    import importlib\n\n    try:\n        module = importlib.import_module(module_name)\n    except Exception as exc:\n        raise RuntimeError(\n            f"{model_label} harus memakai model asli, tetapi adapter \'{module_name}\' "\n            f"belum tersedia/importable. Buat module adapter tersebut atau set env var "\n            f"yang sesuai sebelum menjalankan pipeline final."\n        ) from exc\n\n    builder = getattr(module, builder_name, None)\n    if not callable(builder):\n        raise RuntimeError(\n            f"Adapter \'{module_name}\' tidak punya fungsi callable \'{builder_name}\' "\n            f"untuk membangun {model_label} asli."\n        )\n    return builder\n\n\ndef _validate_decoder_interface(decoder, required_methods, model_label: str):\n    missing = [name for name in required_methods if not hasattr(decoder, name)]\n    if missing:\n        raise TypeError(\n            f"{model_label} adapter tidak kompatibel dengan training/evaluasi pipeline ini. "\n            f"Method yang belum ada: {missing}"\n        )\n    return decoder\n\n\n\ndef build_hybrid_cqtdiff_decoder(device):\n    builder = _load_official_adapter(\n        OFFICIAL_CQTDIFF_ADAPTER,\n        "build_cqtdiff_decoder",\n        "MusicNet CQTdiff+ original",\n    )\n    decoder = builder(\n        device=device,\n        target_sr=TARGET_SR,\n        segment_samples=SEGMENT_SAMPLES,\n        gap_durations_ms=GAP_DURATIONS_MS,\n        cqt_diff_dir=CQT_DIFF_DIR,\n    )\n    decoder = _validate_decoder_interface(\n        decoder,\n        ["get_features", "decode_features", "inpaint", "parameters", "state_dict", "load_state_dict", "train", "eval"],\n        "MusicNet CQTdiff+ original",\n    )\n    decoder.eval()\n    print("MusicNet CQTdiff+ original loaded via official audio-inpainting adapter.")\n    return decoder\n\n\n\ndef build_maid_decoder(device):\n    builder = _load_official_adapter(\n        MAID_ADAPTER,\n        "build_maid_decoder",\n        "MAID original DDPM-Midi2Performance",\n    )\n    decoder = builder(\n        device=device,\n        target_sr=TARGET_SR,\n        segment_samples=SEGMENT_SAMPLES,\n        gap_durations_ms=GAP_DURATIONS_MS,\n        midi2performance_dir=MIDI2PERFORMANCE_DIR,\n    )\n    decoder = _validate_decoder_interface(\n        decoder,\n        ["get_features", "predict_mel_norm", "audio_to_mel_batch", "mask_to_frame_mask", "inpaint",\n         "diffusion_loss", "parameters", "state_dict", "load_state_dict", "train", "eval"],\n        "MAID original DDPM-Midi2Performance",\n    )\n    decoder.eval()\n    print("MAID original DDPM-Midi2Performance loaded via adapter.")\n    return decoder\n\n\n\ndef build_clap_encoder(device):\n    from transformers import ClapModel, ClapProcessor\n\n    torch_dtype = torch.float16\n    processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")\n    model = ClapModel.from_pretrained(\n        "laion/clap-htsat-unfused",\n        torch_dtype=torch_dtype,\n    ).to(device)\n    model.eval()\n\n    def encode(audio_batch, sr: int = TARGET_SR):\n        audio_list = []\n        for audio in ensure_audio_list(audio_batch):\n            if sr != 48000:\n                audio = librosa.resample(audio, orig_sr=sr, target_sr=48000)\n            audio_list.append(audio.astype(np.float32))\n\n        inputs = processor(audio=audio_list, sampling_rate=48000, return_tensors="pt", padding=True)\n        inputs = {\n            key: value.to(device=device, dtype=torch_dtype if value.is_floating_point() else value.dtype)\n            for key, value in inputs.items()\n        }\n\n        with torch.inference_mode():\n            audio_features = model.get_audio_features(**inputs)\n\n        if isinstance(audio_features, torch.Tensor):\n            return audio_features.float()\n        if hasattr(audio_features, "pooler_output"):\n            return audio_features.pooler_output.float()\n        raise TypeError(f"Output CLAP tidak dikenali: {type(audio_features)}")\n\n    return model, encode\n\n\n\ndef build_audiomae_encoder(device):\n    if not os.path.isdir(AUDIO_MAE_DIR):\n        raise RuntimeError(f"Repo AudioMAE tidak ditemukan: {AUDIO_MAE_DIR}")\n    if AUDIO_MAE_DIR not in sys.path:\n        sys.path.insert(0, AUDIO_MAE_DIR)\n    # AudioMAE repo targets older PyTorch where torch._six still existed.\n    try:\n        import types\n        import math as _math\n        import collections.abc as _container_abcs\n        import numpy as _np\n        if not hasattr(_np, "float"):\n            _np.float = float\n        if "torch._six" not in sys.modules:\n            _six = types.ModuleType("torch._six")\n            _six.inf = _math.inf\n            _six.container_abcs = _container_abcs\n            sys.modules["torch._six"] = _six\n    except Exception:\n        pass\n\n    try:\n        import models_mae\n    except Exception as exc:\n        raise RuntimeError(\n            f"AudioMAE harus memakai repo asli di {AUDIO_MAE_DIR}, tetapi import models_mae gagal."\n        ) from exc\n    # AudioMAE was written for older timm where Block accepted qk_scale.\n    try:\n        from timm.models.vision_transformer import Block as _TimmBlock\n\n        class _AudioMAECompatBlock(_TimmBlock):\n            def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=False,\n                         qk_scale=None, norm_layer=nn.LayerNorm, **kwargs):\n                kwargs.pop("qk_scale", None)\n                super().__init__(\n                    dim=dim,\n                    num_heads=num_heads,\n                    mlp_ratio=mlp_ratio,\n                    qkv_bias=qkv_bias,\n                    norm_layer=norm_layer,\n                    **kwargs,\n                )\n\n        models_mae.Block = _AudioMAECompatBlock\n    except Exception as exc:\n        raise RuntimeError("Gagal memasang compatibility shim timm Block untuk AudioMAE.") from exc\n\n    model = models_mae.mae_vit_base_patch16(\n        norm_pix_loss=False,\n        in_chans=1,\n        audio_exp=True,\n        img_size=(1024, 128),\n        alpha=0.0,\n        mode=0,\n        use_custom_patch=False,\n        split_pos=False,\n        pos_trainable=False,\n        use_nce=False,\n        decoder_mode=0,\n        mask_2d=False,\n        mask_t_prob=0.6,\n        mask_f_prob=0.5,\n        no_shift=False,\n    ).to(device)\n\n    ckpt_candidates = [\n        os.environ.get("AUDIOMAE_CHECKPOINT"),\n        os.path.join(AUDIO_MAE_DIR, "ckpt", "pretrained.pth"),\n        os.path.join(AUDIO_MAE_DIR, "ckpt", "finetuned.pth"),\n        os.path.join(AUDIO_MAE_DIR, "pretrained.pth"),\n        os.path.join(AUDIO_MAE_DIR, "finetuned.pth"),\n    ]\n    ckpt_path = next((p for p in ckpt_candidates if p and os.path.exists(p)), None)\n    if ckpt_path is None:\n        raise FileNotFoundError(\n            "Checkpoint AudioMAE asli belum ditemukan. Letakkan checkpoint di "\n            f"{os.path.join(AUDIO_MAE_DIR, \'ckpt\', \'pretrained.pth\')} atau set AUDIOMAE_CHECKPOINT."\n        )\n\n    payload = torch.load(ckpt_path, map_location="cpu", weights_only=False)\n    state = payload.get("model", payload) if isinstance(payload, dict) else payload\n    state = {str(k).replace("module.", "", 1): v for k, v in state.items()}\n    msg = model.load_state_dict(state, strict=False)\n    model.eval()\n    print(f"AudioMAE asli dari repo berhasil diload: {ckpt_path}")\n    print(f"AudioMAE load_state_dict: {msg}")\n\n    fbank_mean = -4.2677393\n    fbank_std = 4.5689974\n\n    def _audio_to_audiomae_input(audio_list, sr):\n        fbanks = []\n        for audio in audio_list:\n            wav = torch.as_tensor(audio, dtype=torch.float32).view(1, -1).cpu()\n            if sr != 16000:\n                wav = torchaudio.functional.resample(wav, sr, 16000)\n            wav = wav - wav.mean()\n            fbank = torchaudio.compliance.kaldi.fbank(\n                wav,\n                htk_compat=True,\n                sample_frequency=16000,\n                use_energy=False,\n                window_type="hanning",\n                num_mel_bins=128,\n                dither=0.0,\n                frame_shift=10,\n            )\n            if fbank.shape[0] < 1024:\n                fbank = F.pad(fbank, (0, 0, 0, 1024 - fbank.shape[0]))\n            elif fbank.shape[0] > 1024:\n                fbank = fbank[:1024, :]\n            fbank = (fbank - fbank_mean) / (fbank_std * 2.0)\n            fbanks.append(fbank)\n        return torch.stack(fbanks, dim=0).unsqueeze(1).to(device, non_blocking=True)\n\n    def encode(audio_batch, sr: int = TARGET_SR):\n        audio_list = ensure_audio_list(audio_batch)\n\n        with torch.inference_mode():\n            inputs = _audio_to_audiomae_input(audio_list, sr)\n            embeddings = model.forward_encoder_no_mask(inputs)\n            return embeddings[:, 0, :].float()\n\n    return model, encode\n\n\n\ndef train_maid_step(decoder, encoder_fn, film, batch, optimizer, scaler, cfg_drop=0.1,\n                    device="cuda", profile_step=False):\n    """Training step MAID: reconstruction loss di mel domain, with GPU mel extraction."""\n    decoder.train()\n    film.train()\n    profile = {}\n\n    _sync_if_profile(profile_step)\n    t0 = time.perf_counter()\n    clean = batch["clean"].to(device, non_blocking=True)\n    masked = batch["masked"].to(device, non_blocking=True)\n    mask = batch["mask"].to(device, non_blocking=True)\n    cached_z = batch.get("encoder_latent")\n    if cached_z is not None:\n        cached_z = cached_z.to(device, non_blocking=True)\n    _sync_if_profile(profile_step)\n    if profile_step:\n        profile["h2d_seconds"] = time.perf_counter() - t0\n\n    batch_size = clean.size(0)\n\n    _sync_if_profile(profile_step)\n    t_pre = time.perf_counter()\n    with torch.no_grad():\n        z = cached_z if cached_z is not None else encoder_fn(masked)\n    _sync_if_profile(profile_step)\n    if profile_step:\n        profile["preprocess_seconds"] = time.perf_counter() - t_pre\n\n    drop = (torch.rand(batch_size, device=device) < cfg_drop).view(-1, *([1] * (z.dim() - 1)))\n    z = torch.where(drop, torch.zeros_like(z), z)\n\n    _sync_if_profile(profile_step)\n    t_gpu = time.perf_counter()\n    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):\n        decoder_features = decoder.get_features(masked, mask)\n        conditioned_features = film(z.float(), decoder_features)\n        if hasattr(decoder, "diffusion_loss"):\n            loss_parts = decoder.diffusion_loss(\n                clean,\n                masked,\n                mask=mask,\n                conditioning=conditioned_features,\n            )\n            loss = loss_parts["loss"]\n            gap_loss = loss_parts.get("gap_loss", loss)\n            full_loss = loss_parts.get("full_loss", loss)\n        else:\n            pred_mel_norm = decoder.predict_mel_norm(masked, conditioning=conditioned_features)\n            _, clean_mel_norm = decoder.audio_to_mel_batch(clean)\n            clean_mel_norm = clean_mel_norm.permute(0, 2, 1)\n\n            frame_mask = decoder.mask_to_frame_mask(mask, pred_mel_norm.shape[1])\n            expanded_mask = frame_mask.unsqueeze(-1).expand_as(pred_mel_norm)\n\n            gap_loss = F.l1_loss(pred_mel_norm[expanded_mask], clean_mel_norm[expanded_mask])\n            full_loss = F.l1_loss(pred_mel_norm, clean_mel_norm)\n            loss = gap_loss + 0.1 * full_loss\n\n    optimizer.zero_grad(set_to_none=True)\n    scaler.scale(loss).backward()\n    scaler.unscale_(optimizer)\n    torch.nn.utils.clip_grad_norm_(\n        list(decoder.parameters()) + list(film.parameters()), max_norm=1.0\n    )\n    scaler.step(optimizer)\n    scaler.update()\n    _sync_if_profile(profile_step)\n    if profile_step:\n        profile["gpu_compute_seconds"] = time.perf_counter() - t_gpu\n\n    return {\n        "loss": loss.item(),\n        "gap_loss": gap_loss.item() if isinstance(gap_loss, torch.Tensor) else float(gap_loss),\n        "full_loss": full_loss.item() if isinstance(full_loss, torch.Tensor) else float(full_loss),\n        "_profile": profile,\n    }\n\n\ndef train_maid_model(decoder, encoder_fn, film, train_loader, val_loader=None,\n                     num_epochs=20, lr=1e-4, device="cuda", checkpoint_dir=None,\n                     model_name="model", batch_size=None, dataset_fraction=None):\n    """Training loop MAID (mel reconstruction) with resume-safe checkpointing."""\n    module_items = [("decoder", decoder), ("film", film)]\n    optimizer, _ = make_trainable_optimizer(\n        model_name, module_items, lr=lr, weight_decay=1e-4\n    )\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)\n    scaler = torch.amp.GradScaler("cuda", enabled=True)\n\n    reset_peak_vram_stats()\n    start_epoch, best_val_loss, history = load_training_checkpoint_if_available(\n        decoder, optimizer, scheduler, scaler, checkpoint_dir, model_name, device, film=film\n    )\n    checkpoint_path = os.path.join(checkpoint_dir, f"{model_name}_best.pt") if checkpoint_dir else None\n\n    if start_epoch >= num_epochs:\n        print(f"{model_name} already reached {num_epochs} epochs. No additional training needed.")\n        save_training_history_artifacts(model_name, history)\n        return decoder, film\n\n    training_start = time.perf_counter()\n\n    for epoch in range(start_epoch, num_epochs):\n        epoch_start = time.perf_counter()\n        decoder.train()\n        film.train()\n        epoch_losses = []\n\n        profiler = EpochProfiler()\n        data_wait_start = time.perf_counter()\n        for step_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)):\n            profiler.add_data_wait(time.perf_counter() - data_wait_start)\n            metrics = train_maid_step(\n                decoder, encoder_fn, film, batch, optimizer, scaler, device=device,\n                profile_step=profiler.should_profile(step_idx),\n            )\n            profiler.add_step_profile(metrics.get("_profile"))\n            epoch_losses.append(metrics["loss"])\n            data_wait_start = time.perf_counter()\n\n        avg_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")\n        epoch_seconds = time.perf_counter() - epoch_start\n        profile_summary = profiler.summary()\n        lr_now = optimizer.param_groups[0]["lr"]\n        print(f"  Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.6f} | LR: {lr_now:.2e} | Time: {format_duration(epoch_seconds)}")\n        print_epoch_profile(model_name, epoch + 1, profile_summary)\n        scheduler.step()\n\n        val_loss = None\n        is_best = False\n        if val_loader is not None and (epoch + 1) % VAL_EVERY_EPOCHS == 0:\n            decoder.eval()\n            film.eval()\n            val_losses = []\n\n            with torch.inference_mode():\n                for batch in val_loader:\n                    clean = batch["clean"].to(device, non_blocking=True)\n                    masked = batch["masked"].to(device, non_blocking=True)\n                    mask_b = batch["mask"].to(device, non_blocking=True)\n                    cached_z = batch.get("encoder_latent")\n                    z = cached_z.to(device, non_blocking=True) if cached_z is not None else encoder_fn(masked)\n                    decoder_features = decoder.get_features(masked, mask_b)\n                    conditioned_features = film(z.float(), decoder_features)\n                    if hasattr(decoder, "diffusion_loss"):\n                        loss_parts = decoder.diffusion_loss(\n                            clean,\n                            masked,\n                            mask=mask_b,\n                            conditioning=conditioned_features,\n                            deterministic_sigma=True,\n                        )\n                        batch_val_loss = loss_parts.get("gap_loss", loss_parts["loss"])\n                    else:\n                        pred_mel_norm = decoder.predict_mel_norm(masked, conditioning=conditioned_features)\n                        _, clean_mel_norm = decoder.audio_to_mel_batch(clean)\n                        clean_mel_norm = clean_mel_norm.permute(0, 2, 1)\n                        frame_mask = decoder.mask_to_frame_mask(mask_b, pred_mel_norm.shape[1])\n                        expanded_mask = frame_mask.unsqueeze(-1).expand_as(pred_mel_norm)\n                        batch_val_loss = F.l1_loss(pred_mel_norm[expanded_mask], clean_mel_norm[expanded_mask])\n                    val_losses.append(batch_val_loss.item())\n\n            val_loss = float(np.mean(val_losses)) if val_losses else float("nan")\n            print(f"  Val Loss: {val_loss:.6f}")\n            logging.getLogger("music_inpainting").info("%s epoch=%s train_loss=%.6f val_loss=%.6f", model_name, epoch + 1, avg_loss, val_loss)\n            if val_loss < best_val_loss:\n                best_val_loss = val_loss\n                is_best = True\n        else:\n            logging.getLogger("music_inpainting").info("%s epoch=%s train_loss=%.6f", model_name, epoch + 1, avg_loss)\n\n        history.append({\n            "epoch": epoch + 1,\n            "train_loss": avg_loss,\n            "val_loss": val_loss,\n            "epoch_seconds": epoch_seconds,\n            "epoch_time": format_duration(epoch_seconds),\n            "learning_rate": lr_now,\n            "peak_vram_gb": get_peak_vram_gb(),\n            **profile_summary,\n        })\n\n        if checkpoint_dir:\n            save_training_checkpoint(\n                decoder, optimizer, epoch, val_loss if val_loss is not None else avg_loss,\n                checkpoint_dir, model_name, film=film, scheduler=scheduler, scaler=scaler,\n                history=history, best_val_loss=best_val_loss, is_best=is_best,\n            )\n        save_training_history_artifacts(model_name, history)\n        if val_loss is not None and should_early_stop(history, best_val_loss, epoch + 1, model_name):\n            break\n\n    if checkpoint_dir:\n        paths = get_training_checkpoint_paths(checkpoint_dir, model_name)\n        if not os.path.exists(paths["best"]):\n            save_training_checkpoint(\n                decoder, optimizer, num_epochs - 1, history[-1]["train_loss"],\n                checkpoint_dir, model_name, film=film, scheduler=scheduler, scaler=scaler,\n                history=history, best_val_loss=best_val_loss, is_best=True,\n            )\n\n    elapsed_this_run = time.perf_counter() - training_start\n    total_seconds = float(sum(row.get("epoch_seconds", 0.0) for row in history))\n    avg_epoch_seconds = float(np.mean([row.get("epoch_seconds", 0.0) for row in history])) if history else 0.0\n    print(f"\\nTraining time this run {model_name}: {format_duration(elapsed_this_run)}")\n    print(f"Total recorded training time {model_name}: {format_duration(total_seconds)}")\n    print(f"Average epoch time {model_name}: {format_duration(avg_epoch_seconds)}")\n    record_training_timing(\n        model_name, total_seconds, num_epochs, lr,\n        batch_size=batch_size, dataset_fraction=dataset_fraction,\n        best_val_loss=best_val_loss, checkpoint_path=checkpoint_path,\n        epoch_times=[row.get("epoch_seconds", 0.0) for row in history],\n        peak_vram_gb=get_peak_vram_gb(),\n    )\n    print(f"\\nMAID training complete. Best val loss: {best_val_loss:.6f}")\n    return decoder, film\n\n\ndef run_hybrid_inpainting_evaluation(model_label: str, encoder_fn, decoder, film_layer, device,\n                                     n_eval_samples: int = 50, model_name: str = None):\n    """\n    Evaluasi hybrid model (encoder + FiLM + decoder).\n\n    Evaluasi hybrid memakai decoder asli melalui adapter resmi.\n    FiLM conditioning di-inject ke fitur decoder sesuai interface adapter.\n    """\n    evaluation_start = time.perf_counter()\n    model_name = model_name or model_name_from_label(model_label)\n    if model_name not in EXPECTED_MODEL_CONFIGS and model_name not in ALL_MODELS:\n        raise ValueError(f"Model name evaluasi tidak dikenal: {model_name}")\n    cache_tag = set_reconstruction_cache_context(\n        model_name,\n        decoder,\n        checkpoint_path=get_model_checkpoint_path(model_name) if hybrid_checkpoint_exists(model_name) else None,\n    )\n    prepare_reconstructed_outputs(model_name, clear=EVAL_CLEAR_RECONSTRUCTIONS)\n    output_manifest_rows = []\n    alignment_rows = []\n    conditioning_rows = []\n    original_audios, masked_by_gap, gap_regions_by_gap = load_preprocessed_data(\n        n_eval_samples, gap_position=EVAL_GAP_POSITION\n    )\n    reconstructed_dict = {}\n    reused_count = 0\n    generated_count = 0\n\n    print(f"\\n🎵 Menjalankan inpainting {model_label}...")\n    print(f"  Cache tag aktif: {cache_tag[:96]}...")\n    if EVAL_REUSE_RECONSTRUCTIONS:\n        print("  Resume rekonstruksi aktif: WAV yang sudah ada akan dipakai ulang.")\n        summarize_reconstruction_cache(model_name, n_eval_samples)\n    if EVAL_CLEAR_RECONSTRUCTIONS:\n        print("  EVAL_CLEAR_RECONSTRUCTIONS aktif: output lama dibersihkan sebelum eval.")\n\n    for gap_ms in GAP_DURATIONS_MS:\n        print(f"\\n  Gap {gap_ms}ms...")\n        reconstructed_list = []\n\n        for index, (orig_audio, masked_audio) in enumerate(zip(original_audios, masked_by_gap[gap_ms])):\n            gap_region = gap_regions_by_gap[gap_ms][index]\n            if (index + 1) % 10 == 0:\n                print(f"    Sample {index+1}/{n_eval_samples}")\n\n            reconstructed, output_path = load_reconstructed_output_if_available(\n                model_name, gap_ms, index, expected_n_samples=len(orig_audio), sr=TARGET_SR\n            )\n\n            if reconstructed is not None:\n                reused_count += 1\n            else:\n                with torch.inference_mode():\n                    # Encode masked audio pakai SSL encoder\n                    encoder_latent = encoder_fn(masked_audio)\n                    if not torch.isfinite(encoder_latent).all():\n                        raise RuntimeError(f"Encoder latent {model_name} mengandung NaN/Inf pada gap={gap_ms}, sample={index}.")\n\n                    masked_tensor = torch.from_numpy(masked_audio).float().unsqueeze(0).to(device)\n                    mask, gap_start, gap_end = build_gap_mask_array(\n                        len(masked_audio), gap_ms, gap_start=gap_region["gap_start"]\n                    )\n                    align = validate_masked_gap_alignment(\n                        orig_audio, masked_audio, mask, gap_ms,\n                        expected_gap_start=gap_start, expected_gap_end=gap_end,\n                    )\n                    align.update({\n                        "model": model_name,\n                        "gap_ms": int(gap_ms),\n                        "sample_index": int(index),\n                        "gap_position": gap_region["gap_position"],\n                    })\n                    alignment_rows.append(align)\n                    mask_tensor = torch.from_numpy(mask).unsqueeze(0).to(device)\n\n                    has_diffusion = hasattr(decoder, "_inpaint_diffusion")\n                    import inspect\n                    sig = inspect.signature(decoder.get_features)\n                    has_mask_param = \'mask\' in sig.parameters\n\n                    if has_diffusion:\n                        # CQT-Diff+: SSL representations condition the diffusion\n                        # sampler directly. Tidak ada post-processing/refinement\n                        # setelah baseline diffusion selesai.\n                        decoder_features = decoder.get_features(masked_tensor, mask_tensor)\n                        conditioned_features = film_layer(encoder_latent.float(), decoder_features)\n                        if not torch.isfinite(decoder_features).all():\n                            raise RuntimeError(f"Decoder features {model_name} mengandung NaN/Inf pada gap={gap_ms}, sample={index}.")\n                        if not torch.isfinite(conditioned_features).all():\n                            raise RuntimeError(f"FiLM conditioning {model_name} mengandung NaN/Inf pada gap={gap_ms}, sample={index}.")\n                        conditioning_rows.append({\n                            "model": model_name,\n                            "gap_ms": int(gap_ms),\n                            "sample_index": int(index),\n                            "encoder_shape": tuple(encoder_latent.shape),\n                            "decoder_features_shape": tuple(decoder_features.shape),\n                            "conditioned_features_shape": tuple(conditioned_features.shape),\n                            "encoder_min": float(encoder_latent.float().min().item()),\n                            "encoder_max": float(encoder_latent.float().max().item()),\n                            "encoder_mean": float(encoder_latent.float().mean().item()),\n                            "encoder_std": float(encoder_latent.float().std(unbiased=False).item()),\n                            "conditioning_min": float(conditioned_features.float().min().item()),\n                            "conditioning_max": float(conditioned_features.float().max().item()),\n                            "conditioning_mean": float(conditioned_features.float().mean().item()),\n                            "conditioning_std": float(conditioned_features.float().std(unbiased=False).item()),\n                        })\n\n                        reconstructed = decoder.inpaint(\n                            masked_tensor,\n                            mask_tensor,\n                            conditioning=conditioned_features,\n                        )\n                    else:\n                        # MAID: diffusion handled internally by decoder.inpaint()\n                        decoder_features = decoder.get_features(masked_tensor, mask_tensor)\n                        conditioned_features = film_layer(encoder_latent.float(), decoder_features)\n                        if not torch.isfinite(decoder_features).all():\n                            raise RuntimeError(f"Decoder features {model_name} mengandung NaN/Inf pada gap={gap_ms}, sample={index}.")\n                        if not torch.isfinite(conditioned_features).all():\n                            raise RuntimeError(f"FiLM conditioning {model_name} mengandung NaN/Inf pada gap={gap_ms}, sample={index}.")\n                        conditioning_rows.append({\n                            "model": model_name,\n                            "gap_ms": int(gap_ms),\n                            "sample_index": int(index),\n                            "encoder_shape": tuple(encoder_latent.shape),\n                            "decoder_features_shape": tuple(decoder_features.shape),\n                            "conditioned_features_shape": tuple(conditioned_features.shape),\n                            "encoder_min": float(encoder_latent.float().min().item()),\n                            "encoder_max": float(encoder_latent.float().max().item()),\n                            "encoder_mean": float(encoder_latent.float().mean().item()),\n                            "encoder_std": float(encoder_latent.float().std(unbiased=False).item()),\n                            "conditioning_min": float(conditioned_features.float().min().item()),\n                            "conditioning_max": float(conditioned_features.float().max().item()),\n                            "conditioning_mean": float(conditioned_features.float().mean().item()),\n                            "conditioning_std": float(conditioned_features.float().std(unbiased=False).item()),\n                        })\n\n                        reconstructed = decoder.inpaint(\n                            masked_tensor,\n                            mask_tensor,\n                            conditioning=conditioned_features,\n                        )\n\n                    # Crossfade buat menghilangkan click artifacts\n                    reconstructed = crossfade_boundary(\n                        orig_audio, reconstructed, gap_start, gap_end,\n                    )\n                output_path = save_reconstructed_output(model_name, gap_ms, index, reconstructed, TARGET_SR)\n                generated_count += 1\n\n            reconstructed_list.append(reconstructed)\n            output_manifest_rows.append({\n                "model": model_name,\n                "gap_ms": gap_ms,\n                "sample_index": index,\n                "sr": TARGET_SR,\n                "n_samples": int(len(reconstructed)),\n                "duration_seconds": float(len(reconstructed) / TARGET_SR),\n                "gap_start": int(gap_region["gap_start"]),\n                "gap_end": int(gap_region["gap_end"]),\n                "gap_position": gap_region["gap_position"],\n                "reconstructed_path": output_path,\n            })\n\n        reconstructed_dict[gap_ms] = reconstructed_list\n\n    save_reconstruction_manifest(model_name, output_manifest_rows)\n    save_reconstruction_diagnostics(\n        model_name, original_audios, reconstructed_dict, alignment_rows,\n        conditioning_rows, gap_regions_by_gap=gap_regions_by_gap,\n    )\n    print(f"  Rekonstruksi reused/generated: {reused_count}/{generated_count}")\n    results_df = evaluate_all_gaps(original_audios, reconstructed_dict, TARGET_SR, gap_regions_by_gap)\n    record_evaluation_timing(model_name, time.perf_counter() - evaluation_start, n_eval_samples)\n    return results_df\n\n\ndef run_baseline_inpainting_evaluation(decoder, device, n_eval_samples: int = 50):\n    """\n    Evaluasi baseline (tanpa encoder, tanpa FiLM).\n    Decoder dipakai langsung tanpa conditioning.\n    """\n    baseline_eval_start = time.perf_counter()\n    model_name = "baseline_cqtdiff"\n    cache_tag = set_reconstruction_cache_context(model_name, decoder, checkpoint_path=None)\n    prepare_reconstructed_outputs(model_name, clear=EVAL_CLEAR_RECONSTRUCTIONS)\n    output_manifest_rows = []\n    alignment_rows = []\n    original_audios, masked_by_gap, gap_regions_by_gap = load_preprocessed_data(\n        n_eval_samples, gap_position=EVAL_GAP_POSITION\n    )\n    reconstructed_dict = {}\n    reused_count = 0\n    generated_count = 0\n\n    print("\\n🎵 Menjalankan baseline CQT-Diff+ (tanpa SSL encoder)...")\n    print(f"  Cache tag aktif: {cache_tag[:96]}...")\n    if EVAL_REUSE_RECONSTRUCTIONS:\n        print("  Resume rekonstruksi aktif: WAV yang sudah ada akan dipakai ulang.")\n        summarize_reconstruction_cache(model_name, n_eval_samples)\n    if EVAL_CLEAR_RECONSTRUCTIONS:\n        print("  EVAL_CLEAR_RECONSTRUCTIONS aktif: output lama dibersihkan sebelum eval.")\n\n    for gap_ms in GAP_DURATIONS_MS:\n        print(f"\\n  Gap {gap_ms}ms...")\n        reconstructed_list = []\n\n        for i, (orig_audio, masked_audio) in enumerate(zip(original_audios, masked_by_gap[gap_ms])):\n            gap_region = gap_regions_by_gap[gap_ms][i]\n            if (i + 1) % 10 == 0:\n                print(f"    Sample {i+1}/{n_eval_samples}")\n\n            reconstructed, output_path = load_reconstructed_output_if_available(\n                model_name, gap_ms, i, expected_n_samples=len(orig_audio), sr=TARGET_SR\n            )\n\n            if reconstructed is not None:\n                reused_count += 1\n            else:\n                with torch.inference_mode():\n                    masked_tensor = torch.from_numpy(masked_audio).float().unsqueeze(0).to(device)\n                    mask, gap_start, gap_end = build_gap_mask_array(\n                        len(masked_audio), gap_ms, gap_start=gap_region["gap_start"]\n                    )\n                    align = validate_masked_gap_alignment(\n                        orig_audio, masked_audio, mask, gap_ms,\n                        expected_gap_start=gap_start, expected_gap_end=gap_end,\n                    )\n                    align.update({\n                        "model": model_name,\n                        "gap_ms": int(gap_ms),\n                        "sample_index": int(i),\n                        "gap_position": gap_region["gap_position"],\n                    })\n                    alignment_rows.append(align)\n                    mask_tensor = torch.from_numpy(mask).unsqueeze(0).to(device)\n\n                    # Baseline: conditioning=None\n                    reconstructed = decoder.inpaint(\n                        masked_tensor, mask_tensor, conditioning=None,\n                    )\n                    reconstructed = crossfade_boundary(\n                        orig_audio, reconstructed, gap_start, gap_end,\n                    )\n                output_path = save_reconstructed_output(model_name, gap_ms, i, reconstructed, TARGET_SR)\n                generated_count += 1\n\n            reconstructed_list.append(reconstructed)\n            output_manifest_rows.append({\n                "model": model_name,\n                "gap_ms": gap_ms,\n                "sample_index": i,\n                "sr": TARGET_SR,\n                "n_samples": int(len(reconstructed)),\n                "duration_seconds": float(len(reconstructed) / TARGET_SR),\n                "gap_start": int(gap_region["gap_start"]),\n                "gap_end": int(gap_region["gap_end"]),\n                "gap_position": gap_region["gap_position"],\n                "reconstructed_path": output_path,\n            })\n\n        reconstructed_dict[gap_ms] = reconstructed_list\n\n    save_reconstruction_manifest(model_name, output_manifest_rows)\n    save_reconstruction_diagnostics(\n        model_name, original_audios, reconstructed_dict, alignment_rows, None,\n        gap_regions_by_gap=gap_regions_by_gap,\n    )\n    print(f"  Rekonstruksi reused/generated: {reused_count}/{generated_count}")\n    results_df = evaluate_all_gaps(original_audios, reconstructed_dict, TARGET_SR, gap_regions_by_gap)\n    record_evaluation_timing(model_name, time.perf_counter() - baseline_eval_start, n_eval_samples)\n    return results_df\n\n\nprint("✅ Shared hybrid training helpers berhasil didefinisikan!")\nprint("   Tersedia: builder encoder asli, adapter decoder asli, checkpoint helpers, trainer MAID,")\nprint("   run_hybrid_inpainting_evaluation, run_baseline_inpainting_evaluation")\n\n\ndef _parse_csv_arg(value, default_items=None):\n    if value is None or str(value).strip() == "":\n        return list(default_items or [])\n    text = str(value).strip()\n    if text.lower() == "all":\n        return list(default_items or [])\n    return [item.strip() for item in text.split(",") if item.strip()]\n\n\ndef _parse_run_selection():\n    import argparse\n\n    parser = argparse.ArgumentParser(add_help=False)\n    parser.add_argument("--phase", "--run-phase", dest="phase", default=os.environ.get("RUN_PHASE", "all"))\n    parser.add_argument("--models", "--run-models", dest="models", default=os.environ.get("RUN_MODELS", "all"))\n    parser.add_argument(\n        "--auto-stop",\n        action="store_true",\n        default=os.environ.get("AUTO_STOP_INSTANCE", "").strip().lower() in {"1", "true", "yes", "on"},\n        help="Stop/power off the instance after all selected phase/model targets finish successfully.",\n    )\n    parser.add_argument(\n        "--auto-stop-command",\n        default=os.environ.get("AUTO_STOP_COMMAND", "sudo shutdown -h now"),\n        help="Command used when --auto-stop is enabled. Default: sudo shutdown -h now",\n    )\n    args, _ = parser.parse_known_args()\n\n    requested_phases = set(_parse_csv_arg(args.phase, ["all"]))\n    if "all" in requested_phases:\n        phases = {"train", "eval", "summary"}\n    else:\n        phases = requested_phases\n\n    valid_phases = {"train", "eval", "summary"}\n    invalid_phases = sorted(phases - valid_phases)\n    if invalid_phases:\n        raise ValueError(f"RUN_PHASE/--phase tidak dikenali: {invalid_phases}. Pilihan: all, train, eval, summary.")\n\n    models = set(_parse_csv_arg(args.models, EXPECTED_MODEL_CONFIGS))\n    invalid_models = sorted(models - set(EXPECTED_MODEL_CONFIGS))\n    if invalid_models:\n        raise ValueError(f"RUN_MODELS/--models tidak dikenali: {invalid_models}. Pilihan: {EXPECTED_MODEL_CONFIGS}")\n\n    return phases, models, bool(args.auto_stop), str(args.auto_stop_command).strip()\n\n\nRUN_PHASES, RUN_MODEL_SELECTION, AUTO_STOP_INSTANCE, AUTO_STOP_COMMAND = _parse_run_selection()\n\n\ndef should_run_train(model_name):\n    return "train" in RUN_PHASES and model_name in RUN_MODEL_SELECTION\n\n\ndef should_run_eval(model_name):\n    return "eval" in RUN_PHASES and model_name in RUN_MODEL_SELECTION\n\n\ndef should_run_summary():\n    return "summary" in RUN_PHASES\n\n\nprint("\\nRun selection:")\nprint(f"   phases: {sorted(RUN_PHASES)}")\nprint(f"   models: {sorted(RUN_MODEL_SELECTION)}")\nprint(f"   auto_stop: {AUTO_STOP_INSTANCE}")\nif "train" in RUN_PHASES and "baseline_cqtdiff" in RUN_MODEL_SELECTION:\n    print(\n        "   note: baseline_cqtdiff adalah pretrained-only; phase train akan dilewati. "\n        "Gunakan baseline_cqtdiff_finetuned untuk baseline no-SSL yang di-train."\n    )\n\n\ndef validate_model_artifact_isolation(model_names=None):\n    """Fail early if two model configs would write to the same model-specific artifact."""\n    model_names = list(model_names or EXPECTED_MODEL_CONFIGS)\n    artifact_groups = {\n        "result_csv": {\n            model: os.path.join(PATHS["results"], f"{evaluation_artifact_name(model)}_results.csv")\n            for model in model_names\n        },\n        "output_dir": {\n            model: os.path.join(PATHS["outputs"], evaluation_artifact_name(model))\n            for model in model_names\n        },\n        "checkpoint_dir": {\n            model: get_model_checkpoint_dir(model)\n            for model in model_names\n        },\n        "checkpoint_best": {\n            model: get_model_checkpoint_path(model)\n            for model in model_names\n        },\n        "checkpoint_latest": {\n            model: os.path.join(get_model_checkpoint_dir(model), f"{model}_latest.pt")\n            for model in model_names\n        },\n        "training_history_csv": {\n            model: os.path.join(PATHS["logs"], f"{model}_training_history.csv")\n            for model in model_names\n        },\n        "training_history_png": {\n            model: os.path.join(PATHS["plots"], f"{model}_training_history.png")\n            for model in model_names\n        },\n        "encoder_cache_train": {\n            model: _encoder_cache_path(model, "train", 12345)\n            for model in model_names\n        },\n        "encoder_cache_val": {\n            model: _encoder_cache_path(model, "val", 12345)\n            for model in model_names\n        },\n    }\n\n    collisions = []\n    for group_name, paths_by_model in artifact_groups.items():\n        seen = {}\n        for model, path in paths_by_model.items():\n            norm_path = os.path.normcase(os.path.abspath(path))\n            if norm_path in seen:\n                collisions.append((group_name, seen[norm_path], model, path))\n            else:\n                seen[norm_path] = model\n\n    if collisions:\n        detail = "\\n".join(\n            f"  - {group}: {left} dan {right} -> {path}"\n            for group, left, right, path in collisions\n        )\n        raise RuntimeError(\n            "Artifact collision terdeteksi antar konfigurasi model. "\n            "Run dihentikan supaya output/checkpoint tidak tertimpa:\\n" + detail\n        )\n\n    pretrained = "baseline_cqtdiff"\n    finetuned = "baseline_cqtdiff_finetuned"\n    print("Artifact isolation OK:")\n    print(f"   pretrained baseline output : {artifact_groups[\'output_dir\'][pretrained]}")\n    print(f"   finetuned baseline output  : {artifact_groups[\'output_dir\'][finetuned]}")\n    print(f"   finetuned checkpoint       : {artifact_groups[\'checkpoint_best\'][finetuned]}")\n\n\nvalidate_model_artifact_isolation()\n\n\ndef _summary_artifact_exists():\n    expected = [\n        os.path.join(PATHS["results"], "experiment_summary.json"),\n        os.path.join(PATHS["results"], "experiment_summary.csv"),\n    ]\n    return any(os.path.exists(path) for path in expected)\n\n\ndef _missing_selected_run_targets():\n    missing = []\n\n    if "train" in RUN_PHASES:\n        for model_name in sorted(RUN_MODEL_SELECTION):\n            # Baseline pakai pretrained weights, gak perlu checkpoint training\n            if model_name == "baseline_cqtdiff":\n                continue\n            ckpt_path = get_model_checkpoint_path(model_name)\n            if not os.path.exists(ckpt_path):\n                missing.append(f"train:{model_name} -> {ckpt_path}")\n\n    if "eval" in RUN_PHASES:\n        for model_name in sorted(RUN_MODEL_SELECTION):\n            result_path = os.path.join(PATHS["results"], f"{evaluation_artifact_name(model_name)}_results.csv")\n            if not os.path.exists(result_path):\n                missing.append(f"eval:{model_name} -> {result_path}")\n\n    if "summary" in RUN_PHASES and not _summary_artifact_exists():\n        missing.append(f"summary -> {os.path.join(PATHS[\'results\'], \'experiment_summary.json\')}")\n\n    return missing\n\n\ndef maybe_auto_stop_instance():\n    if not AUTO_STOP_INSTANCE:\n        return\n\n    missing = _missing_selected_run_targets()\n    if missing:\n        print("\\nAuto-stop diminta, tapi target run yang dipilih belum lengkap. Instance tidak dimatikan.")\n        for item in missing:\n            print(f"  - missing {item}")\n        return\n\n    if not AUTO_STOP_COMMAND:\n        print("\\nAuto-stop diminta, tapi AUTO_STOP_COMMAND kosong. Instance tidak dimatikan.")\n        return\n\n    print(f"\\nAuto-stop: semua target phase/model terpilih selesai. Menjalankan: {AUTO_STOP_COMMAND}")\n    try:\n        import shlex\n        import subprocess\n        subprocess.Popen(shlex.split(AUTO_STOP_COMMAND))\n    except Exception as exc:\n        print(f"Auto-stop gagal dijalankan: {exc}")\n\n\n# ---\n# ## CELL 7 — BASELINE: CQT-Diff+ Standalone (Pretrained)\n#\n# Baseline pretrained mengukur kemampuan checkpoint CQT-Diff+ asli tanpa\n# adaptation ke MusicNet dan tanpa SSL. Klaim kontribusi SSL tidak boleh\n# hanya dibandingkan ke baseline ini; gunakan Cell 7B\n# `baseline_cqtdiff_finetuned` sebagai pembanding fair no-SSL.\n\n\n# ============================================================\n# CELL 7: BASELINE — CQT-Diff+ STANDALONE (PRETRAINED DIFFUSION)\n# ============================================================\n# Baseline = CQT-Diff+ original pakai pretrained weights + proper\n# multi-step reverse diffusion sampling buat inpainting.\n#\n# TIDAK PERLU TRAINING TAMBAHAN karena:\n# - CQT-Diff+ sudah di-pretrain pada diffusion denoising objective\n# - Inpainting dilakukan via multi-step reverse diffusion (T=35 steps)\n#   dengan data consistency (replacement method) — sesuai paper asli\n# - Baseline menunjukkan kemampuan murni CQT-Diff+ tanpa SSL conditioning\n# - Hybrid models (Cell 8-11) menambahkan SSL encoder + FiLM di atas ini\n#\n# Untuk fairness thesis:\n# - baseline_cqtdiff: pretrained, tanpa fine-tuning, tanpa SSL\n# - baseline_cqtdiff_finetuned: fine-tuned adapter no-SSL\n# - clap/audiomae_cqtdiff: fine-tuned adapter dengan SSL-guided residual conditioning\n# ============================================================\n\nimport torch\nimport numpy as np\nimport os\n\nMODEL_NAME = "baseline_cqtdiff"\nFORCE_REEVAL = True\n\nprint(f"Config stage: {PIPELINE_STAGE_NAME} | model: {MODEL_NAME} | mode: pretrained diffusion (no training)")\n\n# Hapus hasil lama kalau FORCE_REEVAL aktif\nif should_run_eval(MODEL_NAME) and FORCE_REEVAL:\n    old_result = os.path.join(PATHS["results"], f"{evaluation_artifact_name(MODEL_NAME)}_results.csv")\n    if os.path.exists(old_result):\n        os.remove(old_result)\n        print(f"Hasil lama dihapus: {old_result}")\n\nif not should_run_eval(MODEL_NAME):\n    print(f"Skip {MODEL_NAME}: tidak dipilih oleh RUN_PHASE/RUN_MODELS.")\nelse:\n    device = torch.device("cuda")\n\n    if should_run_eval(MODEL_NAME):\n        if check_if_done(MODEL_NAME):\n            print(f"Baseline {MODEL_NAME} sudah selesai. Lewati evaluasi.")\n        else:\n            # ============================================================\n            # EVALUASI BASELINE (pretrained CQT-Diff+ diffusion sampling)\n            # ============================================================\n            print("\\nLoading pretrained CQT-Diff+ untuk baseline evaluasi...")\n            print("   Inpainting via multi-step reverse diffusion (bukan single-pass).")\n            print("   Tidak perlu training tambahan — pretrained weights sudah cukup.\\n")\n\n            baseline_model = build_hybrid_cqtdiff_decoder(device)\n            baseline_model.eval()\n            print_gpu_usage("Setelah load baseline")\n\n            print("\\nMengevaluasi baseline...")\n            results_df = run_baseline_inpainting_evaluation(baseline_model, device, n_eval_samples=N_EVAL_SAMPLES)\n\n            print(f"\\nHasil evaluasi BASELINE (CQT-Diff+ pretrained diffusion sampling):")\n            print(results_df.to_string(index=False))\n\n            save_results(results_df, MODEL_NAME)\n\n            print("\\nMembersihkan memori GPU...")\n            clear_gpu_memory(baseline_model)\n\n            print(f"\\nBASELINE {MODEL_NAME} selesai!")\n    else:\n        print(f"Skip evaluasi {MODEL_NAME}: RUN_PHASE tidak memuat eval atau model tidak dipilih.")\n\n\n# ---\n# ## CELL 7B — BASELINE FINE-TUNED: Conditioning Head tanpa SSL\n#\n# Ablation baseline: conditioning head yang sama (FiLM + spec_decoder +\n# condition_gate + sigma_scale_net) di-train pada MusicNet, tapi\n# TANPA SSL encoder (input encoder selalu zeros).\n#\n# Tujuan: memisahkan efek domain adaptation (fine-tuning pada MusicNet)\n# dari kontribusi spesifik SSL encoder (CLAP/AudioMAE).\n#\n# Perbandingan yang fair:\n# - baseline_cqtdiff:           pretrained, tanpa fine-tuning, tanpa SSL\n# - baseline_cqtdiff_finetuned: fine-tuned conditioning head, tanpa SSL\n# - clap_cqtdiff:               fine-tuned conditioning head, DENGAN SSL\n#\n# Jika clap_cqtdiff > baseline_cqtdiff_finetuned → SSL memberikan\n# kontribusi nyata di atas fine-tuning.\n\n\n# ============================================================\n# CELL 7B-A: TRAINING — BASELINE FINE-TUNED (tanpa SSL encoder)\n# ============================================================\n\nMODEL_NAME = "baseline_cqtdiff_finetuned"\nFORCE_RETRAIN = True\nBATCH_SIZE = 32\nNUM_WORKERS = AUTO_NUM_WORKERS\nNUM_EPOCHS = 30\nLEARNING_RATE = 1e-4\n\nprint(f"Config stage: {PIPELINE_STAGE_NAME} | model: {MODEL_NAME} | "\n      f"dataset_fraction={DATASET_FRACTION:.0%} | batch_size={BATCH_SIZE} | epochs={NUM_EPOCHS}")\n\nassert NUM_EPOCHS >= 5, "NUM_EPOCHS minimal 5 agar checkpoint best tervalidasi bisa tersimpan."\n\nif should_run_train(MODEL_NAME) and FORCE_RETRAIN:\n    reset_training_checkpoints_if_requested(MODEL_NAME, FORCE_RETRAIN)\n\nckpt_path = get_model_checkpoint_path(MODEL_NAME)\nif not should_run_train(MODEL_NAME):\n    print(f"Skip training {MODEL_NAME}: RUN_PHASE/RUN_MODELS tidak memilih blok ini.")\nelif hybrid_checkpoint_exists(MODEL_NAME) and not FORCE_RETRAIN:\n    print(f"Checkpoint sudah ada. Skip training: {ckpt_path}")\nelse:\n    device = torch.device("cuda")\n    cqtdiff_model = None\n    film_layer = None\n\n    try:\n        print(f"Device: {device}")\n        check_batch_size_memory(BATCH_SIZE, min_expected_vram_gb=8.0)\n\n        loaders = make_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)\n\n        _ft_encoder_dim = FILM_CONFIGS[MODEL_NAME]["encoder_dim"]\n\n        def _zero_encoder_fn(audio_input):\n            """Dummy encoder: selalu return zeros (tanpa SSL)."""\n            if isinstance(audio_input, np.ndarray):\n                return torch.zeros(1, _ft_encoder_dim, device=device)\n            if isinstance(audio_input, torch.Tensor):\n                return torch.zeros(audio_input.shape[0], _ft_encoder_dim, device=device)\n            return torch.zeros(1, _ft_encoder_dim, device=device)\n\n        loaders = add_encoder_cache_to_loaders(\n            loaders, _zero_encoder_fn, device, MODEL_NAME, num_workers=NUM_WORKERS\n        )\n        cqtdiff_model = build_hybrid_cqtdiff_decoder(device)\n        film_layer = build_film_layer(MODEL_NAME, device)\n\n        print_gpu_usage("Sebelum training")\n        train_model(\n            cqtdiff_model,\n            _zero_encoder_fn,\n            film_layer,\n            loaders["train"],\n            val_loader=loaders["val"],\n            num_epochs=NUM_EPOCHS,\n            lr=LEARNING_RATE,\n            device=device,\n            checkpoint_dir=get_model_checkpoint_dir(MODEL_NAME),\n            model_name=MODEL_NAME,\n            batch_size=BATCH_SIZE,\n            dataset_fraction=DATASET_FRACTION,\n        )\n\n        if not hybrid_checkpoint_exists(MODEL_NAME):\n            raise RuntimeError(f"Checkpoint {MODEL_NAME} tidak tersimpan.")\n\n        print(f"Training selesai. Checkpoint siap dipakai: {ckpt_path}")\n    finally:\n        clear_gpu_memory(cqtdiff_model, film_layer)\n\n\n# ============================================================\n# CELL 7B: EVALUASI — BASELINE FINE-TUNED (tanpa SSL encoder)\n# ============================================================\n\nMODEL_NAME = "baseline_cqtdiff_finetuned"\nFORCE_REEVAL = True\n\nif should_run_eval(MODEL_NAME) and FORCE_REEVAL:\n    old_result = os.path.join(PATHS["results"], f"{evaluation_artifact_name(MODEL_NAME)}_results.csv")\n    if os.path.exists(old_result):\n        os.remove(old_result)\n        print(f"Hasil lama dihapus: {old_result}")\n\nif not should_run_eval(MODEL_NAME):\n    print(f"Skip evaluasi {MODEL_NAME}: RUN_PHASE/RUN_MODELS tidak memilih blok ini.")\nelif check_if_done(MODEL_NAME):\n    print(f"Model {MODEL_NAME} sudah selesai. Lewati cell ini.")\nelse:\n    ckpt_path = get_model_checkpoint_path(MODEL_NAME)\n    if not hybrid_checkpoint_exists(MODEL_NAME):\n        raise FileNotFoundError(\n            f"Checkpoint {MODEL_NAME} belum ditemukan di {ckpt_path}. "\n            "Jalankan CELL 7B-A terlebih dahulu."\n        )\n\n    device = torch.device("cuda")\n    cqtdiff_model = None\n    film_layer = None\n\n    try:\n        print_gpu_usage("Awal")\n        print(f"Menggunakan checkpoint: {ckpt_path}")\n\n        _ft_encoder_dim = FILM_CONFIGS[MODEL_NAME]["encoder_dim"]\n\n        def _zero_encoder_fn(audio_input):\n            """Dummy encoder: selalu return zeros (tanpa SSL)."""\n            if isinstance(audio_input, np.ndarray):\n                return torch.zeros(1, _ft_encoder_dim, device=device)\n            if isinstance(audio_input, torch.Tensor):\n                return torch.zeros(audio_input.shape[0], _ft_encoder_dim, device=device)\n            return torch.zeros(1, _ft_encoder_dim, device=device)\n\n        cqtdiff_model = build_hybrid_cqtdiff_decoder(device)\n        film_layer = build_film_layer(MODEL_NAME, device)\n        load_hybrid_checkpoint(MODEL_NAME, cqtdiff_model, film_layer, device)\n        print_gpu_usage("Setelah load model terlatih")\n\n        results_df = run_hybrid_inpainting_evaluation(\n            "Baseline Fine-tuned (no SSL)",\n            _zero_encoder_fn,\n            cqtdiff_model,\n            film_layer,\n            device,\n            n_eval_samples=N_EVAL_SAMPLES,\n            model_name=MODEL_NAME,\n        )\n\n        print(f"\\nHasil {MODEL_NAME}:")\n        print(results_df.to_string(index=False))\n        save_results(results_df, MODEL_NAME)\n    finally:\n        clear_gpu_memory(cqtdiff_model, film_layer)\n\n    print(f"\\nBASELINE FINE-TUNED {MODEL_NAME} selesai!")\n\n\n# ---\n# ## CELL 8 — Kombinasi 1: CLAP + CQT-Diff+\n#\n# Model hybrid pertama. Dibandingkan dengan baseline di Cell 7,\n# perbedaannya hanya penambahan **CLAP encoder + FiLM conditioning**.\n\n\n# ============================================================\n# CELL 8A: TRAINING — CLAP + CQT-Diff+\n# ============================================================\n# Default: skip training jika checkpoint sudah ada.\n# Set FORCE_RETRAIN = True untuk melatih ulang.\n# ============================================================\n\n# ============================================================\n# CELL 8A: TRAINING — CLAP + CQT-Diff+\n# ============================================================\n# Set FORCE_RETRAIN = True untuk melatih ulang.\n# PENTING: Set True setelah update arsitektur model!\n# ============================================================\n\nMODEL_NAME = "clap_cqtdiff"\nFORCE_RETRAIN = True   # <-- True karena arsitektur model berubah!\n# Stage override: instance memory/throughput test uses batch size 8.\nBATCH_SIZE = 32\nNUM_WORKERS = AUTO_NUM_WORKERS\n# Stage override: instance test keeps the current 10 training epochs.\nNUM_EPOCHS = 30\nLEARNING_RATE = 1e-4\n\nprint(f"Config stage: {PIPELINE_STAGE_NAME} | dataset_fraction={DATASET_FRACTION:.0%} | batch_size={BATCH_SIZE} | epochs={NUM_EPOCHS}")\n\nassert NUM_EPOCHS >= 5, "NUM_EPOCHS minimal 5 agar checkpoint best tervalidasi bisa tersimpan."\n\nif should_run_train(MODEL_NAME) and FORCE_RETRAIN:\n    reset_training_checkpoints_if_requested(MODEL_NAME, FORCE_RETRAIN)\n\nckpt_path = get_model_checkpoint_path(MODEL_NAME)\nif not should_run_train(MODEL_NAME):\n    print(f"⏭️ Skip training {MODEL_NAME}: RUN_PHASE/RUN_MODELS tidak memilih blok ini.")\nelif hybrid_checkpoint_exists(MODEL_NAME) and not FORCE_RETRAIN:\n    print(f"✅ Checkpoint sudah ada. Skip training: {ckpt_path}")\nelse:\n    device = torch.device("cuda")\n    clap_model = None\n    cqtdiff_model = None\n    film_layer = None\n\n    try:\n        print(f"🔧 Device: {device}")\n        check_batch_size_memory(BATCH_SIZE, min_expected_vram_gb=8.0)\n\n        loaders = make_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)\n        clap_model, encoder_fn = build_clap_encoder(device)\n        loaders = add_encoder_cache_to_loaders(loaders, encoder_fn, device, MODEL_NAME, num_workers=NUM_WORKERS)\n        cqtdiff_model = build_hybrid_cqtdiff_decoder(device)\n        film_layer = build_film_layer(MODEL_NAME, device)\n\n        print_gpu_usage("Sebelum training")\n        train_model(\n            cqtdiff_model,\n            encoder_fn,\n            film_layer,\n            loaders["train"],\n            val_loader=loaders["val"],\n            num_epochs=NUM_EPOCHS,\n            lr=LEARNING_RATE,\n            device=device,\n            checkpoint_dir=get_model_checkpoint_dir(MODEL_NAME),\n            model_name=MODEL_NAME,\n            batch_size=BATCH_SIZE,\n            dataset_fraction=DATASET_FRACTION,\n        )\n\n        if not hybrid_checkpoint_exists(MODEL_NAME):\n            raise RuntimeError(f"Checkpoint {MODEL_NAME} tidak tersimpan.")\n\n        print(f"✅ Training selesai. Checkpoint siap dipakai: {ckpt_path}")\n    finally:\n        clear_gpu_memory(clap_model, cqtdiff_model, film_layer)\n\n\n# ============================================================\n# CELL 8: KOMBINASI 1 — CLAP + CQT-Diff+\n# ============================================================\n# Evaluasi hybrid selalu memakai checkpoint terlatih.\n# Jika checkpoint belum ada, jalankan CELL 8A terlebih dahulu.\n# ============================================================\n\nMODEL_NAME = "clap_cqtdiff"\nFORCE_REEVAL = True  # <-- True buat re-evaluasi setelah update arsitektur\n\nif should_run_eval(MODEL_NAME) and FORCE_REEVAL:\n    old_result = os.path.join(PATHS["results"], f"{evaluation_artifact_name(MODEL_NAME)}_results.csv")\n    if os.path.exists(old_result):\n        os.remove(old_result)\n\nif not should_run_eval(MODEL_NAME):\n    print(f"⏭️ Skip evaluasi {MODEL_NAME}: RUN_PHASE/RUN_MODELS tidak memilih blok ini.")\nelif check_if_done(MODEL_NAME):\n    print(f"Model {MODEL_NAME} sudah selesai. Lewati cell ini.")\nelse:\n    ckpt_path = get_model_checkpoint_path(MODEL_NAME)\n    if not hybrid_checkpoint_exists(MODEL_NAME):\n        raise FileNotFoundError(\n            f"Checkpoint {MODEL_NAME} belum ditemukan di {ckpt_path}. Jalankan CELL 8A terlebih dahulu."\n        )\n\n    device = torch.device("cuda")\n    clap_model = None\n    cqtdiff_model = None\n    film_layer = None\n\n    try:\n        print_gpu_usage("Awal")\n        print(f"📦 Menggunakan checkpoint: {ckpt_path}")\n\n        clap_model, encoder_fn = build_clap_encoder(device)\n        cqtdiff_model = build_hybrid_cqtdiff_decoder(device)\n        film_layer = build_film_layer(MODEL_NAME, device)\n        load_hybrid_checkpoint(MODEL_NAME, cqtdiff_model, film_layer, device)\n        print_gpu_usage("Setelah load model terlatih")\n\n        results_df = run_hybrid_inpainting_evaluation(\n            "CLAP + CQT-Diff+",\n            encoder_fn,\n            cqtdiff_model,\n            film_layer,\n            device,\n            n_eval_samples=N_EVAL_SAMPLES,\n            model_name=MODEL_NAME,\n        )\n\n        print(f"\\n📋 Hasil {MODEL_NAME}:")\n        print(results_df.to_string(index=False))\n        save_results(results_df, MODEL_NAME)\n    finally:\n        clear_gpu_memory(clap_model, cqtdiff_model, film_layer)\n\n    print(f"\\n✅ {MODEL_NAME} selesai!")\n\n# ---\n# ## CELL 9 — Kombinasi 2: CLAP + MAID\n\n\n# ============================================================\n# CELL 9A: TRAINING — CLAP + MAID\n# ============================================================\n# Default: skip training jika checkpoint sudah ada.\n# Set FORCE_RETRAIN = True untuk melatih ulang.\n# ============================================================\n\nMODEL_NAME = "clap_maid"\nFORCE_RETRAIN = True   # <-- True karena arsitektur/training berubah!\n# Stage override: instance memory/throughput test uses batch size 8.\nBATCH_SIZE = 32\nNUM_WORKERS = AUTO_NUM_WORKERS\n# Stage override: instance test keeps the current 10 training epochs.\nNUM_EPOCHS = 30\nLEARNING_RATE = 1e-4\n\nprint(f"Config stage: {PIPELINE_STAGE_NAME} | dataset_fraction={DATASET_FRACTION:.0%} | batch_size={BATCH_SIZE} | epochs={NUM_EPOCHS}")\n\nassert NUM_EPOCHS >= 5, "NUM_EPOCHS minimal 5 agar checkpoint best tervalidasi bisa tersimpan."\n\nif should_run_train(MODEL_NAME) and FORCE_RETRAIN:\n    reset_training_checkpoints_if_requested(MODEL_NAME, FORCE_RETRAIN)\n\nckpt_path = get_model_checkpoint_path(MODEL_NAME)\nif not should_run_train(MODEL_NAME):\n    print(f"⏭️ Skip training {MODEL_NAME}: RUN_PHASE/RUN_MODELS tidak memilih blok ini.")\nelif hybrid_checkpoint_exists(MODEL_NAME) and not FORCE_RETRAIN:\n    print(f"✅ Checkpoint sudah ada. Skip training: {ckpt_path}")\nelse:\n    device = torch.device("cuda")\n    clap_model = None\n    maid_model = None\n    film_layer = None\n\n    try:\n        print(f"🔧 Device: {device}")\n        check_batch_size_memory(BATCH_SIZE, min_expected_vram_gb=8.0)\n\n        loaders = make_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)\n        clap_model, encoder_fn = build_clap_encoder(device)\n        loaders = add_encoder_cache_to_loaders(loaders, encoder_fn, device, MODEL_NAME, num_workers=NUM_WORKERS)\n        maid_model = build_maid_decoder(device)\n        film_layer = build_film_layer(MODEL_NAME, device)\n\n        print_gpu_usage("Sebelum training")\n        train_maid_model(\n            maid_model,\n            encoder_fn,\n            film_layer,\n            loaders["train"],\n            val_loader=loaders["val"],\n            num_epochs=NUM_EPOCHS,\n            lr=LEARNING_RATE,\n            device=device,\n            checkpoint_dir=get_model_checkpoint_dir(MODEL_NAME),\n            model_name=MODEL_NAME,\n            batch_size=BATCH_SIZE,\n            dataset_fraction=DATASET_FRACTION,\n        )\n\n        if not hybrid_checkpoint_exists(MODEL_NAME):\n            raise RuntimeError(f"Checkpoint {MODEL_NAME} tidak tersimpan.")\n\n        print(f"✅ Training selesai. Checkpoint siap dipakai: {ckpt_path}")\n    finally:\n        clear_gpu_memory(clap_model, maid_model, film_layer)\n\n\n# ============================================================\n# CELL 9: KOMBINASI 2 — CLAP + MAID\n# ============================================================\n# Evaluasi hybrid selalu memakai checkpoint terlatih.\n# Jika checkpoint belum ada, jalankan CELL 9A terlebih dahulu.\n# ============================================================\n\nMODEL_NAME = "clap_maid"\nFORCE_REEVAL = True  # <-- True buat re-evaluasi setelah update\n\nif should_run_eval(MODEL_NAME) and FORCE_REEVAL:\n    old_result = os.path.join(PATHS["results"], f"{evaluation_artifact_name(MODEL_NAME)}_results.csv")\n    if os.path.exists(old_result):\n        os.remove(old_result)\n\nif not should_run_eval(MODEL_NAME):\n    print(f"⏭️ Skip evaluasi {MODEL_NAME}: RUN_PHASE/RUN_MODELS tidak memilih blok ini.")\nelif check_if_done(MODEL_NAME):\n    print(f"Model {MODEL_NAME} sudah selesai. Lewati cell ini.")\nelse:\n    ckpt_path = get_model_checkpoint_path(MODEL_NAME)\n    if not hybrid_checkpoint_exists(MODEL_NAME):\n        raise FileNotFoundError(\n            f"Checkpoint {MODEL_NAME} belum ditemukan di {ckpt_path}. Jalankan CELL 9A terlebih dahulu."\n        )\n\n    device = torch.device("cuda")\n    clap_model = None\n    maid_model = None\n    film_layer = None\n\n    try:\n        print_gpu_usage("Awal")\n        print(f"📦 Menggunakan checkpoint: {ckpt_path}")\n\n        clap_model, encoder_fn = build_clap_encoder(device)\n        maid_model = build_maid_decoder(device)\n        film_layer = build_film_layer(MODEL_NAME, device)\n        load_hybrid_checkpoint(MODEL_NAME, maid_model, film_layer, device)\n        print_gpu_usage("Setelah load model terlatih")\n\n        results_df = run_hybrid_inpainting_evaluation(\n            "CLAP + MAID",\n            encoder_fn,\n            maid_model,\n            film_layer,\n            device,\n            n_eval_samples=N_EVAL_SAMPLES,\n            model_name=MODEL_NAME,\n        )\n\n        print(f"\\n📋 Hasil {MODEL_NAME}:")\n        print(results_df.to_string(index=False))\n        save_results(results_df, MODEL_NAME)\n    finally:\n        clear_gpu_memory(clap_model, maid_model, film_layer)\n\n    print(f"\\n✅ {MODEL_NAME} selesai!")\n\n# ---\n# ## CELL 10 — Kombinasi 3: AudioMAE + CQT-Diff+\n\n\n# ============================================================\n# CELL 10A: TRAINING — AudioMAE + CQT-Diff+\n# ============================================================\n# Default: skip training jika checkpoint sudah ada.\n# Set FORCE_RETRAIN = True untuk melatih ulang.\n# ============================================================\n\nMODEL_NAME = "audiomae_cqtdiff"\nFORCE_RETRAIN = True   # <-- True karena arsitektur model berubah!\n# Stage override: instance memory/throughput test uses batch size 8.\nBATCH_SIZE = 32\nNUM_WORKERS = AUTO_NUM_WORKERS\n# Stage override: instance test keeps the current 10 training epochs.\nNUM_EPOCHS = 30\nLEARNING_RATE = 1e-4\n\nprint(f"Config stage: {PIPELINE_STAGE_NAME} | dataset_fraction={DATASET_FRACTION:.0%} | batch_size={BATCH_SIZE} | epochs={NUM_EPOCHS}")\n\nassert NUM_EPOCHS >= 5, "NUM_EPOCHS minimal 5 agar checkpoint best tervalidasi bisa tersimpan."\n\nif should_run_train(MODEL_NAME) and FORCE_RETRAIN:\n    reset_training_checkpoints_if_requested(MODEL_NAME, FORCE_RETRAIN)\n\nckpt_path = get_model_checkpoint_path(MODEL_NAME)\nif not should_run_train(MODEL_NAME):\n    print(f"⏭️ Skip training {MODEL_NAME}: RUN_PHASE/RUN_MODELS tidak memilih blok ini.")\nelif hybrid_checkpoint_exists(MODEL_NAME) and not FORCE_RETRAIN:\n    print(f"✅ Checkpoint sudah ada. Skip training: {ckpt_path}")\nelse:\n    device = torch.device("cuda")\n    audiomae_model = None\n    cqtdiff_model = None\n    film_layer = None\n\n    try:\n        print(f"🔧 Device: {device}")\n        check_batch_size_memory(BATCH_SIZE, min_expected_vram_gb=8.0)\n\n        loaders = make_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)\n        audiomae_model, encoder_fn = build_audiomae_encoder(device)\n        loaders = add_encoder_cache_to_loaders(loaders, encoder_fn, device, MODEL_NAME, num_workers=NUM_WORKERS)\n        cqtdiff_model = build_hybrid_cqtdiff_decoder(device)\n        film_layer = build_film_layer(MODEL_NAME, device)\n\n        print_gpu_usage("Sebelum training")\n        train_model(\n            cqtdiff_model,\n            encoder_fn,\n            film_layer,\n            loaders["train"],\n            val_loader=loaders["val"],\n            num_epochs=NUM_EPOCHS,\n            lr=LEARNING_RATE,\n            device=device,\n            checkpoint_dir=get_model_checkpoint_dir(MODEL_NAME),\n            model_name=MODEL_NAME,\n            batch_size=BATCH_SIZE,\n            dataset_fraction=DATASET_FRACTION,\n        )\n\n        if not hybrid_checkpoint_exists(MODEL_NAME):\n            raise RuntimeError(f"Checkpoint {MODEL_NAME} tidak tersimpan.")\n\n        print(f"✅ Training selesai. Checkpoint siap dipakai: {ckpt_path}")\n    finally:\n        clear_gpu_memory(audiomae_model, cqtdiff_model, film_layer)\n\n\n# ============================================================\n# CELL 10: KOMBINASI 3 — AudioMAE + CQT-Diff+\n# ============================================================\n# Evaluasi hybrid selalu memakai checkpoint terlatih.\n# Jika checkpoint belum ada, jalankan CELL 10A terlebih dahulu.\n# ============================================================\n\nMODEL_NAME = "audiomae_cqtdiff"\nFORCE_REEVAL = True  # <-- True buat re-evaluasi setelah update arsitektur\n\nif should_run_eval(MODEL_NAME) and FORCE_REEVAL:\n    old_result = os.path.join(PATHS["results"], f"{evaluation_artifact_name(MODEL_NAME)}_results.csv")\n    if os.path.exists(old_result):\n        os.remove(old_result)\n\nif not should_run_eval(MODEL_NAME):\n    print(f"⏭️ Skip evaluasi {MODEL_NAME}: RUN_PHASE/RUN_MODELS tidak memilih blok ini.")\nelif check_if_done(MODEL_NAME):\n    print(f"Model {MODEL_NAME} sudah selesai. Lewati cell ini.")\nelse:\n    ckpt_path = get_model_checkpoint_path(MODEL_NAME)\n    if not hybrid_checkpoint_exists(MODEL_NAME):\n        raise FileNotFoundError(\n            f"Checkpoint {MODEL_NAME} belum ditemukan di {ckpt_path}. Jalankan CELL 10A terlebih dahulu."\n        )\n\n    device = torch.device("cuda")\n    audiomae_model = None\n    cqtdiff_model = None\n    film_layer = None\n\n    try:\n        print_gpu_usage("Awal")\n        print(f"📦 Menggunakan checkpoint: {ckpt_path}")\n\n        audiomae_model, encoder_fn = build_audiomae_encoder(device)\n        cqtdiff_model = build_hybrid_cqtdiff_decoder(device)\n        film_layer = build_film_layer(MODEL_NAME, device)\n        load_hybrid_checkpoint(MODEL_NAME, cqtdiff_model, film_layer, device)\n        print_gpu_usage("Setelah load model terlatih")\n\n        results_df = run_hybrid_inpainting_evaluation(\n            "AudioMAE + CQT-Diff+",\n            encoder_fn,\n            cqtdiff_model,\n            film_layer,\n            device,\n            n_eval_samples=N_EVAL_SAMPLES,\n            model_name=MODEL_NAME,\n        )\n\n        print(f"\\n📋 Hasil {MODEL_NAME}:")\n        print(results_df.to_string(index=False))\n        save_results(results_df, MODEL_NAME)\n    finally:\n        clear_gpu_memory(audiomae_model, cqtdiff_model, film_layer)\n\n    print(f"\\n✅ {MODEL_NAME} selesai!")\n\n# ---\n# ## CELL 11 — Kombinasi 4: AudioMAE + MAID\n\n\n# ============================================================\n# CELL 11A: TRAINING — AudioMAE + MAID\n# ============================================================\n# Default: skip training jika checkpoint sudah ada.\n# Set FORCE_RETRAIN = True untuk melatih ulang.\n# ============================================================\n\nMODEL_NAME = "audiomae_maid"\nFORCE_RETRAIN = True   # <-- True karena arsitektur/training berubah!\n# Stage override: instance memory/throughput test uses batch size 8.\nBATCH_SIZE = 32\nNUM_WORKERS = AUTO_NUM_WORKERS\nNUM_EPOCHS = 30\nLEARNING_RATE = 1e-4\n\nprint(f"Config stage: {PIPELINE_STAGE_NAME} | dataset_fraction={DATASET_FRACTION:.0%} | batch_size={BATCH_SIZE} | epochs={NUM_EPOCHS}")\n\nassert NUM_EPOCHS >= 5, "NUM_EPOCHS minimal 5 agar checkpoint best tervalidasi bisa tersimpan."\n\nif should_run_train(MODEL_NAME) and FORCE_RETRAIN:\n    reset_training_checkpoints_if_requested(MODEL_NAME, FORCE_RETRAIN)\n\nckpt_path = get_model_checkpoint_path(MODEL_NAME)\nif not should_run_train(MODEL_NAME):\n    print(f"⏭️ Skip training {MODEL_NAME}: RUN_PHASE/RUN_MODELS tidak memilih blok ini.")\nelif hybrid_checkpoint_exists(MODEL_NAME) and not FORCE_RETRAIN:\n    print(f"✅ Checkpoint sudah ada. Skip training: {ckpt_path}")\nelse:\n    device = torch.device("cuda")\n    audiomae_model = None\n    maid_model = None\n    film_layer = None\n\n    try:\n        print(f"🔧 Device: {device}")\n        check_batch_size_memory(BATCH_SIZE, min_expected_vram_gb=8.0)\n\n        loaders = make_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)\n        audiomae_model, encoder_fn = build_audiomae_encoder(device)\n        loaders = add_encoder_cache_to_loaders(loaders, encoder_fn, device, MODEL_NAME, num_workers=NUM_WORKERS)\n        maid_model = build_maid_decoder(device)\n        film_layer = build_film_layer(MODEL_NAME, device)\n\n        print_gpu_usage("Sebelum training")\n        train_maid_model(\n            maid_model,\n            encoder_fn,\n            film_layer,\n            loaders["train"],\n            val_loader=loaders["val"],\n            num_epochs=NUM_EPOCHS,\n            lr=LEARNING_RATE,\n            device=device,\n            checkpoint_dir=get_model_checkpoint_dir(MODEL_NAME),\n            model_name=MODEL_NAME,\n            batch_size=BATCH_SIZE,\n            dataset_fraction=DATASET_FRACTION,\n        )\n\n        if not hybrid_checkpoint_exists(MODEL_NAME):\n            raise RuntimeError(f"Checkpoint {MODEL_NAME} tidak tersimpan.")\n\n        print(f"✅ Training selesai. Checkpoint siap dipakai: {ckpt_path}")\n    finally:\n        clear_gpu_memory(audiomae_model, maid_model, film_layer)\n\n\n# ============================================================\n# CELL 11: KOMBINASI 4 — AudioMAE + MAID\n# ============================================================\n# Evaluasi hybrid selalu memakai checkpoint terlatih.\n# Jika checkpoint belum ada, jalankan CELL 11A terlebih dahulu.\n# ============================================================\n\nMODEL_NAME = "audiomae_maid"\nFORCE_REEVAL = True  # <-- True buat re-evaluasi setelah update\n\nif should_run_eval(MODEL_NAME) and FORCE_REEVAL:\n    old_result = os.path.join(PATHS["results"], f"{evaluation_artifact_name(MODEL_NAME)}_results.csv")\n    if os.path.exists(old_result):\n        os.remove(old_result)\n\nif not should_run_eval(MODEL_NAME):\n    print(f"⏭️ Skip evaluasi {MODEL_NAME}: RUN_PHASE/RUN_MODELS tidak memilih blok ini.")\nelif check_if_done(MODEL_NAME):\n    print(f"Model {MODEL_NAME} sudah selesai. Lewati cell ini.")\nelse:\n    ckpt_path = get_model_checkpoint_path(MODEL_NAME)\n    if not hybrid_checkpoint_exists(MODEL_NAME):\n        raise FileNotFoundError(\n            f"Checkpoint {MODEL_NAME} belum ditemukan di {ckpt_path}. Jalankan CELL 11A terlebih dahulu."\n        )\n\n    device = torch.device("cuda")\n    audiomae_model = None\n    maid_model = None\n    film_layer = None\n\n    try:\n        print_gpu_usage("Awal")\n        print(f"📦 Menggunakan checkpoint: {ckpt_path}")\n\n        audiomae_model, encoder_fn = build_audiomae_encoder(device)\n        maid_model = build_maid_decoder(device)\n        film_layer = build_film_layer(MODEL_NAME, device)\n        load_hybrid_checkpoint(MODEL_NAME, maid_model, film_layer, device)\n        print_gpu_usage("Setelah load model terlatih")\n\n        results_df = run_hybrid_inpainting_evaluation(\n            "AudioMAE + MAID",\n            encoder_fn,\n            maid_model,\n            film_layer,\n            device,\n            n_eval_samples=N_EVAL_SAMPLES,\n            model_name=MODEL_NAME,\n        )\n\n        print(f"\\n📋 Hasil {MODEL_NAME}:")\n        print(results_df.to_string(index=False))\n        save_results(results_df, MODEL_NAME)\n    finally:\n        clear_gpu_memory(audiomae_model, maid_model, film_layer)\n\n    print(f"\\n✅ {MODEL_NAME} selesai!")\n\n# ---\n# ## CELL 12 — Gabungkan & Visualisasikan Semua Hasil\n# \n# Jalankan setelah baseline dan seluruh kombinasi hybrid selesai dievaluasi.\n# Untuk hybrid, pastikan cell training dan cell evaluasinya sudah dijalankan sehingga checkpoint dan CSV hasil tersedia.\n# Grafik akan menampilkan **baseline vs 4 model hybrid** untuk perbandingan langsung.\n\n\n# ============================================================\n# CELL 12: VISUALISASI HASIL LENGKAP\n# ============================================================\n# Menampilkan perbandingan semua model:\n# - Baseline: CQT-Diff+ standalone (garis putus-putus)\n# - 4 kombinasi hybrid (garis solid)\n#\n# Gap durations: 100, 300, 500, 750, 1200, 1700 ms\n# ============================================================\n\nif not should_run_summary():\n    print("⏭️ Skip summary/visualisasi: RUN_PHASE tidak memuat summary.")\n    sys.exit(0)\n\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport matplotlib\nimport os\n\nmatplotlib.rcParams[\'figure.dpi\'] = 150\nmatplotlib.rcParams[\'font.size\'] = 10\n\nmaster_path = os.path.join(PATHS["results"], "all_results.csv")\n\nif not os.path.exists(master_path):\n    print("❌ File hasil belum ada. Pastikan Cell 7-11 sudah dijalankan.")\nelse:\n    all_results = pd.read_csv(master_path)\n\n    # Backward compatibility untuk file hasil lama.\n    if "VISQOL_ODG" not in all_results.columns:\n        if "VISQOL" in all_results.columns:\n            all_results["VISQOL_ODG"] = all_results["VISQOL"]\n    if "gap_position" not in all_results.columns:\n        all_results["gap_position"] = "center"\n    all_results = all_results[all_results["gap_position"].fillna("center") == EVAL_GAP_POSITION].copy()\n    if all_results.empty:\n        print(f"❌ Tidak ada hasil untuk gap_position={EVAL_GAP_POSITION}.")\n        sys.exit(0)\n\n    # Cek model mana yang sudah selesai\n    available_models = all_results["model"].unique()\n    print(f"📊 Model yang tersedia: {list(available_models)}")\n\n    gap_durations = sorted(all_results["gap_ms"].unique())\n\n    # ============================================================\n    # TABEL PERBANDINGAN\n    # ============================================================\n    print("\\n" + "="*70)\n    print("TABEL PERBANDINGAN LENGKAP")\n    print("="*70)\n\n    metric_order = [\n        m for m in [\n            "LSD", "LSD_GAP_ONLY", "GAP_SI_SDR", "GAP_SNR", "GAP_MEL_DISTANCE",\n            "FAD", "VISQOL_ODG", "PEAQ_ODG", "GAP_WINDOW_VISQOL_ODG",\n        ]\n        if m in all_results.columns\n    ]\n    for metric in metric_order:\n        if metric in ["LSD", "LSD_GAP_ONLY", "GAP_LSD", "GAP_MEL_DISTANCE", "FAD"]:\n            direction = "↓ lebih rendah = lebih baik"\n        elif metric in ["GAP_SI_SDR", "GAP_SNR"]:\n            direction = "↑ lebih tinggi = lebih baik"\n        else:\n            direction = "↑ mendekati 0 = lebih baik"\n        print(f"\\n{metric} ({direction}):")\n        pivot = all_results.pivot(index="gap_ms", columns="model", values=metric)\n        # Urutkan kolom: baseline dulu, lalu hybrid\n        ordered_cols = [c for c in [\n            "baseline_cqtdiff", "baseline_cqtdiff_finetuned",\n            "clap_cqtdiff", "audiomae_cqtdiff", "clap_maid", "audiomae_maid",\n        ] if c in pivot.columns]\n        print(pivot[ordered_cols].to_string())\n\n\n    # ============================================================\n    # VISUALISASI\n    # ============================================================\n    # Style per model\n    # Baseline: garis putus-putus hitam untuk mudah dibedakan\n    # Hybrid: garis solid berwarna\n    styles = {\n        "baseline_cqtdiff":  {"color": "#000000", "marker": "x", "linestyle": "--",\n                               "label": "Baseline: CQT-Diff+ pretrained", "linewidth": 2.5, "zorder": 10},\n        "baseline_cqtdiff_finetuned": {"color": "#666666", "marker": "P", "linestyle": "-.",\n                               "label": "Baseline: CQT-Diff+ fine-tuned no SSL", "linewidth": 2.2, "zorder": 9},\n        "clap_cqtdiff":      {"color": "#2196F3", "marker": "o", "linestyle": "-",\n                               "label": "CLAP + CQT-Diff+", "linewidth": 1.5, "zorder": 5},\n        "clap_maid":         {"color": "#4CAF50", "marker": "s", "linestyle": "-",\n                               "label": "CLAP + MAID", "linewidth": 1.5, "zorder": 5},\n        "audiomae_cqtdiff":  {"color": "#FF9800", "marker": "^", "linestyle": "-",\n                               "label": "AudioMAE + CQT-Diff+", "linewidth": 1.5, "zorder": 5},\n        "audiomae_maid":     {"color": "#F44336", "marker": "D", "linestyle": "-",\n                               "label": "AudioMAE + MAID", "linewidth": 1.5, "zorder": 5},\n    }\n\n    metrics_info = {\n        "LSD": {"title": "Log Spectral Distance (LSD)",\n                "ylabel": "LSD (dB)",\n                "note": "↓ lebih rendah = lebih baik"},\n        "LSD_GAP_ONLY": {"title": "Gap-only Log Spectral Distance",\n                "ylabel": "Gap LSD (dB)",\n                "note": "↓ lebih rendah = lebih baik"},\n        "GAP_SI_SDR": {"title": "Gap SI-SDR",\n                "ylabel": "SI-SDR (dB)",\n                "note": "↑ lebih tinggi = lebih baik"},\n        "GAP_SNR": {"title": "Gap SNR",\n                "ylabel": "SNR (dB)",\n                "note": "↑ lebih tinggi = lebih baik"},\n        "GAP_MEL_DISTANCE": {"title": "Gap Mel Spectral Distance",\n                "ylabel": "Mean |Mel dB diff|",\n                "note": "↓ lebih rendah = lebih baik"},\n        "FAD": {"title": "Frechet Audio Distance (FAD)",\n                "ylabel": "FAD Score",\n                "note": "↓ lebih rendah = lebih baik"},\n        "VISQOL_ODG": {"title": "ViSQOL Objective Difference Grade",\n                       "ylabel": "VISQOL_ODG Score",\n                       "note": "↑ mendekati 0 = lebih baik"},\n        "PEAQ_ODG": {"title": "GstPEAQ Objective Difference Grade",\n                     "ylabel": "PEAQ_ODG Score",\n                     "note": "↑ mendekati 0 = lebih baik"},\n        "GAP_WINDOW_VISQOL_ODG": {"title": "Gap-window ViSQOL ODG",\n                     "ylabel": "Gap-window VISQOL_ODG",\n                     "note": "↑ mendekati 0 = lebih baik"},\n    }\n    metrics_info = {k: v for k, v in metrics_info.items() if k in metric_order}\n\n    fig, axes = plt.subplots(1, len(metrics_info), figsize=(6 * len(metrics_info), 6))\n    if len(metrics_info) == 1:\n        axes = [axes]\n    fig.suptitle(\n        "Music Audio Inpainting — Baseline vs Hybrid SSL+Diffusion Models\\n"\n        f"Native MusicNet CQTdiff+: {TARGET_SR} Hz, {SEGMENT_SAMPLES} samples ({SEGMENT_DURATION:.2f}s), "\n        f"gap={EVAL_GAP_POSITION}",\n        fontsize=13, fontweight=\'bold\'\n    )\n\n    for ax, (metric, info) in zip(axes, metrics_info.items()):\n        # Plot baseline dan hybrid\n        # Urutan plot: hybrid dulu, baseline paling atas (zorder lebih tinggi)\n        plot_order = [m for m in [\n            "clap_cqtdiff", "audiomae_cqtdiff", "clap_maid", "audiomae_maid",\n            "baseline_cqtdiff_finetuned", "baseline_cqtdiff",\n        ] if m in available_models]\n\n        for model_name in plot_order:\n            model_data = all_results[all_results["model"] == model_name].sort_values("gap_ms")\n            s = styles.get(model_name, {"color": "gray", "marker": "x",\n                                         "linestyle": "-", "label": model_name,\n                                         "linewidth": 1.5, "zorder": 1})\n            ax.plot(\n                model_data["gap_ms"],\n                model_data[metric],\n                color=s["color"],\n                marker=s["marker"],\n                linestyle=s["linestyle"],\n                label=s["label"],\n                linewidth=s["linewidth"],\n                markersize=7,\n                zorder=s["zorder"]\n            )\n\n        ax.set_title(info["title"], fontsize=11, fontweight=\'bold\')\n        ax.set_xlabel("Gap Duration (ms)", fontsize=10)\n        ax.set_ylabel(info["ylabel"], fontsize=10)\n        ax.set_xticks(gap_durations)\n        ax.set_xticklabels([str(g) for g in gap_durations], rotation=45)\n        ax.legend(fontsize=8, loc=\'best\')\n        ax.grid(True, alpha=0.3)\n        ax.text(0.02, 0.98, info["note"],\n                transform=ax.transAxes, fontsize=8,\n                verticalalignment=\'top\', style=\'italic\', color=\'gray\')\n\n    plt.tight_layout()\n\n    # Simpan grafik\n    plot_path = os.path.join(PATHS["plots"], "comparison_plot.png")\n    plt.savefig(plot_path, bbox_inches=\'tight\', dpi=150)\n    plt.show()\n    print(f"\\n💾 Grafik disimpan: {plot_path}")\n\n\n    # ============================================================\n    # RANGKUMAN: IMPROVEMENT HYBRID vs BASELINE\n    # ============================================================\n    if "baseline_cqtdiff" in available_models:\n        print("\\n" + "="*72)\n        print("📈 IMPROVEMENT HYBRID vs PRETRAINED BASELINE (per gap duration)")\n        print("   Positif = lebih baik dari baseline")\n        print("="*72)\n\n        baseline_data = all_results[all_results["model"] == "baseline_cqtdiff"]\n\n        for gap_ms in gap_durations:\n            bl = baseline_data[baseline_data["gap_ms"] == gap_ms].iloc[0]\n            print(f"\\n  Gap {gap_ms}ms:")\n\n            header_cols = ["Model", "ΔLSD"]\n            if "FAD" in all_results.columns:\n                header_cols.append("ΔFAD")\n            if "VISQOL_ODG" in all_results.columns:\n                header_cols.append("ΔVISQOL_ODG")\n            if "PEAQ_ODG" in all_results.columns:\n                header_cols.append("ΔPEAQ_ODG")\n            print(f"  {header_cols[0]:<25} " + " ".join(f"{h:>10}" for h in header_cols[1:]))\n            print(f"  {\'-\'*65}")\n\n            for model_name in ["clap_cqtdiff", "clap_maid", "audiomae_cqtdiff", "audiomae_maid"]:\n                if model_name not in available_models:\n                    continue\n                hybrid = all_results[\n                    (all_results["model"] == model_name) &\n                    (all_results["gap_ms"] == gap_ms)\n                ].iloc[0]\n\n                # Positif selalu berarti hybrid lebih baik.\n                deltas = [bl["LSD"] - hybrid["LSD"]]\n                if "FAD" in all_results.columns:\n                    deltas.append(bl["FAD"] - hybrid["FAD"])\n                if "VISQOL_ODG" in all_results.columns:\n                    deltas.append(hybrid["VISQOL_ODG"] - bl["VISQOL_ODG"])\n                if "PEAQ_ODG" in all_results.columns:\n                    deltas.append(hybrid["PEAQ_ODG"] - bl["PEAQ_ODG"])\n\n                print(f"  {model_name:<25} " + " ".join(f"{d:>+10.4f}" for d in deltas))\n\n    if "baseline_cqtdiff_finetuned" in available_models:\n        print("\\n" + "="*78)\n        print("📈 CQT HYBRID vs FINE-TUNED NO-SSL BASELINE (fair SSL contribution check)")\n        print("   Positif = hybrid lebih baik dari baseline fine-tuned tanpa SSL")\n        print("="*78)\n\n        ft_data = all_results[all_results["model"] == "baseline_cqtdiff_finetuned"]\n        fair_metrics = [\n            m for m in ["LSD_GAP_ONLY", "GAP_SI_SDR", "GAP_SNR", "GAP_MEL_DISTANCE", "FAD", "PEAQ_ODG"]\n            if m in all_results.columns\n        ]\n        for gap_ms in gap_durations:\n            ft = ft_data[ft_data["gap_ms"] == gap_ms].iloc[0]\n            print(f"\\n  Gap {gap_ms}ms:")\n            print(f"  {\'Model\':<25} " + " ".join(f"Δ{m:>14}" for m in fair_metrics))\n            print(f"  {\'-\'*90}")\n            for model_name in ["clap_cqtdiff", "audiomae_cqtdiff"]:\n                if model_name not in available_models:\n                    continue\n                hybrid = all_results[\n                    (all_results["model"] == model_name) &\n                    (all_results["gap_ms"] == gap_ms)\n                ].iloc[0]\n                deltas = []\n                for metric in fair_metrics:\n                    if metric in ["LSD", "LSD_GAP_ONLY", "GAP_LSD", "GAP_MEL_DISTANCE", "FAD"]:\n                        deltas.append(ft[metric] - hybrid[metric])\n                    else:\n                        deltas.append(hybrid[metric] - ft[metric])\n                print(f"  {model_name:<25} " + " ".join(f"{d:>+15.4f}" for d in deltas))\n\n    summary_df = update_experiment_summary()\n    print(f"\\n✅ Semua hasil tersimpan di: {PATHS[\'results\']}")\n    print(f"Plots tersimpan di: {PATHS[\'plots\']}")\n\n\nmaybe_auto_stop_instance()\n'


In [ ]:
# Embedded adapter modules required by the official-only pipeline.
EMBEDDED_ADAPTER_SOURCES = {'official_cqtdiff_adapter.py': 'import os\nimport sys\nfrom contextlib import contextmanager\n\nimport numpy as np\nimport scipy.signal\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torchaudio\nfrom tqdm import tqdm\n\n\n@contextmanager\ndef _prepend_path(path):\n    path = os.path.abspath(path)\n    already_present = path in sys.path\n    if not already_present:\n        sys.path.insert(0, path)\n    try:\n        yield\n    finally:\n        if not already_present:\n            try:\n                sys.path.remove(path)\n            except ValueError:\n                pass\n\n\n# ============================================================\n# Diffusion Parameters (Karras VE-SDE Elucidating)\n# Replika ringan dari src/sde.py di repo CQTdiff\n# ============================================================\n\nclass DiffusionParams:\n    """Karras VE-SDE noise schedule + preconditioning buat sampling."""\n\n    def __init__(\n        self,\n        sigma_data=0.057,\n        sigma_min=1e-4,\n        sigma_max=1.0,\n        ro=13,\n        Schurn=5,\n        Snoise=1.0,\n        Stmin=0,\n        Stmax=50,\n    ):\n        self.sigma_data = sigma_data\n        self.sigma_min = sigma_min\n        self.sigma_max = sigma_max\n        self.ro = ro\n        self.Schurn = Schurn\n        self.Snoise = Snoise\n        self.Stmin = Stmin\n        self.Stmax = Stmax\n\n    def create_schedule(self, nb_steps):\n        """Buat jadwal noise level dari sigma_max ke 0."""\n        i = torch.arange(0, nb_steps + 1)\n        t = (\n            self.sigma_max ** (1 / self.ro)\n            + i / (nb_steps - 1) * (self.sigma_min ** (1 / self.ro) - self.sigma_max ** (1 / self.ro))\n        ) ** self.ro\n        t[-1] = 0\n        return t\n\n    def sample_prior(self, shape, sigma):\n        if torch.is_tensor(sigma):\n            return torch.randn(shape, device=sigma.device, dtype=sigma.dtype) * sigma\n        return torch.randn(shape) * sigma\n\n    def get_gamma(self, t):\n        """Stochasticity parameter per timestep."""\n        N = t.shape[0]\n        gamma = torch.zeros_like(t)\n        indexes = torch.logical_and(t > self.Stmin, t < self.Stmax)\n        gamma[indexes] = min(self.Schurn / N, 2 ** 0.5 - 1)\n        return gamma\n\n    def cskip(self, sigma):\n        return self.sigma_data ** 2 * (sigma ** 2 + self.sigma_data ** 2) ** -1\n\n    def cout(self, sigma):\n        return sigma * self.sigma_data * (self.sigma_data ** 2 + sigma ** 2) ** (-0.5)\n\n    def cin(self, sigma):\n        return (self.sigma_data ** 2 + sigma ** 2) ** (-0.5)\n\n    def cnoise(self, sigma):\n        return (1 / 4) * torch.log(sigma + 1e-44)\n\n    def denoiser(self, x, model, sigma):\n        """Full denoiser step: preconditioning + model forward."""\n        device_type = x.device.type\n        with torch.autocast(device_type=device_type, enabled=False):\n            x = x.float()\n            sigma = sigma.to(device=x.device, dtype=torch.float32).unsqueeze(-1)\n            return self.cskip(sigma) * x + self.cout(sigma) * model(\n                self.cin(sigma) * x,\n                self.cnoise(sigma),\n            )\n\n\n# ============================================================\n# Helper functions\n# ============================================================\n\ndef _load_cqtdiff_config(cqt_diff_dir, device):\n    try:\n        from omegaconf import OmegaConf\n    except Exception as exc:\n        raise RuntimeError(\n            "CQTdiff membutuhkan omegaconf/hydra-core. Install requirements repo CQTdiff terlebih dahulu."\n        ) from exc\n\n    cfg_path = os.path.join(cqt_diff_dir, "conf", "conf.yaml")\n    if not os.path.exists(cfg_path):\n        raise FileNotFoundError(f"Config CQTdiff tidak ditemukan: {cfg_path}")\n\n    cfg = OmegaConf.load(cfg_path)\n    cfg.device = str(device)\n    cfg.log = False\n    cfg.restore = False\n    cfg.save_model = False\n    return cfg\n\n\n\n\ndef _find_cqtdiff_weights(cqt_diff_dir):\n    candidates = [\n        os.environ.get("CQTDIFF_WEIGHTS"),\n        os.path.join(cqt_diff_dir, "experiments", "cqt", "cqt_weights.pt"),\n    ]\n    weights_dir = os.path.join(cqt_diff_dir, "experiments", "cqt")\n    if os.path.isdir(weights_dir):\n        candidates.extend(\n            os.path.join(weights_dir, name)\n            for name in sorted(os.listdir(weights_dir))\n            if name.startswith("weights-") and name.endswith(".pt")\n        )\n    return next((path for path in candidates if path and os.path.exists(path)), None)\n\n\ndef _strip_module_prefix(state):\n    return {str(k).replace("module.", "", 1): v for k, v in state.items()}\n\n\ndef _load_ema_weights_from_checkpoint(weights_path, device, live_model=None):\n    """\n    Load EMA weights dari checkpoint CQT-Diff.\n    EMA lebih stabil buat inference dibanding raw training weights.\n\n    Parameter live_model diperlukan untuk menentukan key mana yang\n    merupakan parameter (trainable) vs buffer (batch norm stats dll).\n    EMA weights cuma ada buat parameter, bukan buffer.\n    """\n    checkpoint = torch.load(weights_path, map_location=device, weights_only=False)\n\n    if not isinstance(checkpoint, dict):\n        raise TypeError(f"Checkpoint format tidak dikenali: {weights_path}")\n\n    if "ema_weights" not in checkpoint or "model" not in checkpoint:\n        state = checkpoint.get("model", checkpoint)\n        return _strip_module_prefix(state), "raw"\n\n    model_state = checkpoint["model"]\n    ema_list = checkpoint["ema_weights"]\n\n    # Cari tahu key mana yang parameter (trainable) vs buffer\n    if live_model is not None:\n        param_keys = {n for n, _ in live_model.named_parameters()}\n    else:\n        param_keys = None\n\n    dic_ema = {}\n    ema_idx = 0\n\n    for key in model_state.keys():\n        clean_key = str(key).replace("module.", "", 1)\n\n        is_param = False\n        if param_keys is not None:\n            is_param = clean_key in param_keys or key in param_keys\n        else:\n            is_param = model_state[key].is_floating_point()\n\n        if is_param and ema_idx < len(ema_list):\n            dic_ema[clean_key] = ema_list[ema_idx]\n            ema_idx += 1\n        else:\n            dic_ema[clean_key] = model_state[key]\n\n    weight_type = "ema" if ema_idx > 0 else "raw"\n    print(f"  EMA weights mapped: {ema_idx}/{len(ema_list)} params, "\n          f"{len(dic_ema) - ema_idx} buffers from raw checkpoint")\n    return dic_ema, weight_type\n\n\n# ============================================================\n# Main Adapter\n# ============================================================\n\nclass OfficialCQTDiffHybridDecoder(nn.Module):\n    """\n    Adapter di atas official CQTdiff U-Net.\n\n    Dua mode inferensi:\n    - Baseline (conditioning=None): multi-step reverse diffusion sampling\n      menggunakan pipeline resmi CQT-Diff (T langkah Heun sampler)\n    - Hybrid (conditioning!=None): multi-step reverse diffusion sampling yang\n      denoiser-nya dikondisikan oleh representasi SSL\n\n    Training (hybrid path): get_features -> FiLM -> diffusion_loss. STFT head\n    hanya menjadi proyektor conditioning ke residual denoiser, bukan post-process\n    setelah diffusion selesai.\n    """\n\n    DIFFUSION_STEPS = int(os.environ.get("CQTDIFF_DIFFUSION_STEPS", "35"))\n    DIFFUSION_XI = float(os.environ.get("CQTDIFF_DIFFUSION_XI", "0"))\n    DIFFUSION_SIGMA_MIN = float(os.environ.get("CQTDIFF_SIGMA_MIN", "1e-4"))\n    DIFFUSION_SIGMA_MAX = float(os.environ.get("CQTDIFF_SIGMA_MAX", "1.0"))\n\n    def __init__(self, device, target_sr, segment_samples, gap_durations_ms, cqt_diff_dir):\n        super().__init__()\n        self.device_ref = torch.device(device)\n        self.target_sr = int(target_sr)\n        self.target_len = int(segment_samples)\n        self.gap_durations_ms = list(gap_durations_ms)\n        self.cqt_diff_dir = os.path.abspath(cqt_diff_dir)\n        self.architecture_name = "ssl_conditioned_cqtdiff_v2_balanced_gap_loss"\n\n        with _prepend_path(self.cqt_diff_dir):\n            from src.models.unet_cqt import Unet_CQT\n\n            self.args = _load_cqtdiff_config(self.cqt_diff_dir, self.device_ref)\n            self.native_sr = int(self.args.sample_rate)\n            self.native_len = int(self.args.audio_len)\n            self.backbone = Unet_CQT(self.args, self.device_ref).to(self.device_ref)\n\n        require_native = os.environ.get("CQTDIFF_REQUIRE_NATIVE_SHAPE", "1").lower() in {"1", "true", "yes", "on"}\n        if require_native and (self.target_sr != self.native_sr or self.target_len != self.native_len):\n            raise RuntimeError(\n                "CQT-Diff adapter harus berjalan pada native checkpoint shape untuk eksperimen final: "\n                f"target_sr={self.target_sr}, target_len={self.target_len}, "\n                f"native_sr={self.native_sr}, native_len={self.native_len}. "\n                "Ubah pipeline ke 22050 Hz dan 65536 samples, atau set "\n                "CQTDIFF_REQUIRE_NATIVE_SHAPE=0 hanya untuk diagnosis non-paper."\n            )\n        print(\n            "CQT-Diff native setup: "\n            f"sr={self.native_sr} Hz, len={self.native_len} samples, "\n            f"duration={self.native_len / self.native_sr:.3f}s"\n        )\n\n        # ---- Load EMA weights (lebih baik daripada raw training weights) ----\n        weights_path = _find_cqtdiff_weights(self.cqt_diff_dir)\n        if weights_path is None:\n            raise FileNotFoundError(\n                "Checkpoint CQT-Diff+ asli belum ditemukan. Jalankan "\n                "external/CQTdiff/download_weights_and_examples.sh atau set CQTDIFF_WEIGHTS."\n            )\n        self.official_weights_path = os.path.abspath(weights_path)\n\n        ema_state, weight_type = _load_ema_weights_from_checkpoint(\n            weights_path, self.device_ref, live_model=self.backbone\n        )\n        self.official_weights_type = weight_type\n        msg = self.backbone.load_state_dict(ema_state, strict=False)\n        print(f"CQT-Diff+ {weight_type} weights loaded: {weights_path}")\n        print(f"CQT-Diff+ load_state_dict: {msg}")\n\n        # Freeze backbone by default\n        train_backbone = os.environ.get("CQTDIFF_TRAIN_BACKBONE", "0").lower() in {"1", "true", "yes"}\n        if not train_backbone:\n            for param in self.backbone.parameters():\n                param.requires_grad_(False)\n        trainable_backbone_params = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)\n        total_backbone_params = sum(p.numel() for p in self.backbone.parameters())\n        print(\n            "CQT-Diff+ backbone training: "\n            f"{\'enabled\' if train_backbone else \'disabled\'} "\n            f"({trainable_backbone_params:,}/{total_backbone_params:,} trainable params)"\n        )\n\n        # ---- Diffusion sampling parameters (sesuai scripts/sampling_inpainting.sh) ----\n        self.diff_params = DiffusionParams(\n            sigma_data=0.057,\n            sigma_min=self.DIFFUSION_SIGMA_MIN,\n            sigma_max=self.DIFFUSION_SIGMA_MAX,\n            ro=13,\n            Schurn=5,\n            Snoise=1.0,\n            Stmin=0,\n            Stmax=50,\n        )\n        print(\n            f"Diffusion sampling config: T={self.DIFFUSION_STEPS}, "\n            f"sigma=[{self.DIFFUSION_SIGMA_MIN}, {self.DIFFUSION_SIGMA_MAX}], "\n            f"xi={self.DIFFUSION_XI}"\n        )\n\n        # ---- Low-pass filter buat buang artifact Nyquist setelah diffusion ----\n        lpf_coeffs = scipy.signal.firwin(\n            numtaps=100, cutoff=10000, width=1, window="kaiser", fs=self.native_sr\n        )\n        self.register_buffer(\n            "_lpf_kernel",\n            torch.FloatTensor(lpf_coeffs).unsqueeze(0).unsqueeze(0),\n        )\n\n        # ---- Conditioning head untuk hybrid diffusion ----\n        self.n_fft = int(os.environ.get("CQTDIFF_ADAPTER_N_FFT", 2048))\n        self.hop_length = int(os.environ.get("CQTDIFF_ADAPTER_HOP", 512))\n        self.feature_dim = int(os.environ.get("CQTDIFF_ADAPTER_FEATURE_DIM", 256))\n        self.freq_bins = self.n_fft // 2 + 1\n        self.sigma = float(os.environ.get("CQTDIFF_ADAPTER_SIGMA", "0.1"))\n\n        self.feature_encoder = nn.Sequential(\n            nn.Linear(self.freq_bins * 2 + 1, 512),\n            nn.SiLU(),\n            nn.Linear(512, self.feature_dim),\n            nn.SiLU(),\n        )\n        self.spec_decoder = nn.Sequential(\n            nn.Linear(self.feature_dim, 512),\n            nn.SiLU(),\n            nn.Linear(512, self.freq_bins * 2),\n        )\n        self.condition_gate = nn.Sequential(\n            nn.Linear(self.feature_dim, 128),\n            nn.SiLU(),\n            nn.Linear(128, 1),\n        )\n        nn.init.zeros_(self.condition_gate[-1].weight)\n        nn.init.constant_(self.condition_gate[-1].bias, -2.0)\n\n        # σ-adaptive scaling: residual magnitude varies with noise level\n        self.sigma_scale_net = nn.Sequential(\n            nn.Linear(1, 64),\n            nn.SiLU(),\n            nn.Linear(64, 1),\n        )\n        nn.init.zeros_(self.sigma_scale_net[-1].weight)\n        nn.init.zeros_(self.sigma_scale_net[-1].bias)\n\n    # ================================================================\n    # Utility: resample + pad/crop\n    # ================================================================\n\n    def _pad_or_crop(self, x, length):\n        if x.shape[-1] < length:\n            return F.pad(x, (0, length - x.shape[-1]))\n        if x.shape[-1] > length:\n            start = (x.shape[-1] - length) // 2\n            return x[..., start : start + length]\n        return x\n\n    def _to_native(self, audio):\n        x = audio.squeeze(1) if audio.dim() == 3 else audio\n        x = x.float()\n        if self.target_sr != self.native_sr:\n            x = torchaudio.functional.resample(x, self.target_sr, self.native_sr)\n        return self._pad_or_crop(x, self.native_len)\n\n    def _from_native(self, audio, target_len):\n        x = audio.squeeze(1) if audio.dim() == 3 else audio\n        if self.native_sr != self.target_sr:\n            x = torchaudio.functional.resample(x, self.native_sr, self.target_sr)\n        return self._pad_or_crop(x, target_len)\n\n    def _target_mask_to_native_keep(self, user_mask):\n        """\n        Konversi mask target-sr ke domain native CQT-Diff.\n\n        Input user_mask memakai konvensi True=gap. Output memakai konvensi\n        CQT-Diff: 1=observed/keep, 0=gap.\n        """\n        gap = user_mask.float()\n        if self.target_sr != self.native_sr:\n            gap = torchaudio.functional.resample(gap, self.target_sr, self.native_sr)\n        gap = self._pad_or_crop(gap, self.native_len)\n        return (gap < 0.5).float()\n\n    def _conditioning_to_waveform(self, conditioning, target_len):\n        """\n        Decode representasi conditioning menjadi residual waveform.\n\n        Residual ini dipakai di dalam denoiser diffusion pada setiap step, bukan\n        sebagai refinement terpisah setelah hasil CQT-Diff keluar.\n        """\n        if conditioning is None:\n            return None\n        conditioning = conditioning.to(self.device_ref, dtype=torch.float32)\n        if conditioning.dim() == 2:\n            conditioning = conditioning.unsqueeze(1)\n\n        pred_spec_features = self.decode_features(conditioning).permute(0, 2, 1).contiguous()\n        device_type = self.device_ref.type\n        with torch.autocast(device_type=device_type, enabled=False):\n            bsz, _, n_frames = pred_spec_features.shape\n            pred_pairs = (\n                pred_spec_features.float()\n                .reshape(bsz, self.freq_bins, 2, n_frames)\n                .permute(0, 1, 3, 2)\n                .contiguous()\n            )\n            recon_spec = torch.view_as_complex(pred_pairs)\n            window = torch.hann_window(self.n_fft, device=pred_spec_features.device)\n            wave = torch.istft(\n                recon_spec,\n                n_fft=self.n_fft,\n                hop_length=self.hop_length,\n                window=window,\n                length=target_len,\n            )\n\n        pooled = conditioning.mean(dim=1)\n        gate = torch.sigmoid(self.condition_gate(pooled)).clamp(0.0, 1.0)\n        return gate * wave\n\n    def _conditioned_model(self, conditioning, target_len):\n        """\n        Bungkus backbone official dengan σ-adaptive residual conditioning SSL.\n\n        Signature mengikuti Unet_CQT(x, sigma) agar bisa dipakai langsung oleh\n        DiffusionParams.denoiser selama reverse diffusion.\n\n        Residual di-scale oleh sigma_scale_net(σ) sehingga magnitude\n        conditioning bisa beradaptasi terhadap noise level: coarse guidance\n        di σ tinggi vs fine detail di σ rendah.\n        """\n        cond_wave_target = self._conditioning_to_waveform(conditioning, target_len)\n        if cond_wave_target is None:\n            return self.backbone\n\n        cond_wave_native = self._to_native(cond_wave_target)\n        sigma_scale_fn = self.sigma_scale_net\n\n        def model(x, sigma):\n            backbone_trainable = any(param.requires_grad for param in self.backbone.parameters())\n            if backbone_trainable:\n                base = self.backbone(x, sigma)\n            else:\n                self.backbone.eval()\n                with torch.no_grad():\n                    base = self.backbone(x, sigma)\n            residual = cond_wave_native.to(device=x.device, dtype=x.dtype)\n            if residual.shape[0] != x.shape[0]:\n                residual = residual.expand(x.shape[0], -1)\n\n            # sigma is cnoise-transformed: (1/4)*log(raw_sigma), shape (B,1) or (1,1)\n            sigma_val = sigma\n            if sigma_val.dim() == 0:\n                sigma_val = sigma_val.view(1, 1)\n            elif sigma_val.dim() == 1:\n                sigma_val = sigma_val.unsqueeze(-1)\n            scale = 1.0 + sigma_scale_fn(sigma_val)\n            return base + scale * residual\n\n        return model\n\n    # ================================================================\n    # Diffusion sampling loop (Heun 2nd-order, sesuai src/sampler.py)\n    # ================================================================\n\n    def _run_diffusion_sampling(self, y, mask, T=None, show_progress=True, conditioning=None):\n        """\n        Multi-step reverse diffusion inpainting (2nd-order Heun sampler).\n\n        y    : (B, native_len) observed audio, zeros di area gap\n        mask : (B, native_len) float, 1=keep 0=gap (konvensi CQT-Diff)\n        T    : jumlah langkah diffusion (default dari DIFFUSION_STEPS)\n        """\n        if T is None:\n            T = self.DIFFUSION_STEPS\n        device = y.device\n        shape = y.shape\n        denoiser_model = self._conditioned_model(conditioning, target_len=self.target_len)\n\n        t = self.diff_params.create_schedule(T).to(device)\n        x = self.diff_params.sample_prior(shape, t[0]).to(device)\n        gamma = self.diff_params.get_gamma(t).to(device)\n\n        self.backbone.eval()\n        with torch.no_grad():\n            for i in tqdm(range(T), desc="Diffusion inpainting", leave=False, disable=not show_progress):\n                # Stochastic injection (Langevin-style)\n                if gamma[i] == 0:\n                    t_hat = t[i]\n                    x_hat = x\n                else:\n                    t_hat = t[i] + gamma[i] * t[i]\n                    epsilon = torch.randn(shape, device=device) * self.diff_params.Snoise\n                    x_hat = x + ((t_hat ** 2 - t[i] ** 2) ** 0.5) * epsilon\n\n                # Denoise + data consistency (replacement method)\n                # mask=1 → observed (pakai y), mask=0 → gap (pakai prediksi denoiser)\n                denoised = self.diff_params.denoiser(x_hat, denoiser_model, t_hat.unsqueeze(-1))\n                denoised = mask * y + (1.0 - mask) * denoised\n\n                score = (denoised - x_hat) / t_hat ** 2\n                d = -t_hat * score\n                h = t[i + 1] - t_hat\n\n                if t[i + 1] != 0:\n                    # 2nd order Heun correction\n                    x_prime = x_hat + h * d\n                    denoised_prime = self.diff_params.denoiser(\n                        x_prime, denoiser_model, t[i + 1].unsqueeze(-1)\n                    )\n                    denoised_prime = mask * y + (1.0 - mask) * denoised_prime\n\n                    score_prime = (denoised_prime - x_prime) / t[i + 1] ** 2\n                    d_prime = -t[i + 1] * score_prime\n                    x = x_hat + h * (0.5 * d + 0.5 * d_prime)\n                else:\n                    # Last step: 1st order Euler\n                    x = x_hat + h * d\n\n        return x.detach()\n\n    def _apply_lowpass(self, x):\n        """FIR low-pass filter buat buang artifact Nyquist setelah diffusion."""\n        return F.conv1d(\n            x.unsqueeze(1),\n            self._lpf_kernel.to(x.device),\n            padding="same",\n        ).squeeze(1)\n\n    # ================================================================\n    # Inpainting: baseline (diffusion) vs hybrid (reconstruction)\n    # ================================================================\n\n    def _inpaint_diffusion(self, masked_audio, mask, T=None, show_progress=True, conditioning=None):\n        """\n        Inpainting via multi-step reverse diffusion.\n\n        masked_audio : (B, target_len) audio di target_sr, zeros di gap\n        mask         : (B, target_len) bool, True=gap\n        conditioning : optional SSL-conditioned features untuk hybrid diffusion\n        """\n        B = masked_audio.shape[0]\n        target_len = masked_audio.shape[-1]\n        if os.environ.get("CQTDIFF_REQUIRE_NATIVE_SHAPE", "1").lower() in {"1", "true", "yes", "on"}:\n            if target_len != self.native_len:\n                raise RuntimeError(\n                    f"Input CQT-Diff harus native_len={self.native_len}, tetapi menerima {target_len}."\n                )\n            if mask.shape[-1] != self.native_len:\n                raise RuntimeError(\n                    f"Mask CQT-Diff harus native_len={self.native_len}, tetapi menerima {mask.shape[-1]}."\n                )\n\n        # Cari batas gap di domain asli (target_sr)\n        mask_bool = mask[0].bool()\n        gap_indices = torch.where(mask_bool)[0]\n        if len(gap_indices) == 0:\n            out = masked_audio.detach().cpu().numpy()\n            return out[0] if B == 1 else out\n\n        gap_start_orig = gap_indices[0].item()\n        gap_end_orig = gap_indices[-1].item() + 1\n\n        # Resample ke native SR\n        if self.target_sr != self.native_sr:\n            audio_native = torchaudio.functional.resample(\n                masked_audio, self.target_sr, self.native_sr\n            )\n        else:\n            audio_native = masked_audio.clone()\n        nat_len = audio_native.shape[-1]\n\n        # Center crop ke native_len (model expectation)\n        if nat_len > self.native_len:\n            crop_offset = (nat_len - self.native_len) // 2\n            audio_crop = audio_native[..., crop_offset : crop_offset + self.native_len]\n        elif nat_len < self.native_len:\n            crop_offset = 0\n            audio_crop = F.pad(audio_native, (0, self.native_len - nat_len))\n        else:\n            crop_offset = 0\n            audio_crop = audio_native\n\n        # Map gap boundaries ke domain native yang sudah di-crop\n        sr_scale = self.native_sr / self.target_sr\n        gap_start_nat = int(round(gap_start_orig * sr_scale)) - crop_offset\n        gap_end_nat = int(round(gap_end_orig * sr_scale)) - crop_offset\n        gap_start_nat = max(0, gap_start_nat)\n        gap_end_nat = min(self.native_len, gap_end_nat)\n\n        # Buat mask CQT-Diff: 1=keep, 0=gap (kebalikan dari user convention)\n        cqtdiff_mask = torch.ones(\n            (B, self.native_len), device=masked_audio.device, dtype=torch.float32\n        )\n        cqtdiff_mask[..., gap_start_nat:gap_end_nat] = 0.0\n\n        # y = observed audio (gap sudah nol)\n        y = cqtdiff_mask * audio_crop\n\n        # Jalankan multi-step reverse diffusion\n        x_hat = self._run_diffusion_sampling(\n            y,\n            cqtdiff_mask,\n            T=T,\n            show_progress=show_progress,\n            conditioning=conditioning,\n        )\n        x_hat = self._apply_lowpass(x_hat)\n\n        # Ambil konten gap dari hasil diffusion\n        gap_native = x_hat[..., gap_start_nat:gap_end_nat].clone()\n\n        # Resample gap content balik ke target SR\n        if self.target_sr != self.native_sr:\n            gap_target = torchaudio.functional.resample(\n                gap_native, self.native_sr, self.target_sr\n            )\n        else:\n            gap_target = gap_native\n\n        # Sesuaikan panjang ke ukuran gap asli\n        gap_len_orig = gap_end_orig - gap_start_orig\n        if gap_target.shape[-1] > gap_len_orig:\n            gap_target = gap_target[..., :gap_len_orig]\n        elif gap_target.shape[-1] < gap_len_orig:\n            gap_target = F.pad(gap_target, (0, gap_len_orig - gap_target.shape[-1]))\n\n        # Taruh gap content ke audio asli\n        output = masked_audio.clone()\n        output[..., gap_start_orig:gap_end_orig] = gap_target\n\n        out = output.detach().cpu().numpy()\n        return out[0] if B == 1 else out\n\n    _DETERMINISTIC_SIGMA_STEPS = 8\n\n    def diffusion_loss(self, clean_audio, masked_audio, mask, conditioning=None,\n                       deterministic_sigma=False):\n        """\n        Denoising objective untuk hybrid CQT-Diff.\n\n        Representasi SSL masuk ke denoiser diffusion melalui conditioning.\n        Loss dihitung pada gap region sehingga model belajar mengisi bagian\n        hilang dengan bantuan SSL, bukan merefine hasil sampling setelahnya.\n\n        deterministic_sigma: jika True, gunakan set sigma tetap agar validation\n        loss stabil dan reproducible antar epoch.\n        """\n        if os.environ.get("CQTDIFF_REQUIRE_NATIVE_SHAPE", "1").lower() in {"1", "true", "yes", "on"}:\n            if clean_audio.shape[-1] != self.native_len or masked_audio.shape[-1] != self.native_len:\n                raise RuntimeError(\n                    f"Training CQT-Diff harus native_len={self.native_len}, "\n                    f"clean={clean_audio.shape[-1]}, masked={masked_audio.shape[-1]}."\n                )\n        clean_native = self._to_native(clean_audio).float()\n        masked_native = self._to_native(masked_audio).float()\n        keep_mask = self._target_mask_to_native_keep(mask).to(clean_native.device)\n        gap_mask = 1.0 - keep_mask\n\n        batch_size = clean_native.shape[0]\n        if deterministic_sigma:\n            log_sigma_grid = torch.linspace(\n                np.log(self.DIFFUSION_SIGMA_MIN),\n                np.log(self.DIFFUSION_SIGMA_MAX),\n                steps=self._DETERMINISTIC_SIGMA_STEPS,\n                device=clean_native.device,\n            )\n            idx = torch.arange(batch_size, device=clean_native.device) % self._DETERMINISTIC_SIGMA_STEPS\n            sigma = torch.exp(log_sigma_grid[idx]).unsqueeze(1)\n        else:\n            sigma = torch.exp(\n                torch.empty(batch_size, 1, device=clean_native.device).uniform_(\n                    np.log(self.DIFFUSION_SIGMA_MIN),\n                    np.log(self.DIFFUSION_SIGMA_MAX),\n                )\n            )\n        noise = torch.randn_like(clean_native) * sigma\n        x_noisy = keep_mask * masked_native + gap_mask * (clean_native + noise)\n\n        denoiser_model = self._conditioned_model(conditioning, target_len=clean_audio.shape[-1])\n        denoised = self.diff_params.denoiser(x_noisy, denoiser_model, sigma.squeeze(1))\n        denoised = keep_mask * masked_native + gap_mask * denoised\n\n        denom = gap_mask.sum(dim=1).clamp_min(1.0)\n        # Balance per sample, not per gap sample. Otherwise long gaps dominate\n        # the gradient: 1700ms contributes ~17x more elements than 100ms.\n        per_sample_gap_l1 = ((denoised - clean_native).abs() * gap_mask).sum(dim=1) / denom\n        gap_loss = per_sample_gap_l1.mean()\n        full_loss = (denoised - clean_native).abs().mean(dim=1).mean()\n\n        pred_rms = torch.sqrt(((denoised * gap_mask).pow(2).sum(dim=1) / denom).clamp_min(1e-10))\n        target_rms = torch.sqrt(((clean_native * gap_mask).pow(2).sum(dim=1) / denom).clamp_min(1e-10))\n        energy_loss = F.l1_loss(torch.log(pred_rms + 1e-5), torch.log(target_rms + 1e-5))\n        loss = gap_loss + 0.1 * full_loss + 0.05 * energy_loss\n        return {\n            "loss": loss,\n            "gap_loss": gap_loss,\n            "full_loss": full_loss,\n            "energy_loss": energy_loss,\n        }\n\n    def _inpaint_reconstruction(self, masked_audio, mask, conditioning,\n                                diffusion_audio=None):\n        return self._inpaint_diffusion(masked_audio, mask, conditioning=conditioning)\n        """\n        Legacy path disabled: hybrid CQT uses SSL-conditioned diffusion.\n\n        Stage 1: Multi-step diffusion (sama dgn baseline) → initial recon\n                 Legacy branch is unreachable after the compatibility return.\n        Stage 2: FiLM-conditioned spec_decoder → iSTFT → refined gap\n                 Kept only to avoid breaking old references.\n\n        conditioning: (B, T_frames, feature_dim) — sudah di-FiLM-kan di luar\n        diffusion_audio: legacy arg, ignored by active path\n        """\n        # Stage 1: Diffusion inpainting\n        if diffusion_audio is not None:\n            diffusion_tensor = diffusion_audio.to(self.device_ref)\n        else:\n            diffusion_result = self._inpaint_diffusion(masked_audio, mask)\n            if isinstance(diffusion_result, np.ndarray):\n                diffusion_tensor = torch.from_numpy(diffusion_result).float()\n                if diffusion_tensor.dim() == 1:\n                    diffusion_tensor = diffusion_tensor.unsqueeze(0)\n                diffusion_tensor = diffusion_tensor.to(self.device_ref)\n            else:\n                diffusion_tensor = diffusion_result\n\n        # Stage 2: Refine gap pakai FiLM-conditioned spectrogram decoder\n        conditioning = conditioning.to(self.device_ref, dtype=torch.float32)\n        if conditioning.dim() == 2:\n            conditioning = conditioning.unsqueeze(1)\n\n        pred_spec_features = self.decode_features(conditioning).permute(0, 2, 1).contiguous()\n\n        device_type = self.device_ref.type\n        with torch.autocast(device_type=device_type, enabled=False):\n            bsz, _, n_frames = pred_spec_features.shape\n            pred_pairs = (\n                pred_spec_features.float()\n                .reshape(bsz, self.freq_bins, 2, n_frames)\n                .permute(0, 1, 3, 2)\n                .contiguous()\n            )\n            recon_spec = torch.view_as_complex(pred_pairs)\n            window = torch.hann_window(self.n_fft, device=masked_audio.device)\n            refined = torch.istft(\n                recon_spec,\n                n_fft=self.n_fft,\n                hop_length=self.hop_length,\n                window=window,\n                length=masked_audio.shape[-1],\n            )\n\n        # Blend: gap pakai rata-rata diffusion + refinement, observed pakai asli\n        gap_blend = 0.5 * diffusion_tensor + 0.5 * refined\n        output = torch.where(mask.bool(), gap_blend, masked_audio)\n        out = output.detach().cpu().numpy()\n        return out[0] if output.shape[0] == 1 else out\n\n    def inpaint(self, masked_audio, mask, conditioning=None, diffusion_audio=None):\n        """\n        Entry point inpainting.\n\n        conditioning=None  -> baseline: multi-step reverse diffusion\n        conditioning!=None -> hybrid: SSL-conditioned reverse diffusion\n        diffusion_audio    -> legacy arg, diabaikan\n        """\n        if masked_audio.dim() == 1:\n            masked_audio = masked_audio.unsqueeze(0)\n        if mask.dim() == 1:\n            mask = mask.unsqueeze(0)\n        masked_audio = masked_audio.to(self.device_ref, dtype=torch.float32)\n        mask = mask.to(self.device_ref)\n\n        return self._inpaint_diffusion(masked_audio, mask, conditioning=conditioning)\n\n    # ================================================================\n    # Feature extraction (tetap buat hybrid FiLM training)\n    # ================================================================\n\n    def _backbone_predict(self, audio):\n        """Single-pass backbone prediction (buat feature extraction, bukan inpainting)."""\n        device_type = self.device_ref.type\n        with torch.autocast(device_type=device_type, enabled=False):\n            native = self._to_native(audio).float()\n            sigma = torch.full(\n                (native.shape[0], 1), self.sigma, device=native.device, dtype=torch.float32\n            )\n            backbone_trainable = any(param.requires_grad for param in self.backbone.parameters())\n            if backbone_trainable:\n                pred = self.backbone(native, sigma)\n            else:\n                self.backbone.eval()\n                with torch.no_grad():\n                    pred = self.backbone(native, sigma)\n            return self._from_native(pred.float(), audio.shape[-1])\n\n    def _sample_to_frame_mask(self, mask, n_frames):\n        pooled = F.avg_pool1d(\n            mask.float().unsqueeze(1),\n            kernel_size=self.hop_length,\n            stride=self.hop_length,\n            ceil_mode=True,\n        ).squeeze(1)\n        if pooled.shape[1] < n_frames:\n            pooled = F.pad(pooled, (0, n_frames - pooled.shape[1]))\n        elif pooled.shape[1] > n_frames:\n            pooled = pooled[:, :n_frames]\n        return pooled\n\n    def get_features(self, x, mask=None):\n        """Extract STFT-based features (buat hybrid training dengan FiLM)."""\n        device_type = self.device_ref.type\n        with torch.autocast(device_type=device_type, enabled=False):\n            x = x.squeeze(1) if x.dim() == 3 else x\n            x = x.float()\n            backbone_pred = self._backbone_predict(x)\n            if mask is not None:\n                base = torch.where(mask.bool(), backbone_pred, x)\n            else:\n                base = backbone_pred\n\n            window = torch.hann_window(self.n_fft, device=base.device)\n            spec = torch.stft(\n                base.float(),\n                n_fft=self.n_fft,\n                hop_length=self.hop_length,\n                window=window,\n                return_complex=True,\n            )\n            spec_ri = torch.view_as_real(spec).permute(0, 2, 1, 3).contiguous()\n            spec_features = spec_ri.reshape(spec.shape[0], spec.shape[2], self.freq_bins * 2)\n            if mask is None:\n                frame_mask = torch.zeros(\n                    spec_features.shape[0],\n                    spec_features.shape[1],\n                    device=spec_features.device,\n                    dtype=spec_features.dtype,\n                )\n            else:\n                frame_mask = self._sample_to_frame_mask(mask, spec_features.shape[1]).to(\n                    spec_features.dtype\n                )\n            return self.feature_encoder(\n                torch.cat([spec_features, frame_mask.unsqueeze(-1)], dim=-1).float()\n            )\n\n    def decode_features(self, features):\n        return self.spec_decoder(features)\n\n    def forward(self, x, mask=None, conditioning=None):\n        features = self.get_features(x, mask)\n        if conditioning is not None:\n            if conditioning.dim() == 2:\n                conditioning = conditioning.unsqueeze(1)\n            features = features + conditioning\n        return self.decode_features(features)\n\n\ndef build_cqtdiff_decoder(device, target_sr, segment_samples, gap_durations_ms, cqt_diff_dir):\n    return OfficialCQTDiffHybridDecoder(\n        device=device,\n        target_sr=target_sr,\n        segment_samples=segment_samples,\n        gap_durations_ms=gap_durations_ms,\n        cqt_diff_dir=cqt_diff_dir,\n    ).to(device)\n', 'official_audio_inpainting_cqtdiff_adapter.py': 'import os\nimport sys\nfrom contextlib import contextmanager\n\nimport numpy as np\nimport scipy.signal\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torchaudio\nfrom omegaconf import OmegaConf\nfrom tqdm import tqdm\n\nfrom official_cqtdiff_adapter import DiffusionParams\n\n\n@contextmanager\ndef _prepend_path(path):\n    path = os.path.abspath(path)\n    already_present = path in sys.path\n    if not already_present:\n        sys.path.insert(0, path)\n    try:\n        yield\n    finally:\n        if not already_present:\n            try:\n                sys.path.remove(path)\n            except ValueError:\n                pass\n\n\n@contextmanager\ndef _numpy_clip_int_inf_compat():\n    """Compatibility for cqt_nsgt_pytorch 0.0.8 under NumPy 2.x."""\n    original_clip = np.clip\n\n    def clip_compat(a, a_min=None, a_max=None, out=None, **kwargs):\n        if out is not None and np.issubdtype(np.asarray(out).dtype, np.integer):\n            try:\n                max_is_inf = bool(np.all(np.isinf(a_max)))\n            except TypeError:\n                max_is_inf = False\n            if max_is_inf:\n                result = original_clip(a, a_min, None, **kwargs)\n                np.copyto(out, np.asarray(result, dtype=out.dtype), casting="unsafe")\n                return out\n        return original_clip(a, a_min, a_max, out=out, **kwargs)\n\n    np.clip = clip_compat\n    try:\n        yield\n    finally:\n        np.clip = original_clip\n\n\ndef _load_audio_inpainting_config(repo_dir, device):\n    conf_dir = os.path.join(repo_dir, "conf")\n    network_cfg = os.environ.get(\n        "AUDIO_INPAINTING_NETWORK", "paper_1912_unet_cqt_oct_attention_44k_2"\n    )\n    exp_cfg = os.environ.get("AUDIO_INPAINTING_EXP", "musicnet44k_4s")\n    diff_cfg = os.environ.get("AUDIO_INPAINTING_DIFF_PARAMS", "edm")\n\n    paths = {\n        "network": os.path.join(conf_dir, "network", f"{network_cfg}.yaml"),\n        "exp": os.path.join(conf_dir, "exp", f"{exp_cfg}.yaml"),\n        "diff_params": os.path.join(conf_dir, "diff_params", f"{diff_cfg}.yaml"),\n    }\n    missing = [path for path in paths.values() if not os.path.exists(path)]\n    if missing:\n        raise FileNotFoundError("Audio-inpainting config tidak ditemukan: " + ", ".join(missing))\n\n    cfg = OmegaConf.create(\n        {\n            "network": OmegaConf.load(paths["network"]),\n            "exp": OmegaConf.load(paths["exp"]),\n            "diff_params": OmegaConf.load(paths["diff_params"]),\n            "device": str(device),\n            "model_dir": "experiments/audio_inpainting_musicnet",\n        }\n    )\n    cfg.exp.sample_rate = int(cfg.exp.sample_rate)\n    cfg.exp.audio_len = int(cfg.exp.audio_len)\n    cfg.exp.resample_factor = int(cfg.exp.resample_factor)\n    return cfg\n\n\ndef _find_audio_inpainting_weights(repo_dir):\n    candidates = [\n        os.environ.get("AUDIO_INPAINTING_CQTDIFF_WEIGHTS"),\n        os.path.join(repo_dir, "experiments", "musicnet_44k_4s-560000.pt"),\n        os.path.join(repo_dir, "experiments", "musicnet_44k_4s_560000.pt"),\n        os.path.join(repo_dir, "musicnet_44k_4s-560000.pt"),\n        os.path.join(repo_dir, "musicnet_44k_4s_560000.pt"),\n    ]\n    exp_dir = os.path.join(repo_dir, "experiments")\n    if os.path.isdir(exp_dir):\n        candidates.extend(\n            os.path.join(exp_dir, name)\n            for name in sorted(os.listdir(exp_dir))\n            if name.endswith(".pt") and ("musicnet" in name.lower() or "44k_4s" in name.lower())\n        )\n    return next((path for path in candidates if path and os.path.exists(path)), None)\n\n\ndef _strip_module_prefix(state):\n    return {str(k).replace("module.", "", 1): v for k, v in state.items()}\n\n\ndef _checkpoint_to_state_dict(checkpoint):\n    if not isinstance(checkpoint, dict):\n        raise TypeError("Checkpoint format tidak dikenali.")\n\n    for key in ("ema", "network", "model", "state_dict"):\n        value = checkpoint.get(key)\n        if isinstance(value, dict):\n            return _strip_module_prefix(value), key\n\n    if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):\n        return _strip_module_prefix(checkpoint), "raw"\n\n    raise KeyError(f"Tidak menemukan state dict model di checkpoint keys={list(checkpoint.keys())}")\n\n\nclass OfficialAudioInpaintingCQTDiffDecoder(nn.Module):\n    """\n    Adapter for the 44.1 kHz MusicNet CQTdiff+ model from audio-inpainting-diffusion.\n\n    It preserves the same decoder interface used by code_final_run_v2.py:\n    baseline calls inpaint(..., conditioning=None), while hybrid models pass SSL\n    conditioning into the diffusion denoiser.\n    """\n\n    DIFFUSION_STEPS = int(os.environ.get("CQTDIFF_DIFFUSION_STEPS", "35"))\n    DIFFUSION_SIGMA_MIN = float(os.environ.get("CQTDIFF_SIGMA_MIN", "1e-4"))\n    DIFFUSION_SIGMA_MAX = float(os.environ.get("CQTDIFF_SIGMA_MAX", "1.0"))\n    DIFFUSION_SCHURN = float(os.environ.get("CQTDIFF_SCHURN", "10"))\n    DIFFUSION_SIGMA_DATA = float(os.environ.get("CQTDIFF_SIGMA_DATA", "0.063"))\n\n    def __init__(self, device, target_sr, segment_samples, gap_durations_ms, audio_inpainting_dir):\n        super().__init__()\n        self.device_ref = torch.device(device)\n        self.target_sr = int(target_sr)\n        self.target_len = int(segment_samples)\n        self.gap_durations_ms = list(gap_durations_ms)\n        self.audio_inpainting_dir = os.path.abspath(audio_inpainting_dir)\n        self.architecture_name = "ssl_conditioned_audio_inpainting_cqtdiffplus_musicnet44k_v1"\n\n        with _prepend_path(self.audio_inpainting_dir):\n            from networks.unet_cqt_oct_with_projattention_adaLN_2 import (\n                Unet_CQT_oct_with_attention,\n            )\n\n            self.args = _load_audio_inpainting_config(self.audio_inpainting_dir, self.device_ref)\n            self.native_sr = int(self.args.exp.sample_rate)\n            self.native_len = int(self.args.exp.audio_len)\n            with _numpy_clip_int_inf_compat():\n                self.backbone = Unet_CQT_oct_with_attention(self.args, self.device_ref).to(\n                    self.device_ref\n                )\n\n        require_native = os.environ.get("CQTDIFF_REQUIRE_NATIVE_SHAPE", "1").lower() in {\n            "1",\n            "true",\n            "yes",\n            "on",\n        }\n        if require_native and (self.target_sr != self.native_sr or self.target_len != self.native_len):\n            raise RuntimeError(\n                "Audio-inpainting MusicNet CQTdiff+ harus berjalan pada native checkpoint shape: "\n                f"target_sr={self.target_sr}, target_len={self.target_len}, "\n                f"native_sr={self.native_sr}, native_len={self.native_len}. "\n                "Gunakan 44100 Hz dan 184184 samples, atau set "\n                "CQTDIFF_REQUIRE_NATIVE_SHAPE=0 hanya untuk diagnosis."\n            )\n\n        weights_path = _find_audio_inpainting_weights(self.audio_inpainting_dir)\n        if weights_path is None:\n            raise FileNotFoundError(\n                "Checkpoint MusicNet CQTdiff+ belum ditemukan. Download "\n                "musicnet_44k_4s-560000.pt dari "\n                "https://huggingface.co/Eloimoliner/audio-inpainting-diffusion "\n                "atau set AUDIO_INPAINTING_CQTDIFF_WEIGHTS."\n            )\n        self.official_weights_path = os.path.abspath(weights_path)\n\n        checkpoint = torch.load(weights_path, map_location=self.device_ref, weights_only=False)\n        state, weight_type = _checkpoint_to_state_dict(checkpoint)\n        self.official_weights_type = weight_type\n        msg = self.backbone.load_state_dict(state, strict=False)\n        print(f"Audio-inpainting CQTdiff+ {weight_type} weights loaded: {weights_path}")\n        print(f"Audio-inpainting CQTdiff+ load_state_dict: {msg}")\n        print(\n            "Audio-inpainting MusicNet native setup: "\n            f"sr={self.native_sr} Hz, len={self.native_len} samples, "\n            f"duration={self.native_len / self.native_sr:.3f}s"\n        )\n\n        train_backbone = os.environ.get("CQTDIFF_TRAIN_BACKBONE", "0").lower() in {\n            "1",\n            "true",\n            "yes",\n        }\n        if not train_backbone:\n            for param in self.backbone.parameters():\n                param.requires_grad_(False)\n        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)\n        total = sum(p.numel() for p in self.backbone.parameters())\n        print(\n            "Audio-inpainting CQTdiff+ backbone training: "\n            f"{\'enabled\' if train_backbone else \'disabled\'} ({trainable:,}/{total:,})"\n        )\n\n        self.diff_params = DiffusionParams(\n            sigma_data=self.DIFFUSION_SIGMA_DATA,\n            sigma_min=self.DIFFUSION_SIGMA_MIN,\n            sigma_max=self.DIFFUSION_SIGMA_MAX,\n            ro=13,\n            Schurn=self.DIFFUSION_SCHURN,\n            Snoise=1.0,\n            Stmin=0,\n            Stmax=50,\n        )\n        print(\n            f"Diffusion sampling config: T={self.DIFFUSION_STEPS}, "\n            f"sigma=[{self.DIFFUSION_SIGMA_MIN}, {self.DIFFUSION_SIGMA_MAX}], "\n            f"Schurn={self.DIFFUSION_SCHURN}, sigma_data={self.DIFFUSION_SIGMA_DATA}"\n        )\n\n        cutoff = float(os.environ.get("CQTDIFF_FINAL_LOWPASS_HZ", "20000"))\n        cutoff = min(cutoff, self.native_sr / 2 - 100.0)\n        lpf_coeffs = scipy.signal.firwin(\n            numtaps=100, cutoff=cutoff, width=1, window="kaiser", fs=self.native_sr\n        )\n        self.register_buffer("_lpf_kernel", torch.FloatTensor(lpf_coeffs).unsqueeze(0).unsqueeze(0))\n\n        self.n_fft = int(os.environ.get("CQTDIFF_ADAPTER_N_FFT", 4096))\n        self.hop_length = int(os.environ.get("CQTDIFF_ADAPTER_HOP", 1024))\n        self.feature_dim = int(os.environ.get("CQTDIFF_ADAPTER_FEATURE_DIM", 256))\n        self.freq_bins = self.n_fft // 2 + 1\n        self.sigma = float(os.environ.get("CQTDIFF_ADAPTER_SIGMA", "0.1"))\n\n        self.feature_encoder = nn.Sequential(\n            nn.Linear(self.freq_bins * 2 + 1, 512),\n            nn.SiLU(),\n            nn.Linear(512, self.feature_dim),\n            nn.SiLU(),\n        )\n        self.spec_decoder = nn.Sequential(\n            nn.Linear(self.feature_dim, 512),\n            nn.SiLU(),\n            nn.Linear(512, self.freq_bins * 2),\n        )\n        self.condition_gate = nn.Sequential(\n            nn.Linear(self.feature_dim, 128),\n            nn.SiLU(),\n            nn.Linear(128, 1),\n        )\n        nn.init.zeros_(self.condition_gate[-1].weight)\n        nn.init.constant_(self.condition_gate[-1].bias, -2.0)\n\n        self.sigma_scale_net = nn.Sequential(\n            nn.Linear(1, 64),\n            nn.SiLU(),\n            nn.Linear(64, 1),\n        )\n        nn.init.zeros_(self.sigma_scale_net[-1].weight)\n        nn.init.zeros_(self.sigma_scale_net[-1].bias)\n\n    def _pad_or_crop(self, x, length):\n        if x.shape[-1] < length:\n            return F.pad(x, (0, length - x.shape[-1]))\n        if x.shape[-1] > length:\n            start = (x.shape[-1] - length) // 2\n            return x[..., start : start + length]\n        return x\n\n    def _to_native(self, audio):\n        x = audio.squeeze(1) if audio.dim() == 3 else audio\n        x = x.float()\n        if self.target_sr != self.native_sr:\n            x = torchaudio.functional.resample(x, self.target_sr, self.native_sr)\n        return self._pad_or_crop(x, self.native_len)\n\n    def _from_native(self, audio, target_len):\n        x = audio.squeeze(1) if audio.dim() == 3 else audio\n        if self.native_sr != self.target_sr:\n            x = torchaudio.functional.resample(x, self.native_sr, self.target_sr)\n        return self._pad_or_crop(x, target_len)\n\n    def _target_mask_to_native_keep(self, user_mask):\n        gap = user_mask.float()\n        if self.target_sr != self.native_sr:\n            gap = torchaudio.functional.resample(gap, self.target_sr, self.native_sr)\n        gap = self._pad_or_crop(gap, self.native_len)\n        return (gap < 0.5).float()\n\n    def _apply_cqt_dc_filter(self, x):\n        try:\n            return self.backbone.CQTransform.apply_hpf_DC(x)\n        except Exception:\n            return x\n\n    def _conditioning_to_waveform(self, conditioning, target_len, return_stats=False):\n        if conditioning is None:\n            return (None, {}) if return_stats else None\n        conditioning = conditioning.to(self.device_ref, dtype=torch.float32)\n        if conditioning.dim() == 2:\n            conditioning = conditioning.unsqueeze(1)\n\n        pred_spec_features = self.decode_features(conditioning).permute(0, 2, 1).contiguous()\n        with torch.autocast(device_type=self.device_ref.type, enabled=False):\n            bsz, _, n_frames = pred_spec_features.shape\n            pred_pairs = (\n                pred_spec_features.float()\n                .reshape(bsz, self.freq_bins, 2, n_frames)\n                .permute(0, 1, 3, 2)\n                .contiguous()\n            )\n            recon_spec = torch.view_as_complex(pred_pairs)\n            window = torch.hann_window(self.n_fft, device=pred_spec_features.device)\n            wave = torch.istft(\n                recon_spec,\n                n_fft=self.n_fft,\n                hop_length=self.hop_length,\n                window=window,\n                length=target_len,\n            )\n\n        pooled = conditioning.mean(dim=1)\n        gate = torch.sigmoid(self.condition_gate(pooled)).clamp(0.0, 1.0)\n        conditioned_wave = gate * wave\n        if return_stats:\n            stats = {\n                "condition_gate_mean": gate.detach().mean(),\n                "conditioning_wave_rms": torch.sqrt(wave.detach().pow(2).mean().clamp_min(1e-12)),\n                "conditioned_residual_rms": torch.sqrt(\n                    conditioned_wave.detach().pow(2).mean().clamp_min(1e-12)\n                ),\n            }\n            return conditioned_wave, stats\n        return conditioned_wave\n\n    def _conditioned_model(self, conditioning, target_len, return_stats=False):\n        if return_stats:\n            cond_wave_target, cond_stats = self._conditioning_to_waveform(\n                conditioning, target_len, return_stats=True\n            )\n        else:\n            cond_wave_target = self._conditioning_to_waveform(conditioning, target_len)\n            cond_stats = {}\n        if cond_wave_target is None:\n            return (self.backbone, cond_stats) if return_stats else self.backbone\n\n        cond_wave_native = self._to_native(cond_wave_target)\n        sigma_scale_fn = self.sigma_scale_net\n\n        def model(x, sigma):\n            backbone_trainable = any(param.requires_grad for param in self.backbone.parameters())\n            if backbone_trainable:\n                base = self.backbone(x, sigma)\n            else:\n                self.backbone.eval()\n                with torch.no_grad():\n                    base = self.backbone(x, sigma)\n            residual = cond_wave_native.to(device=x.device, dtype=x.dtype)\n            if residual.shape[0] != x.shape[0]:\n                residual = residual.expand(x.shape[0], -1)\n\n            sigma_val = sigma\n            if sigma_val.dim() == 0:\n                sigma_val = sigma_val.view(1, 1)\n            elif sigma_val.dim() == 1:\n                sigma_val = sigma_val.unsqueeze(-1)\n            scale = 1.0 + sigma_scale_fn(sigma_val)\n            return base + scale * residual\n\n        return (model, cond_stats) if return_stats else model\n\n    def _run_diffusion_sampling(self, y, mask, T=None, show_progress=True, conditioning=None):\n        if T is None:\n            T = self.DIFFUSION_STEPS\n        device = y.device\n        shape = y.shape\n        denoiser_model = self._conditioned_model(conditioning, target_len=self.target_len)\n\n        t = self.diff_params.create_schedule(T).to(device)\n        x = self.diff_params.sample_prior(shape, t[0]).to(device)\n        gamma = self.diff_params.get_gamma(t).to(device)\n\n        self.backbone.eval()\n        with torch.no_grad():\n            for i in tqdm(range(T), desc="Diffusion inpainting", leave=False, disable=not show_progress):\n                if gamma[i] == 0:\n                    t_hat = t[i]\n                    x_hat = x\n                else:\n                    t_hat = t[i] + gamma[i] * t[i]\n                    epsilon = torch.randn(shape, device=device) * self.diff_params.Snoise\n                    x_hat = x + ((t_hat**2 - t[i] ** 2) ** 0.5) * epsilon\n\n                denoised = self.diff_params.denoiser(x_hat, denoiser_model, t_hat.unsqueeze(-1))\n                denoised = self._apply_cqt_dc_filter(denoised)\n                denoised = mask * y + (1.0 - mask) * denoised\n\n                score = (denoised - x_hat) / t_hat**2\n                d = -t_hat * score\n                h = t[i + 1] - t_hat\n\n                if t[i + 1] != 0:\n                    x_prime = x_hat + h * d\n                    denoised_prime = self.diff_params.denoiser(\n                        x_prime, denoiser_model, t[i + 1].unsqueeze(-1)\n                    )\n                    denoised_prime = self._apply_cqt_dc_filter(denoised_prime)\n                    denoised_prime = mask * y + (1.0 - mask) * denoised_prime\n\n                    score_prime = (denoised_prime - x_prime) / t[i + 1] ** 2\n                    d_prime = -t[i + 1] * score_prime\n                    x = x_hat + h * (0.5 * d + 0.5 * d_prime)\n                else:\n                    x = x_hat + h * d\n\n        return x.detach()\n\n    def _apply_lowpass(self, x):\n        return F.conv1d(\n            x.unsqueeze(1),\n            self._lpf_kernel.to(x.device),\n            padding="same",\n        ).squeeze(1)\n\n    def _inpaint_diffusion(self, masked_audio, mask, T=None, show_progress=True, conditioning=None):\n        bsz = masked_audio.shape[0]\n        target_len = masked_audio.shape[-1]\n        if os.environ.get("CQTDIFF_REQUIRE_NATIVE_SHAPE", "1").lower() in {\n            "1",\n            "true",\n            "yes",\n            "on",\n        }:\n            if target_len != self.native_len or mask.shape[-1] != self.native_len:\n                raise RuntimeError(\n                    f"Input harus native_len={self.native_len}; "\n                    f"audio={target_len}, mask={mask.shape[-1]}."\n                )\n\n        mask_bool = mask[0].bool()\n        gap_indices = torch.where(mask_bool)[0]\n        if len(gap_indices) == 0:\n            out = masked_audio.detach().cpu().numpy()\n            return out[0] if bsz == 1 else out\n\n        gap_start_orig = gap_indices[0].item()\n        gap_end_orig = gap_indices[-1].item() + 1\n\n        audio_native = self._to_native(masked_audio)\n        crop_offset = 0\n        sr_scale = self.native_sr / self.target_sr\n        gap_start_nat = int(round(gap_start_orig * sr_scale)) - crop_offset\n        gap_end_nat = int(round(gap_end_orig * sr_scale)) - crop_offset\n        gap_start_nat = max(0, gap_start_nat)\n        gap_end_nat = min(self.native_len, gap_end_nat)\n\n        cqtdiff_mask = torch.ones(\n            (bsz, self.native_len), device=masked_audio.device, dtype=torch.float32\n        )\n        cqtdiff_mask[..., gap_start_nat:gap_end_nat] = 0.0\n        y = cqtdiff_mask * audio_native\n\n        x_hat = self._run_diffusion_sampling(\n            y, cqtdiff_mask, T=T, show_progress=show_progress, conditioning=conditioning\n        )\n        x_hat = self._apply_lowpass(x_hat)\n\n        gap_native = x_hat[..., gap_start_nat:gap_end_nat].clone()\n        gap_target = self._from_native(gap_native, gap_end_orig - gap_start_orig)\n\n        output = masked_audio.clone()\n        output[..., gap_start_orig:gap_end_orig] = gap_target\n        out = output.detach().cpu().numpy()\n        return out[0] if bsz == 1 else out\n\n    _DETERMINISTIC_SIGMA_STEPS = 8\n\n    def diffusion_loss(self, clean_audio, masked_audio, mask, conditioning=None, deterministic_sigma=False):\n        if os.environ.get("CQTDIFF_REQUIRE_NATIVE_SHAPE", "1").lower() in {\n            "1",\n            "true",\n            "yes",\n            "on",\n        }:\n            if clean_audio.shape[-1] != self.native_len or masked_audio.shape[-1] != self.native_len:\n                raise RuntimeError(\n                    f"Training harus native_len={self.native_len}; "\n                    f"clean={clean_audio.shape[-1]}, masked={masked_audio.shape[-1]}."\n                )\n        clean_native = self._to_native(clean_audio).float()\n        masked_native = self._to_native(masked_audio).float()\n        keep_mask = self._target_mask_to_native_keep(mask).to(clean_native.device)\n        gap_mask = 1.0 - keep_mask\n\n        batch_size = clean_native.shape[0]\n        if deterministic_sigma:\n            log_sigma_grid = torch.linspace(\n                np.log(self.DIFFUSION_SIGMA_MIN),\n                np.log(self.DIFFUSION_SIGMA_MAX),\n                steps=self._DETERMINISTIC_SIGMA_STEPS,\n                device=clean_native.device,\n            )\n            idx = torch.arange(batch_size, device=clean_native.device) % self._DETERMINISTIC_SIGMA_STEPS\n            sigma = torch.exp(log_sigma_grid[idx]).unsqueeze(1)\n        else:\n            sigma = torch.exp(\n                torch.empty(batch_size, 1, device=clean_native.device).uniform_(\n                    np.log(self.DIFFUSION_SIGMA_MIN), np.log(self.DIFFUSION_SIGMA_MAX)\n                )\n            )\n        noise = torch.randn_like(clean_native) * sigma\n        x_noisy = keep_mask * masked_native + gap_mask * (clean_native + noise)\n\n        denoiser_model, cond_stats = self._conditioned_model(\n            conditioning, target_len=clean_audio.shape[-1], return_stats=True\n        )\n        denoised = self.diff_params.denoiser(x_noisy, denoiser_model, sigma.squeeze(1))\n        denoised = self._apply_cqt_dc_filter(denoised)\n        denoised = keep_mask * masked_native + gap_mask * denoised\n\n        denom = gap_mask.sum(dim=1).clamp_min(1.0)\n        per_sample_gap_l1 = ((denoised - clean_native).abs() * gap_mask).sum(dim=1) / denom\n        gap_loss = per_sample_gap_l1.mean()\n        full_loss = (denoised - clean_native).abs().mean(dim=1).mean()\n\n        pred_rms = torch.sqrt(((denoised * gap_mask).pow(2).sum(dim=1) / denom).clamp_min(1e-10))\n        target_rms = torch.sqrt(((clean_native * gap_mask).pow(2).sum(dim=1) / denom).clamp_min(1e-10))\n        energy_loss = F.l1_loss(torch.log(pred_rms + 1e-5), torch.log(target_rms + 1e-5))\n        loss = gap_loss + 0.1 * full_loss + 0.05 * energy_loss\n        return {\n            "loss": loss,\n            "gap_loss": gap_loss,\n            "full_loss": full_loss,\n            "energy_loss": energy_loss,\n            **cond_stats,\n        }\n\n    def inpaint(self, masked_audio, mask, conditioning=None, diffusion_audio=None):\n        if masked_audio.dim() == 1:\n            masked_audio = masked_audio.unsqueeze(0)\n        if mask.dim() == 1:\n            mask = mask.unsqueeze(0)\n        masked_audio = masked_audio.to(self.device_ref, dtype=torch.float32)\n        mask = mask.to(self.device_ref)\n        return self._inpaint_diffusion(masked_audio, mask, conditioning=conditioning)\n\n    def _backbone_predict(self, audio):\n        with torch.autocast(device_type=self.device_ref.type, enabled=False):\n            native = self._to_native(audio).float()\n            sigma = torch.full(\n                (native.shape[0], 1), self.sigma, device=native.device, dtype=torch.float32\n            )\n            backbone_trainable = any(param.requires_grad for param in self.backbone.parameters())\n            if backbone_trainable:\n                pred = self.backbone(native, sigma)\n            else:\n                self.backbone.eval()\n                with torch.no_grad():\n                    pred = self.backbone(native, sigma)\n            return self._from_native(pred.float(), audio.shape[-1])\n\n    def _sample_to_frame_mask(self, mask, n_frames):\n        pooled = F.avg_pool1d(\n            mask.float().unsqueeze(1),\n            kernel_size=self.hop_length,\n            stride=self.hop_length,\n            ceil_mode=True,\n        ).squeeze(1)\n        if pooled.shape[1] < n_frames:\n            pooled = F.pad(pooled, (0, n_frames - pooled.shape[1]))\n        elif pooled.shape[1] > n_frames:\n            pooled = pooled[:, :n_frames]\n        return pooled\n\n    def get_features(self, x, mask=None):\n        with torch.autocast(device_type=self.device_ref.type, enabled=False):\n            x = x.squeeze(1) if x.dim() == 3 else x\n            x = x.float()\n            backbone_pred = self._backbone_predict(x)\n            base = torch.where(mask.bool(), backbone_pred, x) if mask is not None else backbone_pred\n\n            window = torch.hann_window(self.n_fft, device=base.device)\n            spec = torch.stft(\n                base.float(),\n                n_fft=self.n_fft,\n                hop_length=self.hop_length,\n                window=window,\n                return_complex=True,\n            )\n            spec_ri = torch.view_as_real(spec).permute(0, 2, 1, 3).contiguous()\n            spec_features = spec_ri.reshape(spec.shape[0], spec.shape[2], self.freq_bins * 2)\n            if mask is None:\n                frame_mask = torch.zeros(\n                    spec_features.shape[0],\n                    spec_features.shape[1],\n                    device=spec_features.device,\n                    dtype=spec_features.dtype,\n                )\n            else:\n                frame_mask = self._sample_to_frame_mask(mask, spec_features.shape[1]).to(\n                    spec_features.dtype\n                )\n            return self.feature_encoder(\n                torch.cat([spec_features, frame_mask.unsqueeze(-1)], dim=-1).float()\n            )\n\n    def decode_features(self, features):\n        return self.spec_decoder(features)\n\n    def forward(self, x, mask=None, conditioning=None):\n        features = self.get_features(x, mask)\n        if conditioning is not None:\n            if conditioning.dim() == 2:\n                conditioning = conditioning.unsqueeze(1)\n            features = features + conditioning\n        return self.decode_features(features)\n\n\ndef build_cqtdiff_decoder(\n    device, target_sr, segment_samples, gap_durations_ms, cqt_diff_dir=None, audio_inpainting_dir=None\n):\n    repo_dir = (\n        audio_inpainting_dir\n        or os.environ.get("AUDIO_INPAINTING_DIR")\n        or os.path.join(os.getcwd(), "external", "audio-inpainting-diffusion")\n    )\n    return OfficialAudioInpaintingCQTDiffDecoder(\n        device=device,\n        target_sr=target_sr,\n        segment_samples=segment_samples,\n        gap_durations_ms=gap_durations_ms,\n        audio_inpainting_dir=repo_dir,\n    ).to(device)\n', 'official_maid_adapter.py': 'import importlib.util\nimport os\nfrom pathlib import Path\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torchaudio\n\n\ndef _load_module(module_name: str, file_path: Path):\n    spec = importlib.util.spec_from_file_location(module_name, file_path)\n    if spec is None or spec.loader is None:\n        raise ImportError(f"Tidak bisa memuat module {module_name} dari {file_path}")\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef _find_repo_dir(explicit_dir=None):\n    candidates = [\n        explicit_dir,\n        os.environ.get("MIDI2PERFORMANCE_DIR"),\n        os.environ.get("DDPM_MIDI2PERFORMANCE_DIR"),\n        Path(__file__).resolve().parent / "external" / "DDPM-Midi2Performance-Model",\n    ]\n    for candidate in candidates:\n        if not candidate:\n            continue\n        repo_dir = Path(candidate).expanduser().resolve()\n        if (repo_dir / "main" / "models" / "diffusion" / "unet_openai.py").exists():\n            return repo_dir\n    raise FileNotFoundError(\n        "Repo DDPM-Midi2Performance-Model tidak ditemukan. Set MIDI2PERFORMANCE_DIR "\n        "atau letakkan repo di external/DDPM-Midi2Performance-Model."\n    )\n\n\ndef _parse_channel_mult(value):\n    if isinstance(value, str):\n        return tuple(int(v) for v in value.split(",") if v)\n    return tuple(value)\n\n\nclass DDPMMidi2PerformanceDecoder(nn.Module):\n    """\n    Adapter MAID untuk pipeline audio inpainting.\n\n    Backbone denoiser dan DDPM scheduler diambil dari repository\n    DDPM-Midi2Performance-Model. Kode lokal hanya menjembatani audio waveform,\n    FiLM SSL conditioning, dan interface training/evaluasi pipeline ini.\n    """\n\n    def __init__(\n        self,\n        device,\n        repo_dir=None,\n        target_sr=44100,\n        n_mels=128,\n        feature_dim=512,\n        n_fft=2048,\n        hop_length=512,\n        model_channels=64,\n        num_res_blocks=2,\n        channel_mult=(1, 1, 2, 2, 4, 4),\n        dropout=0.0,\n        beta_1=1e-4,\n        beta_2=0.02,\n        n_timesteps=1000,\n        checkpoint_path=None,\n    ):\n        super().__init__()\n        self.repo_dir = _find_repo_dir(repo_dir)\n        self.target_sr = int(target_sr)\n        self.n_mels = int(n_mels)\n        self.feature_dim = int(feature_dim)\n        self.n_fft = int(n_fft)\n        self.hop_length = int(hop_length)\n        self.n_timesteps = int(n_timesteps)\n\n        diffusion_dir = self.repo_dir / "main" / "models" / "diffusion"\n        cpu_rng_state = torch.random.get_rng_state()\n        cuda_rng_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n        try:\n            unet_mod = _load_module("ddpm_m2p_unet_openai", diffusion_dir / "unet_openai.py")\n            ddpm_mod = _load_module("ddpm_m2p_ddpm", diffusion_dir / "ddpm.py")\n        finally:\n            torch.random.set_rng_state(cpu_rng_state)\n            if cuda_rng_states is not None:\n                torch.cuda.set_rng_state_all(cuda_rng_states)\n\n        backbone = unet_mod.SuperResModel(\n            in_channels=1,\n            model_channels=int(model_channels),\n            out_channels=1,\n            num_res_blocks=int(num_res_blocks),\n            channel_mult=_parse_channel_mult(channel_mult),\n            use_checkpoint=False,\n            dropout=float(dropout),\n            dims=2,\n        )\n        self.diffusion = ddpm_mod.DDPM(\n            backbone,\n            beta_1=float(beta_1),\n            beta_2=float(beta_2),\n            T=self.n_timesteps,\n            var_type="fixedsmall",\n        )\n\n        self.feature_pool = nn.Sequential(\n            nn.Linear(self.n_mels + 1, self.feature_dim),\n            nn.SiLU(),\n            nn.Linear(self.feature_dim, self.feature_dim),\n            nn.SiLU(),\n        )\n        self.condition_to_mel = nn.Sequential(\n            nn.LayerNorm(self.feature_dim),\n            nn.Linear(self.feature_dim, self.n_mels),\n        )\n        self.register_buffer("_hann_window", torch.empty(0), persistent=False)\n        self.register_buffer("_mel_basis", torch.empty(0), persistent=False)\n\n        self._load_pretrained_if_available(checkpoint_path)\n        self.to(device)\n\n    @property\n    def device(self):\n        try:\n            return next(self.parameters()).device\n        except StopIteration:\n            return torch.device("cpu")\n\n    @property\n    def decoder(self):\n        return self.diffusion.decoder\n\n    def _get_hann_window(self, device):\n        if self._hann_window.numel() != self.n_fft or self._hann_window.device != device:\n            self._hann_window = torch.hann_window(self.n_fft, device=device)\n        return self._hann_window\n\n    def _get_mel_basis(self, sr, device):\n        expected_shape = (self.n_mels, self.n_fft // 2 + 1)\n        if self._mel_basis.shape != expected_shape or self._mel_basis.device != device:\n            self._mel_basis = torchaudio.functional.melscale_fbanks(\n                n_freqs=self.n_fft // 2 + 1,\n                f_min=0.0,\n                f_max=sr / 2,\n                n_mels=self.n_mels,\n                sample_rate=sr,\n                norm="slaney",\n                mel_scale="slaney",\n            ).to(device=device, dtype=torch.float32).T.contiguous()\n        return self._mel_basis\n\n    def _candidate_checkpoints(self, checkpoint_path):\n        candidates = [\n            checkpoint_path,\n            os.environ.get("MAID_M2P_CHECKPOINT"),\n            os.environ.get("DDPM_M2P_CHECKPOINT"),\n        ]\n        models_dir = self.repo_dir / "Models"\n        if models_dir.exists():\n            for pattern in ("*.ckpt", "*.pt", "*.pth"):\n                candidates.extend(str(path) for path in sorted(models_dir.glob(pattern)))\n        return [Path(path).expanduser().resolve() for path in candidates if path]\n\n    def _load_pretrained_if_available(self, checkpoint_path=None):\n        ckpt = next((path for path in self._candidate_checkpoints(checkpoint_path) if path.exists()), None)\n        if ckpt is None:\n            msg = (\n                "DDPM-Midi2Performance checkpoint tidak ditemukan. "\n                "Set MAID_M2P_CHECKPOINT/DDPM_M2P_CHECKPOINT ke checkpoint resmi, "\n                "atau set ALLOW_RANDOM_MAID=1 hanya untuk smoke test non-paper."\n            )\n            allow_random = os.environ.get("ALLOW_RANDOM_MAID", "0").lower() in {"1", "true", "yes"}\n            if allow_random:\n                print(f"WARNING: {msg} MAID memakai arsitektur asli dan dilatih dari awal.")\n                return\n            raise FileNotFoundError(msg)\n            return\n\n        payload = torch.load(ckpt, map_location="cpu", weights_only=False)\n        state = payload.get("state_dict", payload.get("model", payload)) if isinstance(payload, dict) else payload\n        if not isinstance(state, dict):\n            raise TypeError(f"Format checkpoint MAID tidak dikenali: {ckpt}")\n\n        prefixes = [\n            "target_network.decoder.",\n            "online_network.decoder.",\n            "diffusion.decoder.",\n            "decoder.",\n            "model.",\n        ]\n        stripped = {}\n        for key, value in state.items():\n            name = str(key)\n            for prefix in prefixes:\n                if name.startswith(prefix):\n                    name = name[len(prefix):]\n                    break\n            stripped[name] = value\n\n        decoder_state = self.decoder.state_dict()\n        matched_keys = [\n            name for name, value in stripped.items()\n            if name in decoder_state and tuple(decoder_state[name].shape) == tuple(value.shape)\n        ]\n        if not matched_keys:\n            raise RuntimeError(\n                f"Checkpoint MAID tidak memiliki key yang cocok dengan decoder saat ini: {ckpt}"\n            )\n\n        msg = self.decoder.load_state_dict(stripped, strict=False)\n        print(f"DDPM-Midi2Performance pretrained checkpoint diload: {ckpt}")\n        print(f"MAID matched pretrained keys: {len(matched_keys)} / {len(decoder_state)}")\n        print(f"MAID backbone load_state_dict: {msg}")\n\n    def audio_to_mel_batch(self, audio_batch, sr=None, return_ref=False):\n        sr = int(sr or self.target_sr)\n        if isinstance(audio_batch, torch.Tensor):\n            audio = audio_batch.to(self.device, dtype=torch.float32)\n        else:\n            audio = torch.as_tensor(audio_batch, dtype=torch.float32, device=self.device)\n        if audio.dim() == 1:\n            audio = audio.unsqueeze(0)\n        if audio.dim() == 3:\n            audio = audio.squeeze(1)\n\n        with torch.autocast(device_type="cuda" if audio.device.type == "cuda" else "cpu", enabled=False):\n            window = self._get_hann_window(audio.device)\n            spec = torch.stft(\n                audio.float(),\n                n_fft=self.n_fft,\n                hop_length=self.hop_length,\n                window=window,\n                return_complex=True,\n            )\n            power = spec.abs().pow(2)\n            mel_basis = self._get_mel_basis(sr, audio.device)\n            mel_power = torch.einsum("mf,bft->bmt", mel_basis, power).clamp_min(1e-10)\n            mel_db_raw = 10.0 * torch.log10(mel_power)\n            ref_db = mel_db_raw.amax(dim=(1, 2), keepdim=True)\n            mel_db = torch.clamp(mel_db_raw - ref_db, min=-80.0)\n            mel_norm = torch.clamp((mel_db + 40.0) / 40.0, min=-1.0, max=1.0)\n        if return_ref:\n            return mel_db, mel_norm, ref_db\n        return mel_db, mel_norm\n\n    def mask_to_frame_mask(self, mask_batch, frame_count):\n        if isinstance(mask_batch, torch.Tensor):\n            mask_t = mask_batch.to(self.device).bool()\n        else:\n            mask_t = torch.as_tensor(mask_batch, dtype=torch.bool, device=self.device)\n        if mask_t.dim() == 1:\n            mask_t = mask_t.unsqueeze(0)\n        pooled = F.max_pool1d(\n            mask_t.float().unsqueeze(1),\n            kernel_size=self.hop_length,\n            stride=self.hop_length,\n            ceil_mode=True,\n        ).squeeze(1)\n        if pooled.shape[1] < frame_count:\n            pooled = F.pad(pooled, (0, frame_count - pooled.shape[1]))\n        elif pooled.shape[1] > frame_count:\n            pooled = pooled[:, :frame_count]\n        return pooled.bool()\n\n    def get_features(self, x, mask=None):\n        _, mel_norm = self.audio_to_mel_batch(x)   # (B, n_mels, T)\n        mel_norm_t = mel_norm.permute(0, 2, 1)      # (B, T, n_mels)\n        T_frames = mel_norm_t.shape[1]\n\n        if mask is not None:\n            frame_mask = self.mask_to_frame_mask(mask, T_frames)  # (B, T) bool\n            mask_indicator = frame_mask.float().unsqueeze(-1)     # (B, T, 1)\n        else:\n            mask_indicator = torch.zeros(\n                mel_norm_t.shape[0], T_frames, 1,\n                device=mel_norm_t.device, dtype=mel_norm_t.dtype,\n            )\n\n        features_in = torch.cat([mel_norm_t, mask_indicator], dim=-1)  # (B, T, n_mels+1)\n        return self.feature_pool(features_in)  # (B, T, feature_dim)\n\n    def _conditioning_image(self, masked_mel_norm, conditioning=None):\n        cond = masked_mel_norm.unsqueeze(1)  # (B, 1, mel, T)\n        if conditioning is None:\n            return cond\n        # FIX 5 lanjutan: conditioning sekarang bisa (B, feature_dim) global\n        # atau (B, T, feature_dim) per-frame. Keduanya ditangani di sini.\n        if conditioning.dim() == 2:\n            # global vector → broadcast ke semua frame\n            bias = self.condition_to_mel(conditioning.float())          # (B, n_mels)\n            bias = torch.tanh(bias).unsqueeze(-1).expand(-1, -1, masked_mel_norm.shape[-1])\n        else:\n            # per-frame (B, T, feature_dim) → per-frame bias\n            bias = self.condition_to_mel(conditioning.float())          # (B, T, n_mels)\n            bias = torch.tanh(bias).permute(0, 2, 1)                   # (B, n_mels, T)\n            target_frames = masked_mel_norm.shape[-1]\n            if bias.shape[-1] < target_frames:\n                bias = F.pad(bias, (0, target_frames - bias.shape[-1]), value=0.0)\n            elif bias.shape[-1] > target_frames:\n                bias = bias[..., :target_frames]\n        return torch.clamp(cond + 0.25 * bias.unsqueeze(1), min=-1.0, max=1.0)\n\n    def _pad_frames_for_unet(self, mel_norm):\n        frame_count = mel_norm.shape[-1]\n        stride = 2 ** (len(self.decoder.channel_mult) - 1)\n        pad_frames = (stride - frame_count % stride) % stride\n        if pad_frames:\n            mel_norm = F.pad(mel_norm, (0, pad_frames), value=-1.0)\n        return mel_norm, frame_count\n\n    def diffusion_loss(self, clean_audio, masked_audio, mask=None, conditioning=None,\n                       deterministic_sigma=False):\n        _, clean_mel_norm = self.audio_to_mel_batch(clean_audio)\n        _, masked_mel_norm = self.audio_to_mel_batch(masked_audio)\n        clean_mel_norm, _ = self._pad_frames_for_unet(clean_mel_norm)\n        masked_mel_norm, _ = self._pad_frames_for_unet(masked_mel_norm)\n        target = clean_mel_norm.unsqueeze(1)\n        cond = self._conditioning_image(masked_mel_norm, conditioning=conditioning)\n\n        if deterministic_sigma:\n            t = torch.full((target.size(0),), self.n_timesteps // 2, device=target.device, dtype=torch.long)\n        else:\n            t = torch.randint(0, self.n_timesteps, size=(target.size(0),), device=target.device)\n        eps = torch.randn_like(target)\n        eps_pred = self.diffusion(target, eps, t, y=None, cond=cond)\n\n        full_loss = F.l1_loss(eps_pred, eps)\n        if mask is None:\n            gap_loss = full_loss\n        else:\n            frame_mask = self.mask_to_frame_mask(mask, target.shape[-1]).unsqueeze(1).unsqueeze(1)\n            expanded_mask = frame_mask.expand_as(target)\n            if expanded_mask.any():\n                gap_loss = F.l1_loss(eps_pred[expanded_mask], eps[expanded_mask])\n            else:\n                gap_loss = full_loss\n        loss = 0.7 * gap_loss + 0.3 * full_loss  # FIX 4: dari (gap + 0.1*full) → (0.7*gap + 0.3*full) agar model tidak mengabaikan konsistensi keseluruhan audio\n        return {"loss": loss, "gap_loss": gap_loss, "full_loss": full_loss}\n\n    def predict_mel_norm(self, masked_audio, conditioning=None):\n        # FIX 2: ganti t=0 shortcut dengan proper DDPM sampling agar\n        # konsisten dengan cara inpaint() bekerja saat inference.\n        # t=0 sebelumnya menyebabkan training-inference mismatch.\n        _, masked_mel_norm = self.audio_to_mel_batch(masked_audio)\n        masked_mel_norm, frame_count = self._pad_frames_for_unet(masked_mel_norm)\n        cond = self._conditioning_image(masked_mel_norm, conditioning=conditioning)\n        n_steps = int(os.environ.get("MAID_M2P_INFERENCE_STEPS", 50))\n        n_steps = max(1, min(n_steps, self.n_timesteps))\n        x_t = torch.randn_like(cond)\n        with torch.no_grad():\n            samples = self.diffusion.sample(x_t, y=None, cond=cond, n_steps=n_steps, checkpoints=[n_steps])\n        pred = torch.clamp(samples[str(n_steps)].squeeze(1), min=-1.0, max=1.0)[..., :frame_count]\n        return pred.permute(0, 2, 1)\n\n    def _mel_db_to_audio_tensor(self, mel_db_pred, target_len):\n        if mel_db_pred.dim() == 2:\n            mel_db_pred = mel_db_pred.unsqueeze(0)\n        mel_power = torch.pow(10.0, mel_db_pred.float() / 10.0).clamp_min(1e-10)\n        inverse_mel = torchaudio.transforms.InverseMelScale(\n            n_stft=self.n_fft // 2 + 1,\n            n_mels=self.n_mels,\n            sample_rate=self.target_sr,\n            f_min=0.0,\n            f_max=self.target_sr / 2,\n            norm="slaney",\n            mel_scale="slaney",\n        ).to(mel_power.device)\n        griffinlim = torchaudio.transforms.GriffinLim(\n            n_fft=self.n_fft,\n            hop_length=self.hop_length,\n            n_iter=128,  # FIX 1: dinaikkan dari 32 → 128 untuk kualitas phase reconstruction lebih baik\n            power=1.0,\n        ).to(mel_power.device)\n        linear_power = inverse_mel(mel_power).clamp_min(1e-10)\n        audio = griffinlim(linear_power.sqrt())\n        if audio.shape[-1] < target_len:\n            audio = F.pad(audio, (0, target_len - audio.shape[-1]))\n        elif audio.shape[-1] > target_len:\n            audio = audio[..., :target_len]\n        return audio\n\n    def inpaint(self, masked_audio, mask, conditioning=None, n_steps=50):\n        if masked_audio.dim() == 1:\n            masked_audio = masked_audio.unsqueeze(0)\n        if mask.dim() == 1:\n            mask = mask.unsqueeze(0)\n        masked_audio = masked_audio.to(self.device, dtype=torch.float32)\n        mask = mask.to(self.device).bool()\n\n        _, masked_mel_norm, ref_db = self.audio_to_mel_batch(masked_audio, return_ref=True)\n        masked_mel_padded, frame_count = self._pad_frames_for_unet(masked_mel_norm)\n        cond = self._conditioning_image(masked_mel_padded, conditioning=conditioning)\n\n        # FIX 3: gap-preserving replacement method.\n        # Di setiap langkah DDPM sampling, paksa frame di luar gap kembali ke\n        # nilai noisy dari audio asli. Ini mencegah model "mengubah" bagian audio\n        # yang seharusnya tidak disentuh dan membuat boundary gap lebih natural.\n        frame_mask_padded = self.mask_to_frame_mask(mask, masked_mel_padded.shape[-1]).unsqueeze(1)  # (B,1,T)\n        known_mel = masked_mel_padded.unsqueeze(1)  # (B,1,mel,T)\n\n        n_steps = int(os.environ.get("MAID_M2P_INFERENCE_STEPS", n_steps))\n        n_steps = max(1, min(n_steps, self.n_timesteps))\n\n        # Hitung alpha_bar untuk setiap timestep (dibutuhkan untuk noisy known frames)\n        betas = self.diffusion.betas  # (T,)\n        alphas = 1.0 - betas\n        alpha_bar = torch.cumprod(alphas, dim=0).to(self.device)  # (T,)\n\n        x_t = torch.randn_like(cond)\n        step_size = self.n_timesteps // n_steps\n        timesteps = list(reversed(range(0, self.n_timesteps, step_size)))[:n_steps]\n\n        for i, t_val in enumerate(timesteps):\n            t_tensor = torch.full((x_t.shape[0],), t_val, device=self.device, dtype=torch.long)\n\n            # Satu langkah denoising dari backbone\n            with torch.no_grad():\n                eps_pred = self.decoder(x_t, t_tensor, y=None, cond=cond)\n            beta_t = betas[t_val]\n            alpha_t = alphas[t_val]\n            alpha_bar_t = alpha_bar[t_val]\n            x0_pred = (x_t - (1 - alpha_bar_t).sqrt() * eps_pred) / alpha_bar_t.sqrt()\n            x0_pred = torch.clamp(x0_pred, -1.0, 1.0)\n            if t_val > 0:\n                noise = torch.randn_like(x_t)\n                x_t = alpha_bar[t_val - step_size].sqrt() * x0_pred + (1 - alpha_bar[t_val - step_size]).sqrt() * noise\n            else:\n                x_t = x0_pred\n\n            # Replacement: paksa frame non-gap kembali ke nilai noisy dari known mel\n            if t_val > 0:\n                t_prev = max(t_val - step_size, 0)\n                ab_prev = alpha_bar[t_prev]\n                known_noisy = ab_prev.sqrt() * known_mel + (1 - ab_prev).sqrt() * torch.randn_like(known_mel)\n            else:\n                known_noisy = known_mel\n            x_t = torch.where(frame_mask_padded.unsqueeze(2).expand_as(x_t), x_t, known_noisy)\n\n        pred_mel_norm = torch.clamp(x_t.squeeze(1), min=-1.0, max=1.0)[..., :frame_count]\n        output_mel_norm = torch.where(\n            self.mask_to_frame_mask(mask, pred_mel_norm.shape[-1]).unsqueeze(1),\n            pred_mel_norm,\n            masked_mel_norm,\n        )\n        output_mel_db = output_mel_norm * 40.0 - 40.0 + ref_db\n        reconstructed = self._mel_db_to_audio_tensor(output_mel_db, masked_audio.shape[-1])\n        output = torch.where(mask, reconstructed, masked_audio)\n        if output.shape[0] == 1:\n            return output[0].detach().cpu().numpy()\n        return output.detach().cpu().numpy()\n\n\ndef build_maid_decoder(\n    device,\n    target_sr,\n    segment_samples,\n    gap_durations_ms,\n    midi2performance_dir=None,\n    checkpoint_path=None,\n):\n    return DDPMMidi2PerformanceDecoder(\n        device=device,\n        repo_dir=midi2performance_dir,\n        target_sr=target_sr,\n        checkpoint_path=checkpoint_path,\n    )\n'}


In [ ]:
def install_embedded_adapters(paths: dict[str, Path]) -> Path:
    adapter_dir = paths["external"] / "embedded_adapters"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    for filename, source in EMBEDDED_ADAPTER_SOURCES.items():
        (adapter_dir / filename).write_text(source, encoding="utf-8")
    adapter_text = str(adapter_dir)
    if adapter_text not in sys.path:
        sys.path.insert(0, adapter_text)
    info(f"Embedded official adapter modules ready: {adapter_dir}")
    return adapter_dir


In [ ]:
def prepare_embedded_code_for_device(source: str) -> str:
    try:
        import torch
    except Exception:
        return source
    if torch.cuda.is_available():
        return source

    patched = source.replace(
        "        sys.exit()\nexcept ImportError:",
        "        print('CPU fallback enabled; continue without CUDA.')\nexcept ImportError:",
        1,
    )
    patched = patched.replace(
        'torch.device("cuda")',
        'torch.device("cuda" if torch.cuda.is_available() else "cpu")',
    )
    patched = patched.replace(
        'device="cuda"',
        'device=("cuda" if torch.cuda.is_available() else "cpu")',
    )
    return patched


In [ ]:
def run_baseline_eval(paths: dict[str, Path]) -> None:
    install_embedded_adapters(paths)
    sys.argv = ["code_final_run_v2.py", "--phase", "eval", "--models", TARGET_MODEL]
    info(f"Starting embedded eval: python code_final_run_v2.py --phase eval --models {TARGET_MODEL}")
    exec_globals = {
        "__name__": "__main__",
        "__file__": "code_final_run_v2.py",
        "__builtins__": __builtins__,
    }
    try:
        embedded_code = prepare_embedded_code_for_device(EMBEDDED_CODE_FINAL_RUN_V2)
        exec(
            compile(embedded_code, "code_final_run_v2.py", "exec"),
            exec_globals,
        )
    except SystemExit as exc:
        if exc.code == 0:
            return
        raise


In [ ]:
def main() -> None:
    maybe_mount_google_drive()
    paths = path_config()
    print_path_summary(paths)
    configure_environment(paths)
    ensure_musicnet_dataset(paths["dataset"])
    prepare_stage_layout(paths)
    validate_eval_artifacts(paths)
    ensure_python_requirements(paths["root"])
    ensure_external_repos(paths)
    ensure_checkpoint(paths["ckpt_path"])
    validate_cuda()
    run_baseline_eval(paths)


In [ ]:
if __name__ == "__main__":
    main()
